# Arabic AI Voice Agent for Saudi Labor Law  
## Stage 02 — Data Preparation and Knowledge Base Construction

This notebook implements the second stage of the project: preparing the legal and FAQ datasets for a Retrieval-Augmented Generation (RAG) system. The objective is to transform the raw scraped data into a clean, auditable, and evaluation-safe knowledge base that can later be indexed using embeddings and a vector database.

A key methodological decision in this stage is to separate **legal completeness** from **retrieval safety**. All cleaned legal article records are preserved for auditability, including repealed articles and valid legal variants. However, only active legal articles are allowed into the experimental RAG knowledge base. This prevents the system from retrieving outdated or repealed legal provisions when answering users.

**Inputs:** Raw scraping outputs stored under `saudi_labor_law_voice_agent_project/01_raw/`  
**Outputs:** Clean datasets, safe RAG knowledge bases, evaluation datasets, chunk files, quality reports, and academic visualisations.


## Stage 00 — Project Setup

This stage defines the project directory structure, imports the required Python libraries, and prepares the folders used throughout the data preparation pipeline.

The setup cell should be executed once at the beginning of the notebook. It ensures that all subsequent outputs are stored in a consistent and reproducible project structure.


In [1]:
# =========================================================
# Stage 00 - Project Setup
# =========================================================

# أول مرة فقط إذا احتجت:
# !pip install pandas openpyxl numpy scikit-learn matplotlib
# اختياري لتحسين عرض العربية في الرسوم:
# !pip install arabic-reshaper python-bidi

import os
import re
import json
import shutil
import hashlib
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

PROJECT_DIR = Path("saudi_labor_law_voice_agent_project")
RAW_DIR = PROJECT_DIR / "01_raw"
CLEAN_DIR = PROJECT_DIR / "02_clean"
FINAL_DIR = PROJECT_DIR / "03_final"
CHUNKS_DIR = PROJECT_DIR / "04_chunks"
REPORTS_DIR = PROJECT_DIR / "05_reports"
FIGURES_DIR = REPORTS_DIR / "figures"

for folder in [RAW_DIR, CLEAN_DIR, FINAL_DIR, CHUNKS_DIR, REPORTS_DIR, FIGURES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# محاولة اختيار خط يدعم العربية إن كان متاحاً على الجهاز
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 220
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["font.family"] = ["Tahoma", "Arial", "DejaVu Sans"]

print("Project folder is ready:", PROJECT_DIR.resolve())
print("Raw data folder:", RAW_DIR.resolve())
print("Figures folder:", FIGURES_DIR.resolve())

Project folder is ready: C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project
Raw data folder: C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\01_raw
Figures folder: C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\05_reports\figures


## Stage 01 — Input File Configuration

This stage defines the expected raw input files produced by the web scraping notebook. The files may be placed either in the same directory as this notebook or inside the project raw-data folder.

The notebook expects two main sources:

1. Saudi Labor Law articles scraped from the official HRSD source.
2. Classified HRSD FAQ records containing questions, answers, and filter metadata.

This step also defines the FAQ evaluation split ratio used later to prevent data leakage between indexing and evaluation.


In [2]:
# =========================================================
# Stage 01 - Input Files
# =========================================================

LABOR_ARTICLES_FILENAME = "saudi_labor_law_by_bab_articles.csv"
FAQ_CLASSIFIED_FILENAME = "hrsd_faq_with_filters_classification.csv"

# نسبة أسئلة FAQ التي سنتركها للتقييم لمنع تسريب البيانات في التجربة
FAQ_EVAL_RATIO = 0.30
RANDOM_SEED = 42


def find_input_file(filename, search_roots=None):
    """Find a file in the final scraping output folder first, then fallback to common locations."""
    if search_roots is None:
        search_roots = [
            RAW_DIR,
            PROJECT_DIR / "01_raw",
            Path.cwd() / "saudi_labor_law_voice_agent_project" / "01_raw",
            Path.cwd() / "01_raw",
            Path.cwd(),
            PROJECT_DIR,
            Path.cwd() / "data",
        ]

    checked = []
    for root in search_roots:
        candidate = root / filename
        checked.append(str(candidate))
        if candidate.exists():
            return candidate

    # last resort: recursive search in current directory
    for candidate in Path.cwd().glob(f"**/{filename}"):
        if candidate.is_file():
            return candidate

    raise FileNotFoundError(
        f"لم أجد الملف: {filename}\n"
        f"ضع مخرجات مرحلة السحب داخل: {RAW_DIR.resolve()}\n\n"
        f"المسارات التي تم فحصها:\n" + "\n".join(checked)
    )


LABOR_ARTICLES_FILE = find_input_file(LABOR_ARTICLES_FILENAME)
FAQ_CLASSIFIED_FILE = find_input_file(FAQ_CLASSIFIED_FILENAME)

# نسخ نسخة من الملفات الخام داخل 01_raw للتوثيق، بدون إعادة سحب من الموقع
for src in [LABOR_ARTICLES_FILE, FAQ_CLASSIFIED_FILE]:
    dst = RAW_DIR / src.name
    if src.resolve() != dst.resolve():
        shutil.copy2(src, dst)

print("Labor articles file:", LABOR_ARTICLES_FILE)
print("FAQ classified file:", FAQ_CLASSIFIED_FILE)
print("Raw files are available in:", RAW_DIR.resolve())

Labor articles file: saudi_labor_law_voice_agent_project\01_raw\saudi_labor_law_by_bab_articles.csv
FAQ classified file: saudi_labor_law_voice_agent_project\01_raw\hrsd_faq_with_filters_classification.csv
Raw files are available in: C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\01_raw


## Stage 02 — Load Raw Scraped Data

This stage loads the raw legal article dataset and the classified FAQ dataset exactly as produced by the scraping stage.

The goal is not to transform the data yet, but to verify that the raw files are readable and that their initial shapes are available for later comparison during the cleaning and validation stages.


In [3]:
# =========================================================
# Stage 02 - Load Scraped Data
# تحميل البيانات الخام وعرضها بشكل احترافي
# =========================================================

from IPython.display import display, Markdown
import pandas as pd
import numpy as np
import re
from pathlib import Path

# ---------------------------------------------------------
# 1) Load raw scraped datasets
# ---------------------------------------------------------

df_articles_raw = pd.read_csv(LABOR_ARTICLES_FILE, encoding="utf-8-sig")
df_faq_raw = pd.read_csv(FAQ_CLASSIFIED_FILE, encoding="utf-8-sig")

# ---------------------------------------------------------
# 2) Professional display helpers
# ---------------------------------------------------------

def clean_display_value(value):
    if value is None:
        return "N/A"
    try:
        if pd.isna(value):
            return "N/A"
    except Exception:
        pass

    value = str(value).strip()
    if value.lower() in ["", "nan", "none", "<na>", "n/a"]:
        return "N/A"

    return value


def short_text(text, max_len=140):
    text = clean_display_value(text)
    if text == "N/A":
        return "N/A"

    text = re.sub(r"\s+", " ", text).strip()
    return text[:max_len] + ("..." if len(text) > max_len else "")


def hide_index_safe(styler):
    try:
        return styler.hide(axis="index")
    except Exception:
        try:
            return styler.hide_index()
        except Exception:
            return styler


def apply_style_compatible(styler, func, subset):
    if hasattr(styler, "map"):
        return styler.map(func, subset=subset)
    return styler.applymap(func, subset=subset)


def professional_table(df, caption="", width="100%", font_size="12px"):
    styler = (
        df.style
        .set_caption(caption)
        .set_table_styles([
            {
                "selector": "caption",
                "props": [
                    ("caption-side", "top"),
                    ("font-size", "17px"),
                    ("font-weight", "bold"),
                    ("color", "#1F4E78"),
                    ("text-align", "center"),
                    ("padding", "10px")
                ]
            },
            {
                "selector": "th",
                "props": [
                    ("background-color", "#1F4E78"),
                    ("color", "white"),
                    ("font-weight", "bold"),
                    ("text-align", "center"),
                    ("border", "1px solid #D9E2F3"),
                    ("padding", "8px"),
                    ("white-space", "normal")
                ]
            },
            {
                "selector": "td",
                "props": [
                    ("text-align", "center"),
                    ("border", "1px solid #D9E2F3"),
                    ("padding", "7px"),
                    ("vertical-align", "middle"),
                    ("white-space", "normal")
                ]
            },
            {
                "selector": "tbody tr:nth-child(even)",
                "props": [("background-color", "#F8FBFD")]
            },
            {
                "selector": "tbody tr:nth-child(odd)",
                "props": [("background-color", "#FFFFFF")]
            },
            {
                "selector": "table",
                "props": [
                    ("border-collapse", "collapse"),
                    ("width", width),
                    ("font-family", "Arial, Tahoma, sans-serif"),
                    ("font-size", font_size),
                    ("table-layout", "fixed"),
                    ("margin", "12px 0")
                ]
            }
        ])
    )

    return hide_index_safe(styler)


def color_status(value):
    value = str(value)

    if value in ["Loaded", "Available", "Yes", "OK"]:
        return "background-color:#E2F0D9; color:#375623; font-weight:bold;"

    if value in ["Missing", "No", "Warning"]:
        return "background-color:#FCE4D6; color:#9C0006; font-weight:bold;"

    return ""


# ---------------------------------------------------------
# 3) Stage explanation
# ---------------------------------------------------------

display(Markdown("""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:14px;
    margin:12px 0;
    border-radius:6px;
">

<h3 style="color:#1F4E78; margin-top:0;">
Stage 02 — Load Scraped Data
</h3>

<p>
تهدف هذه المرحلة إلى تحميل البيانات الخام التي تم سحبها من المصادر الرسمية، وتشمل مواد نظام العمل السعودي وأسئلة FAQ المصنفة.
يتم هنا فحص حجم البيانات، الأعمدة، القيم المفقودة، والتكرارات قبل الدخول في مراحل التنظيف وبناء قاعدة المعرفة.
</p>

</div>
"""))

# ---------------------------------------------------------
# 4) Source files summary
# ---------------------------------------------------------

source_files_summary = pd.DataFrame([
    {
        "Dataset": "Saudi Labor Law Articles",
        "Source File": str(LABOR_ARTICLES_FILE),
        "Rows": df_articles_raw.shape[0],
        "Columns": df_articles_raw.shape[1],
        "Status": "Loaded"
    },
    {
        "Dataset": "HRSD FAQ Classified",
        "Source File": str(FAQ_CLASSIFIED_FILE),
        "Rows": df_faq_raw.shape[0],
        "Columns": df_faq_raw.shape[1],
        "Status": "Loaded"
    }
])

source_style = professional_table(
    source_files_summary,
    caption="Table 2.1 — Raw Data Sources Loaded Successfully",
    width="100%",
    font_size="12px"
)

source_style = apply_style_compatible(
    source_style,
    color_status,
    subset=["Status"]
)

source_style = source_style.set_properties(
    subset=["Source File"],
    **{
        "text-align": "left",
        "direction": "ltr",
        "font-size": "11px"
    }
)

display(source_style)

# ---------------------------------------------------------
# 5) Dataset quality summary
# ---------------------------------------------------------

def dataset_quality_summary(df, dataset_name):
    total_cells = df.shape[0] * df.shape[1]
    missing_cells = int(df.isna().sum().sum())
    duplicate_rows = int(df.duplicated().sum())

    return {
        "Dataset": dataset_name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Total Cells": total_cells,
        "Missing Cells": missing_cells,
        "Missing %": round((missing_cells / total_cells) * 100, 2) if total_cells > 0 else 0,
        "Duplicate Rows": duplicate_rows
    }


quality_summary = pd.DataFrame([
    dataset_quality_summary(df_articles_raw, "Saudi Labor Law Articles"),
    dataset_quality_summary(df_faq_raw, "HRSD FAQ Classified")
])

quality_style = professional_table(
    quality_summary,
    caption="Table 2.2 — Initial Raw Data Quality Overview",
    width="100%",
    font_size="12px"
)

display(quality_style)

# ---------------------------------------------------------
# 6) Column schema overview
# ---------------------------------------------------------

articles_columns_df = pd.DataFrame({
    "Dataset": "Saudi Labor Law Articles",
    "Column Name": list(df_articles_raw.columns),
    "Data Type": [str(df_articles_raw[col].dtype) for col in df_articles_raw.columns],
    "Missing Values": [int(df_articles_raw[col].isna().sum()) for col in df_articles_raw.columns],
    "Non-Null Values": [int(df_articles_raw[col].notna().sum()) for col in df_articles_raw.columns],
})

faq_columns_df = pd.DataFrame({
    "Dataset": "HRSD FAQ Classified",
    "Column Name": list(df_faq_raw.columns),
    "Data Type": [str(df_faq_raw[col].dtype) for col in df_faq_raw.columns],
    "Missing Values": [int(df_faq_raw[col].isna().sum()) for col in df_faq_raw.columns],
    "Non-Null Values": [int(df_faq_raw[col].notna().sum()) for col in df_faq_raw.columns],
})

schema_overview = pd.concat(
    [articles_columns_df, faq_columns_df],
    ignore_index=True
)

schema_style = professional_table(
    schema_overview,
    caption="Table 2.3 — Raw Dataset Column Schema and Missing Values",
    width="100%",
    font_size="11.5px"
)

schema_style = schema_style.set_properties(
    subset=["Column Name"],
    **{
        "text-align": "left",
        "direction": "ltr",
        "font-weight": "bold"
    }
)

display(schema_style)

# ---------------------------------------------------------
# 7) Professional preview of articles dataset
# ---------------------------------------------------------

article_preview_priority_cols = [
    "source_type",
    "legal_category",
    "bab_title",
    "chapter_title",
    "article_number_int",
    "article_number",
    "article_title",
    "article_status",
    "chunk_text",
    "article_text",
    "text"
]

article_preview_cols = [
    col for col in article_preview_priority_cols
    if col in df_articles_raw.columns
]

if len(article_preview_cols) == 0:
    article_preview_cols = list(df_articles_raw.columns[:8])

articles_preview = df_articles_raw[article_preview_cols].head(5).copy()

for col in articles_preview.columns:
    if articles_preview[col].dtype == "object":
        articles_preview[col] = articles_preview[col].apply(lambda x: short_text(x, 160))

articles_style = professional_table(
    articles_preview,
    caption="Table 2.4 — Preview of Raw Saudi Labor Law Articles",
    width="100%",
    font_size="11.5px"
)

rtl_article_cols = [
    col for col in articles_preview.columns
    if col not in ["source_type", "article_number_int", "article_number", "article_status"]
]

if rtl_article_cols:
    articles_style = articles_style.set_properties(
        subset=rtl_article_cols,
        **{
            "text-align": "right",
            "direction": "rtl",
            "line-height": "1.7"
        }
    )

display(articles_style)

# ---------------------------------------------------------
# 8) Professional preview of FAQ dataset
# ---------------------------------------------------------

faq_preview_priority_cols = [
    "question",
    "answer",
    "beneficiaries",
    "sector",
    "subsite",
    "category",
    "classification",
    "source",
    "page_url"
]

faq_preview_cols = [
    col for col in faq_preview_priority_cols
    if col in df_faq_raw.columns
]

if len(faq_preview_cols) == 0:
    faq_preview_cols = list(df_faq_raw.columns[:8])

faq_preview = df_faq_raw[faq_preview_cols].head(5).copy()

for col in faq_preview.columns:
    if faq_preview[col].dtype == "object":
        faq_preview[col] = faq_preview[col].apply(lambda x: short_text(x, 160))

faq_style = professional_table(
    faq_preview,
    caption="Table 2.5 — Preview of Raw HRSD FAQ Records",
    width="100%",
    font_size="11.5px"
)

rtl_faq_cols = [
    col for col in faq_preview.columns
    if col not in ["source", "page_url"]
]

if rtl_faq_cols:
    faq_style = faq_style.set_properties(
        subset=rtl_faq_cols,
        **{
            "text-align": "right",
            "direction": "rtl",
            "line-height": "1.7"
        }
    )

if "page_url" in faq_preview.columns:
    faq_style = faq_style.set_properties(
        subset=["page_url"],
        **{
            "text-align": "left",
            "direction": "ltr",
            "font-size": "10px"
        }
    )

display(faq_style)

# ---------------------------------------------------------
# 9) Final confirmation
# ---------------------------------------------------------

display(Markdown(f"""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#E2F0D9;
    border-right:5px solid #375623;
    padding:14px;
    margin:12px 0;
    border-radius:6px;
">

<b>تم تحميل البيانات الخام بنجاح.</b><br>

عدد مواد نظام العمل الخام: <b>{df_articles_raw.shape[0]}</b><br>
عدد سجلات FAQ الخام: <b>{df_faq_raw.shape[0]}</b><br>

ستنتقل الخلايا التالية إلى تنظيف النصوص، إزالة التكرارات، تصنيف FAQ، وبناء قاعدة معرفة مناسبة لنظام RAG.

</div>
"""))

print("Stage 02 completed successfully.")
print("df_articles_raw shape:", df_articles_raw.shape)
print("df_faq_raw shape:", df_faq_raw.shape)


<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:14px;
    margin:12px 0;
    border-radius:6px;
">

<h3 style="color:#1F4E78; margin-top:0;">
Stage 02 — Load Scraped Data
</h3>

<p>
تهدف هذه المرحلة إلى تحميل البيانات الخام التي تم سحبها من المصادر الرسمية، وتشمل مواد نظام العمل السعودي وأسئلة FAQ المصنفة.
يتم هنا فحص حجم البيانات، الأعمدة، القيم المفقودة، والتكرارات قبل الدخول في مراحل التنظيف وبناء قاعدة المعرفة.
</p>

</div>


Dataset,Source File,Rows,Columns,Status
Saudi Labor Law Articles,saudi_labor_law_voice_agent_project\01_raw\saudi_labor_law_by_bab_articles.csv,249,13,Loaded
HRSD FAQ Classified,saudi_labor_law_voice_agent_project\01_raw\hrsd_faq_with_filters_classification.csv,711,10,Loaded


Dataset,Rows,Columns,Total Cells,Missing Cells,Missing %,Duplicate Rows
Saudi Labor Law Articles,249,13,3237,110,3.400000,0
HRSD FAQ Classified,711,10,7110,87,1.220000,0


Dataset,Column Name,Data Type,Missing Values,Non-Null Values
Saudi Labor Law Articles,bab_order,int64,0,249
Saudi Labor Law Articles,bab_label,str,0,249
Saudi Labor Law Articles,bab_title,str,0,249
Saudi Labor Law Articles,fasl,str,110,139
Saudi Labor Law Articles,article_title,str,0,249
Saudi Labor Law Articles,article_number,str,0,249
Saudi Labor Law Articles,article_number_label,str,0,249
Saudi Labor Law Articles,article_number_int,int64,0,249
Saudi Labor Law Articles,article_text,str,0,249
Saudi Labor Law Articles,source_url,str,0,249


bab_title,article_number_int,article_number,article_title,article_status,article_text
التعريفات / الأحكام العامة,1,الأولى,المادة الأولى:,active,يسمى هذا النظام نظام العمل.
التعريفات / الأحكام العامة,2,الثانية,المادة الثانية :,active,يقصد بالألفاظ والعبارات الآتية – أينما وردت في هذا النظام – المعاني المبينة أمامها ما لم يقتض السياق خلاف ذلك: -الوزارة: وزارة الموارد البشرية والتنمية الاجتماعية. -الوزير: وزير الموارد البشرية والتنمية الاجتماعية. -مكتب العمل: الجهة الإدارية المنوط بها شؤون العمل في النطاق المكاني الذي يحدد بقرار من الوزير. -صاحب العمل: كل شخص طبيعي أو اعتباري يشغّل عاملاً أو أكثر مقابل أجر. -العامل: كل شخص طبيعي –ذكراً أو أنثى- يعمل لمصلحة صاحب عمل وتحت إدارته أو إشرافه مقابل أجر، ولو كان بعيداً عن نظارته. -الحدث: الشخص الذي أتم الخامسة عشرة من عمره ولم يبلغ الثامنة عشرة. -العمل: الجهد المبذول في النشاطات الإنسانية كافة، تنفيذاً لعقد عمل (مكتوب أو غير مكتوب) بصرف النظر عن طبيعتها أو نوعها، صناعية كانت أو تجارية، أو زراعية أو فنية، أو غيرها، عضلية كانت أو ذهنية. -العمل الأصلي: بالنسبة للأفراد: موضوع نشاطهم المعتاد، وبالنسبة للمنشآت: الأعمال التي أنشئت المنشأة من أجل القيام بها والمنصوص عليها في عقد تأسيسها أو في عقد الامتياز – إن كانت من شركات الامتياز – أو في السجل التجاري. -العمل المؤقت: العمل الذي يدخل بطبيعته فيما يزاوله صاحب العمل من نشاط وتقتضي طبيعة إنجازه مدة محددة، أو ينصب على عمل بذاته وينتهي بانتهائه، ولا يتجاوز في الحالتين تسعين يوماً. -العمل العرضي: العمل الذي لا يدخل بطبيعته فيما يزاوله صاحب العمل في نشاطه المعتاد، ولا يستغرق تنفيذه أكثر من تسعين يوماً. -العمل الموسمي: العمل الذي يتم في مواسم دورية متعارف عليها. -العمل لبعض الوقت: العمل الذي يؤديه عامل غير متفرغ لدى صاحب عمل ولساعات عمل تقل عن نصف ساعات العمل اليومية المعتادة لدى المنشأة، سواء كان هذا العامل يؤدي ساعات عمله يومياً أو بعض أيام الأسبوع. -الخدمة المستمرة: خدمة العامل غير المنقطعة مع صاحب العمل نفسه أو خلفه النظامي، من تاريخ ابتداء الخدمة. وتعد الخدمة مستمرة في الحالات الآتية: 1.الإجازات والعطل المقررة نظاماً. 2.فترة الانقطاع لأداء الامتحانات وفقاً لما هو منصوص عليه في هذا النظام. 3.حالات غياب العامل عن عمله بدون أجر التي لا تزيد مدتها على عشرين يوماً متقطعة خلال سنة العمل. -الإسناد: خدمة توفير عامل للعمل لدى غير صاحب العمل وذلك من خلال منشأة مرخص لها لهذا الغرض. -الاستقالة: إفصاح العامل كتابة عن رغبته دون إكراه في إنهاء عقد عمل محدد المدة دون تعليق على قيد أو شرط، وقبول صاحب العمل بها. -الأجر الأساسي: كل ما يعطى للعامل مقابل عمله، بموجب عقد عمل مكتوب أو غير مكتوب، مهما كان نوع الأجر أو طريقة أدائه، مضافاً إليه العلاوات الدورية. -الأجر الفعلي: الأجر الأساسي مضافاً إليه سائر الزيادات المستحقة الأخرى التي تتقرر للعامل مقابل جهد بذله في العمل، أو مخاطر يتعرض لها في أداء عمله، أو التي تتقرر للعامل لقاء العمل بموجب عقد العمل أو لائحة تنظيم العمل. ومن ذلك: 1- العمولة، أو النسبة المئوية من المبيعات، أو النسبة المئوية من الأرباح، التي تدفع مقابل ما يقوم بتسويقه، أو إنتاجه، أو تحصيله، أو ما يحققه من زيادة الإنتاج أو تحسينه. 2- البدلات التي يستحقها العامل لقاء طاقة يبذلها، أو مخاطر يتعرض لها في أداء عمله. 3- الزيادات التي قد تمنح وفقاً لمستوى المعيشة، أو لمواجهة أعباء العائلة. 4- المنحة أو المكافأة: هي التي يعطيها صاحب العمل للعامل، وما يصرف له جزاء أمانته، أو كفايته، وما شابه ذلك، إذا كانت هذه المنحة أو المكافأة مقررة في عقد العمل، أو لائحة تنظيم العمل للمنشأة، أو جرت العادة بمنحها، حتى أصبح العمال يعدونها جزءاً من الأجر لا تبرعاً. 5- الميزات العينية: هي التي يلتزم صاحب العمل بتوفيرها للعامل مقابل عمله، بالنص عليها في عقد العمل أو في لائحة تنظيم العمل. وتقدر بحد أقصى يعادل الأجر الأساسي لمدة شهرين عن كل سنة ما لم تقدر في عقد العمل أو لائحة تنظيم العمل بما يزيد على ذلك. -الأجر: الأجر الفعلي. -المنشأة: كل مشروع يديره شخص طبيعي، أو اعتباري، يشغًل عاملاً أو أكثر، لقاء أجر أياً كان نوعه. -الشهر: ثلاثون يوماً ما لم ينص على خلاف ذلك في عقد العمل أو في لائحة تنظيم العمل. -اللائحة: اللائحة التنفيذية لهذا النظام.
التعريفات / الأحكام العامة,3,الثالثة,المادة الثالثة:,active,العمل حق للمواطن ، لا يجوز لغيره ممارسته إلا بعد توافر الشروط المنصوص عليها في هذا النظام ، والمواطنون متساوون في حق العمل دون أي تمييز على أساس الجنس أو الإعاقة أو السن أو أي شكل من أشكا

question,answer,beneficiaries,sector,subsite,source,page_url
هل تستخدم البوابة والانظمة التابعة لها الحلول الوطنية eSignature؟,"تقدم الوزارة خدمة التوقيع الالكتروني في اطار ادارة وتوثيق عقود العمل عبر منصة ""قوى"" يمكن للمنشآت انشاء وتوثيق وانهاء عقود العمل للموظفين بشكل الكرتوني بالكامل.",أصحاب عمل | أفراد | العمالة المنزلية | كبار السن,قطاع العمل,مركز الرياض للسياسات السلوكية,HRSD FAQ,https://www.hrsd.gov.sa/contact-us/faq
ماذا أفعل إذا تم إرجاع الطلب للمراجعة؟,الدخول إلى “مهامي” ← اختيار مهمة طلب مراجعة ← تعديل البيانات ← إعادة الإرسال.,التخصصات الاجتماعية | العمالة المنزلية | كبار السن,قطاع التنمية الاجتماعية,مركز الرياض للسياسات السلوكية,HRSD FAQ,https://www.hrsd.gov.sa/contact-us/faq
ماذا يحدث بعد إرسال الطلب؟,يتم تحويل الطلب إلى اللجنة للمراجعة ويتم إشعار المتقدم بحالة الطلب.,التخصصات الاجتماعية | العمالة المنزلية | كبار السن,قطاع التنمية الاجتماعية,مركز الرياض للسياسات السلوكية,HRSD FAQ,https://www.hrsd.gov.sa/contact-us/faq
كيف أقدم طلب تعديل فئة مهنية؟,من خلال تسجيل الدخول ← الخدمات ← اختيار “طلب تعديل الفئة المهنية” ← تعبئة النموذج ← إرسال الطلب.,التخصصات الاجتماعية | العمالة المنزلية | كبار السن,قطاع التنمية الاجتماعية,مركز الرياض للسياسات السلوكية,HRSD FAQ,https://www.hrsd.gov.sa/contact-us/faq
ما متطلبات التقديم؟,أن يكون المتقدم حاصل على ترخيص سابق.,التخصصات الاجتماعية | العمالة المنزلية | كبار السن,قطاع التنمية الاجتماعية,مركز الرياض للسياسات السلوكية,HRSD FAQ,https://www.hrsd.gov.sa/contact-us/faq



<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#E2F0D9;
    border-right:5px solid #375623;
    padding:14px;
    margin:12px 0;
    border-radius:6px;
">

<b>تم تحميل البيانات الخام بنجاح.</b><br>

عدد مواد نظام العمل الخام: <b>249</b><br>
عدد سجلات FAQ الخام: <b>711</b><br>

ستنتقل الخلايا التالية إلى تنظيف النصوص، إزالة التكرارات، تصنيف FAQ، وبناء قاعدة معرفة مناسبة لنظام RAG.

</div>


Stage 02 completed successfully.
df_articles_raw shape: (249, 13)
df_faq_raw shape: (711, 10)


## Stage 03 — Arabic Text Cleaning Utilities

This stage defines reusable helper functions for cleaning Arabic legal and FAQ text.

The cleaning process is designed to be conservative: it removes noise such as extra spaces, unnecessary symbols, and formatting artefacts while preserving the legal meaning of the original text. A separate indexing-friendly version of the text is later created for retrieval purposes.


In [4]:
# =========================================================
# Stage 03 - Arabic Text Cleaning Helpers
# =========================================================

ARABIC_DIACRITICS = re.compile(r"[\u0617-\u061A\u064B-\u0652]")
TATWEEL = "\u0640"


def clean_basic_text(text):
    """
    تنظيف محافظ لا يغيّر معنى النص القانوني.
    نستخدمه للنصوص الأصلية والأعمدة العامة.
    """
    if pd.isna(text):
        return ""
    text = str(text)
    text = text.replace("\xa0", " ")
    text = text.replace("\u200f", " ")
    text = text.replace("\u200e", " ")
    text = text.replace("\ufeff", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def normalize_arabic_for_search(text):
    """
    نسخة مخصصة للفهرسة/البحث فقط، وليست بديلاً عن النص القانوني الأصلي.
    """
    text = clean_basic_text(text)
    text = text.replace(TATWEEL, "")
    text = ARABIC_DIACRITICS.sub("", text)
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ؤ", "و", text)
    text = re.sub("ئ", "ي", text)
    text = re.sub("ة", "ه", text)
    text = re.sub(r"[^\u0600-\u06FFa-zA-Z0-9\s،؛؟:.,()\-/%]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def safe_col(df, col, default=""):
    """Create missing column if not found."""
    if col not in df.columns:
        df[col] = default
    return df


def make_hash(*values):
    """Stable id for records/chunks."""
    raw = "|||".join(clean_basic_text(v) for v in values)
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()


def deterministic_split_by_hash(value, eval_ratio=0.20):
    """
    Deterministic split without random leakage.
    Returns True if record should go to evaluation.
    """
    h = hashlib.md5(str(value).encode("utf-8")).hexdigest()
    bucket = int(h[:8], 16) / 0xFFFFFFFF
    return bucket < eval_ratio

## Stage 04 — Clean Saudi Labor Law Articles

This stage cleans the legal article dataset while preserving legal auditability.

Repealed articles are **not deleted** at this stage. Instead, they are explicitly labelled using the `article_status` field. This design keeps the cleaned legal dataset complete for documentation and audit purposes, while allowing later stages to safely exclude repealed provisions from RAG indexing.

The cleaned legal dataset therefore supports two goals:

1. Maintaining a complete legal audit dataset.
2. Building a safe retrieval knowledge base using active articles only.


In [5]:
# =========================================================
# Stage 04 - Clean Legal Articles
# الاحتفاظ بالمواد الملغاة ومواد "مكرر" لأنها مهمة قانونياً
# =========================================================

df_articles = df_articles_raw.copy()

required_article_cols = [
    "bab_order", "bab_label", "bab_title", "fasl",
    "article_title", "article_number", "article_number_label",
    "article_number_int", "article_text", "source_url"
]

for col in required_article_cols:
    df_articles = safe_col(df_articles, col, "")

# تنظيف محافظ للأعمدة النصية
for col in required_article_cols:
    df_articles[col] = df_articles[col].apply(clean_basic_text)

# إذا لم يكن article_number_label متوفراً بشكل جيد، نستخدم article_number كبديل
df_articles["article_number_label"] = np.where(
    df_articles["article_number_label"].astype(str).str.strip().eq(""),
    df_articles["article_number"],
    df_articles["article_number_label"],
)

# تحويل الرقم إلى قيمة رقمية مع الاحتفاظ بالنسخة النصية القانونية في article_number_label
df_articles["article_number_int"] = pd.to_numeric(df_articles["article_number_int"], errors="coerce")

# تنظيف نص المادة مع إزالة التطويل فقط، بدون حذف المواد القصيرة
before_raw_rows = len(df_articles)
df_articles["article_text"] = (
    df_articles["article_text"]
    .fillna("")
    .astype(str)
    .str.replace(r"[ـ]+", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df_articles["text_len"] = df_articles["article_text"].str.len()

# تصنيف حالة المادة
df_articles["article_status"] = df_articles["article_text"].apply(
    lambda x: "repealed" if ("ملغاة" in x or "ملغى" in x) else "active"
)

# حذف الصفوف الفارغة فقط، وليس المواد القصيرة مثل "ملغاة"
df_articles = df_articles[df_articles["article_text"].str.len() > 0].copy()
before_dedup_rows = len(df_articles)

# إزالة التكرار الحقيقي حسب article_number_label، مع الاحتفاظ بأطول نص لنفس المادة إن تكرر نفس label فعلاً
df_articles = (
    df_articles
    .sort_values(["article_number_label", "text_len"], ascending=[True, False])
    .drop_duplicates(subset=["article_number_label"], keep="first")
    .sort_values(["article_number_int", "article_number_label"], na_position="last")
    .reset_index(drop=True)
)

# Metadata

df_articles["source_type"] = "legal_article"
df_articles["source_name"] = "نظام العمل السعودي"
df_articles["legal_category"] = df_articles["bab_title"].replace("", np.nan).fillna("نظام العمل")

df_articles["document_unit_id"] = df_articles.apply(
    lambda r: make_hash(
        r["source_type"],
        r["article_number_label"],
        r["article_title"],
        r["article_text"]
    ),
    axis=1
)

df_articles["text_for_indexing"] = df_articles.apply(
    lambda r: clean_basic_text(
        f"المصدر: {r['source_name']}\n"
        f"{r['bab_label']} - {r['bab_title']}\n"
        f"{r['fasl']}\n"
        f"{r['article_title']}\n"
        f"رقم المادة: {r['article_number_label']}\n"
        f"حالة المادة: {r['article_status']}\n"
        f"النص: {r['article_text']}"
    ),
    axis=1
)

df_articles["normalized_text"] = df_articles["text_for_indexing"].apply(normalize_arabic_for_search)

# حفظ النسخة النظيفة

df_articles.to_csv(CLEAN_DIR / "clean_labor_law_articles.csv", index=False, encoding="utf-8-sig")
df_articles.to_excel(CLEAN_DIR / "clean_labor_law_articles.xlsx", index=False)

# فحوصات مهمة
valid_nums = df_articles["article_number_int"].dropna().astype(int)
nums = set(valid_nums)
max_num = max(nums) if nums else 0
missing_numeric_articles = [n for n in range(1, max_num + 1) if n not in nums]
true_duplicate_labels = int(df_articles["article_number_label"].duplicated().sum())
repeated_numeric_numbers = int(df_articles["article_number_int"].duplicated().sum())

print("Articles raw rows:", before_raw_rows)
print("Articles after empty-text filter:", before_dedup_rows)
print("Legal article records after true deduplication:", len(df_articles))
print("Unique numeric article numbers:", len(nums), "| max:", max_num)
print("Missing numeric article numbers:", missing_numeric_articles)
print("True duplicated article labels:", true_duplicate_labels)
print("Repeated numeric numbers, often legal 'مكرر' articles:", repeated_numeric_numbers)
print("Empty article_text:", int((df_articles["article_text"].astype(str).str.strip() == "").sum()))

print("\nArticle status distribution:")
display(df_articles["article_status"].value_counts().to_frame("count"))

print("\nShort non-repealed articles. This should usually contain only very short valid articles such as Article 1:")
display(
    df_articles[
        (df_articles["text_len"] < 30) &
        (df_articles["article_status"] != "repealed")
    ][["article_number_label", "article_title", "article_text", "text_len", "article_status"]]
)

display(df_articles[[
    "bab_label", "bab_title", "fasl", "article_title",
    "article_number_label", "article_status", "article_text"
]].head(10))

Articles raw rows: 249
Articles after empty-text filter: 249
Legal article records after true deduplication: 249
Unique numeric article numbers: 245 | max: 245
Missing numeric article numbers: []
True duplicated article labels: 0
Repeated numeric numbers, often legal 'مكرر' articles: 4
Empty article_text: 0

Article status distribution:


,count
article_status,
active,211
repealed,38



Short non-repealed articles. This should usually contain only very short valid articles such as Article 1:


,article_number_label,article_title,article_text,text_len,article_status
0,1,المادة الأولى:,يسمى هذا النظام نظام العمل.,27,active


,bab_label,bab_title,fasl,article_title,article_number_label,article_status,article_text
0,الباب الأول,التعريفات / الأحكام العامة,الفصل الأول,المادة الأولى:,1,active,يسمى هذا النظام نظام العمل.
1,الباب الأول,التعريفات / الأحكام العامة,الفصل الأول,المادة الثانية :,2,active,يقصد بالألفاظ والعبارات الآتية – أينما وردت في هذا النظام – المعاني المبينة أمامها ما لم يقتض السياق خلاف ذلك: -الوزارة: وزارة الموارد البشرية والتنمية الاجتماعية. -الوزير: وزي...
2,الباب الأول,التعريفات / الأحكام العامة,الفصل الثاني,المادة الثالثة:,3,active,العمل حق للمواطن ، لا يجوز لغيره ممارسته إلا بعد توافر الشروط المنصوص عليها في هذا النظام ، والمواطنون متساوون في حق العمل دون أي تمييز على أساس الجنس أو الإعاقة أو السن أو أي ...
3,الباب الأول,التعريفات / الأحكام العامة,الفصل الثاني,المادة الرابعة:,4,active,يجب على صاحب العمل والعامل عند تطبيق أحكام هذا النظام الإلتزام بمقتضيات أحكام الشريعة الإسلامية .
4,الباب الأول,التعريفات / الأحكام العامة,الفصل الثاني,المادة الخامسة:,5,active,تسري أحكام هذا النظام على الآتي: كل عقد عمل يلتزم بمقتضاه أي شخص بالعمل لمصلحة صاحب عمل وتحت إدارته أو إشرافه ؛ مقابل أجر. عمال الحكومة والهيئات والمؤسسات العامة ، بمن فيهم الذ...
5,الباب الأول,التعريفات / الأحكام العامة,الفصل الثاني,المادة السادسة:,6,active,تسري على العامل العرضي والموسمي والمؤقت الأحكام الخاصة بالواجبات وقواعد التأديب ، والحد الأقصى لساعات العمل ، وفترات الراحة اليومية والراحة الأسبوعية ، والتشغيل الإضافي ، والعط...
6,الباب الأول,التعريفات / الأحكام العامة,الفصل الثاني,المادة السابعة :,7,active,1- يستثنى من تطبيق أحكام هذا النظام كل من: أ‌- أفراد أسرة صاحب العمل، وهم زوجه وأصوله وفروعه الذين يعملون في المنشأة التي لا تضم سواهم. ب‌- لاعبو الأندية والاتحادات الرياضية وم...
7,الباب الأول,التعريفات / الأحكام العامة,الفصل الثاني,المادة الثامنة:,8,active,يبطل كل شرط يخالف أحكام هذا النظام ، ويبطل كل إبراء ، أو مصالحة عن الحقوق الناشئة للعامل بموجب هذا النظام ، أثناء سريان عقد العمل ، ما لم يكن أكثر فائدة للعامل.
8,الباب الأول,التعريفات / الأحكام العامة,الفصل الثاني,المادة التاسعة:,9,active,اللغة العربية هي الواجبة الإستعمال في البيانات والسجلات والملفات وعقود العمل وغيرها مما هو منصوص عليه في هذا النظام ، أو في أي قرار صادر تطبيقاً لأحكامه ، وكذلك التعليمات التي ...
9,الباب الأول,التعريفات / الأحكام العامة,الفصل الثاني,المادة العاشرة:,10,active,تحسب جميع المدد والمواعيد المنصوص عليها في هذا النظام بالتقويم الهجري ، مالم ينص في عقد العمل أو لائحة تنظيم العمل على خلاف ذلك .


## Stage 04.1 — Separate Active and Repealed Legal Articles

This stage separates the cleaned legal dataset into active articles and repealed articles.

Active articles are allowed to enter the RAG indexing knowledge base, while repealed articles are archived separately and excluded from retrieval. This separation is essential in a legal RAG system because retrieving repealed provisions may lead to outdated or misleading answers.

The archived repealed articles remain available for transparency, documentation, and future legal review, but they are not used as retrievable knowledge in the experimental system.


In [6]:
# =========================================================
# Stage 04.1 - Separate Active and Repealed Legal Articles
# فصل المواد الفعالة عن المواد الملغاة
# =========================================================

# جميع المواد محفوظة في df_articles لأغراض التدقيق والتحليل.
# لكن الاسترجاع RAG يجب أن يستخدم المواد الفعالة فقط.

df_articles_active = df_articles[df_articles["article_status"] == "active"].copy()
df_articles_repealed = df_articles[df_articles["article_status"] == "repealed"].copy()

# وسم الاستخدام داخل المشروع
df_articles_active["rag_usage"] = "indexing_allowed_active_law"
df_articles_repealed["rag_usage"] = "archive_only_not_indexed"

# حفظ الملفات المنفصلة
active_articles_path = CLEAN_DIR / "clean_labor_law_articles_active_only_for_rag.csv"
repealed_articles_path = CLEAN_DIR / "clean_labor_law_articles_repealed_archive_not_indexed.csv"

df_articles_active.to_csv(active_articles_path, index=False, encoding="utf-8-sig")
df_articles_active.to_excel(CLEAN_DIR / "clean_labor_law_articles_active_only_for_rag.xlsx", index=False)

df_articles_repealed.to_csv(repealed_articles_path, index=False, encoding="utf-8-sig")
df_articles_repealed.to_excel(CLEAN_DIR / "clean_labor_law_articles_repealed_archive_not_indexed.xlsx", index=False)

# فحص سلامة الفصل
split_audit = pd.DataFrame({
    "dataset": [
        "All legal articles - cleaned audit dataset",
        "Active legal articles - allowed for RAG indexing",
        "Repealed legal articles - archive only, not indexed"
    ],
    "count": [
        len(df_articles),
        len(df_articles_active),
        len(df_articles_repealed)
    ],
    "rag_policy": [
        "retained_for_audit",
        "indexed",
        "excluded_from_index"
    ]
})

split_audit.to_csv(REPORTS_DIR / "legal_article_rag_policy_split.csv", index=False, encoding="utf-8-sig")
split_audit.to_excel(REPORTS_DIR / "legal_article_rag_policy_split.xlsx", index=False)

print("إجمالي مواد النظام في ملف التدقيق:", len(df_articles))
print("المواد الفعالة المستخدمة في RAG:", len(df_articles_active))
print("المواد الملغاة المحفوظة كأرشيف فقط:", len(df_articles_repealed))
print("مواد ملغاة داخل مواد RAG الفعالة:", int((df_articles_active["article_status"] == "repealed").sum()))

display(split_audit)

print("Saved active-only articles:", active_articles_path)
print("Saved repealed archive:", repealed_articles_path)


إجمالي مواد النظام في ملف التدقيق: 249
المواد الفعالة المستخدمة في RAG: 211
المواد الملغاة المحفوظة كأرشيف فقط: 38
مواد ملغاة داخل مواد RAG الفعالة: 0


,dataset,count,rag_policy
0,All legal articles - cleaned audit dataset,249,retained_for_audit
1,Active legal articles - allowed for RAG indexing,211,indexed
2,"Repealed legal articles - archive only, not indexed",38,excluded_from_index


Saved active-only articles: saudi_labor_law_voice_agent_project\02_clean\clean_labor_law_articles_active_only_for_rag.csv
Saved repealed archive: saudi_labor_law_voice_agent_project\02_clean\clean_labor_law_articles_repealed_archive_not_indexed.csv


## Stage 05 — Clean Classified HRSD FAQ Records

This stage cleans the classified FAQ dataset extracted from the HRSD website.

Each FAQ record is treated as a question-answer unit enriched with metadata such as beneficiary type, sector, subsite, and source URL. The cleaned FAQ records are later divided into indexing and evaluation subsets to ensure that the retrieval evaluation is performed on unseen questions.


In [7]:
# =========================================================
# Stage 05 - Clean Classified FAQ
# =========================================================

df_faq = df_faq_raw.copy()

required_faq_cols = [
    "question", "answer", "beneficiaries", "sector", "subsite", "page_url"
]

for col in required_faq_cols:
    df_faq = safe_col(df_faq, col, "")

for col in required_faq_cols:
    df_faq[col] = df_faq[col].apply(clean_basic_text)

# حذف الصفوف الفارغة أو غير المفيدة
before_filter = len(df_faq)
df_faq = df_faq[
    (df_faq["question"].str.len() > 5) &
    (df_faq["answer"].str.len() > 10)
].copy()

# Metadata
df_faq["source_type"] = "faq"
df_faq["source_name"] = "HRSD FAQ"

df_faq["legal_category"] = df_faq.apply(
    lambda r: clean_basic_text(
        f"{r['beneficiaries']} | {r['sector']} | {r['subsite']}"
    ).strip(" |"), axis=1
)

df_faq["document_unit_id"] = df_faq.apply(
    lambda r: make_hash(r["source_type"], r["question"], r["answer"]), axis=1
)

df_faq["text_for_indexing"] = df_faq.apply(
    lambda r: clean_basic_text(
        f"المصدر: {r['source_name']}\n"
        f"التصنيف: {r['legal_category']}\n"
        f"السؤال: {r['question']}\n"
        f"الإجابة: {r['answer']}"
    ), axis=1
)

df_faq["normalized_text"] = df_faq["text_for_indexing"].apply(normalize_arabic_for_search)

before_dedup = len(df_faq)
df_faq = df_faq.drop_duplicates(subset=["document_unit_id"]).reset_index(drop=True)

df_faq.to_csv(CLEAN_DIR / "clean_hrsd_faq_classified.csv", index=False, encoding="utf-8-sig")
df_faq.to_excel(CLEAN_DIR / "clean_hrsd_faq_classified.xlsx", index=False)

print("FAQ raw rows:", before_filter)
print("FAQ after empty filter:", before_dedup)
print("FAQ after dedup:", len(df_faq))

display(df_faq[[
    "question", "answer", "beneficiaries", "sector", "subsite"
]].head(10))

FAQ raw rows: 711
FAQ after empty filter: 703
FAQ after dedup: 703


,question,answer,beneficiaries,sector,subsite
0,هل تستخدم البوابة والانظمة التابعة لها الحلول الوطنية eSignature؟,"تقدم الوزارة خدمة التوقيع الالكتروني في اطار ادارة وتوثيق عقود العمل عبر منصة ""قوى"" يمكن للمنشآت انشاء وتوثيق وانهاء عقود العمل للموظفين بشكل الكرتوني بالكامل.",أصحاب عمل | أفراد | العمالة المنزلية | كبار السن,قطاع العمل,مركز الرياض للسياسات السلوكية
1,ماذا أفعل إذا تم إرجاع الطلب للمراجعة؟,الدخول إلى “مهامي” ← اختيار مهمة طلب مراجعة ← تعديل البيانات ← إعادة الإرسال.,التخصصات الاجتماعية | العمالة المنزلية | كبار السن,قطاع التنمية الاجتماعية,مركز الرياض للسياسات السلوكية
2,ماذا يحدث بعد إرسال الطلب؟,يتم تحويل الطلب إلى اللجنة للمراجعة ويتم إشعار المتقدم بحالة الطلب.,التخصصات الاجتماعية | العمالة المنزلية | كبار السن,قطاع التنمية الاجتماعية,مركز الرياض للسياسات السلوكية
3,كيف أقدم طلب تعديل فئة مهنية؟,من خلال تسجيل الدخول ← الخدمات ← اختيار “طلب تعديل الفئة المهنية” ← تعبئة النموذج ← إرسال الطلب.,التخصصات الاجتماعية | العمالة المنزلية | كبار السن,قطاع التنمية الاجتماعية,مركز الرياض للسياسات السلوكية
4,ما متطلبات التقديم؟,أن يكون المتقدم حاصل على ترخيص سابق.,التخصصات الاجتماعية | العمالة المنزلية | كبار السن,قطاع التنمية الاجتماعية,مركز الرياض للسياسات السلوكية
5,من يمكنه التقديم على خدمة تعديل الفئة المهنية؟,الممارس الحاصل على ترخيص من خلال خدمة التسجيل والتصنيف المهني.,التخصصات الاجتماعية | العمالة المنزلية | كبار السن,,مركز الرياض للسياسات السلوكية
6,هل يمكن للمتقدم أن يحصل على أكثر من تصنيف وبالتالي أكثر من رخصة؟,نعم، يمكن للمتقدم الحصول على أكثر من رخصة إذا كان يحمل مؤهلات أو دورات أو خبرات تؤهله لذلك.,التخصصات الاجتماعية | العمالة المنزلية | كبار السن,قطاع التنمية الاجتماعية,مركز الرياض للسياسات السلوكية
7,إذا رفض طلب التسجيل والتصنيف هل يحق لي استرداد رسوم الخدمة؟,نعم، يحق لك استرداد رسوم خدمة التسجيل والتصنيف إذا رفض الطلب.,التخصصات الاجتماعية | العمالة المنزلية | كبار السن,قطاع التنمية الاجتماعية,مركز الرياض للسياسات السلوكية
8,ماذا يحدث في حال رفض الطلب؟,يمكنك الاطلاع على سبب الرفض من خلال تفاصيل الطلب، ثم التقديم بطلب جديد بعد معالجة الأسباب.,التخصصات الاجتماعية | العمالة المنزلية | كبار السن,,مركز الرياض للسياسات السلوكية
9,هل يمكن إلغاء الفاتورة؟,نعم، يمكن إلغاء الفاتورة قبل السداد، مع ضرورة إدخال سبب الإلغاء.,التخصصات الاجتماعية | العمالة المنزلية | كبار السن,قطاع التنمية الاجتماعية,مركز الرياض للسياسات السلوكية


## Stage 05.5 — FAQ Domain Filtering for Labor-Law RAG

This stage filters the HRSD FAQ records to retain only questions that are relevant to the Saudi Labor Law voice agent.  

The raw FAQ file is not modified; instead, the notebook creates separate files for labor-related FAQ records, excluded non-labor FAQ records, and records requiring manual review.


In [8]:
# =========================================================
# Stage 05.5 - FAQ Domain Filtering for Labor-Law RAG
# Strict but safe domain filtering for labor-law and labor-service FAQ
# =========================================================

from IPython.display import display, Markdown
import pandas as pd
import numpy as np
import re
from pathlib import Path

# ---------------------------------------------------------
# 1) Select FAQ dataframe
# ---------------------------------------------------------

if "df_faq" in globals():
    faq_df = df_faq.copy()
elif "df_faq_raw" in globals():
    faq_df = df_faq_raw.copy()
else:
    raise NameError("df_faq أو df_faq_raw غير موجود. شغّل خلايا تحميل وتنظيف FAQ أولًا.")

required_cols = ["question", "answer"]
missing_cols = [c for c in required_cols if c not in faq_df.columns]

if missing_cols:
    raise KeyError(f"الأعمدة التالية غير موجودة في FAQ: {missing_cols}")

if "CLEAN_DIR" not in globals():
    CLEAN_DIR = Path("02_clean")

if "FINAL_DIR" not in globals():
    FINAL_DIR = Path("03_final")

before_domain_filter = len(faq_df)

# ---------------------------------------------------------
# 2) Helper functions
# ---------------------------------------------------------

def clean_text(value):
    if value is None:
        return ""

    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass

    value = str(value)
    value = re.sub(r"\s+", " ", value).strip()
    return value


def short_text(text, max_len=160):
    text = clean_text(text)
    return text[:max_len] + ("..." if len(text) > max_len else "")


def contains_any(text, keywords):
    text = clean_text(text)
    return any(k in text for k in keywords)


def count_keywords(text, keywords):
    text = clean_text(text)
    return sum(1 for k in keywords if k in text)


def hide_index_safe(styler):
    try:
        return styler.hide(axis="index")
    except Exception:
        try:
            return styler.hide_index()
        except Exception:
            return styler


def apply_style_compatible(styler, func, subset):
    if hasattr(styler, "map"):
        return styler.map(func, subset=subset)
    return styler.applymap(func, subset=subset)


def professional_table(df, caption="", width="100%", font_size="12px"):
    styler = (
        df.style
        .set_caption(caption)
        .set_table_styles([
            {
                "selector": "caption",
                "props": [
                    ("caption-side", "top"),
                    ("font-size", "17px"),
                    ("font-weight", "bold"),
                    ("color", "#1F4E78"),
                    ("text-align", "center"),
                    ("padding", "10px")
                ]
            },
            {
                "selector": "th",
                "props": [
                    ("background-color", "#1F4E78"),
                    ("color", "white"),
                    ("font-weight", "bold"),
                    ("text-align", "center"),
                    ("border", "1px solid #D9E2F3"),
                    ("padding", "8px"),
                    ("white-space", "normal")
                ]
            },
            {
                "selector": "td",
                "props": [
                    ("text-align", "center"),
                    ("border", "1px solid #D9E2F3"),
                    ("padding", "7px"),
                    ("vertical-align", "middle"),
                    ("white-space", "normal")
                ]
            },
            {
                "selector": "tbody tr:nth-child(even)",
                "props": [("background-color", "#F8FBFD")]
            },
            {
                "selector": "tbody tr:nth-child(odd)",
                "props": [("background-color", "#FFFFFF")]
            },
            {
                "selector": "table",
                "props": [
                    ("border-collapse", "collapse"),
                    ("width", width),
                    ("font-family", "Arial, Tahoma, sans-serif"),
                    ("font-size", font_size),
                    ("table-layout", "fixed"),
                    ("margin", "12px 0")
                ]
            }
        ])
    )

    return hide_index_safe(styler)


def color_domain_label(value):
    value = str(value)

    if value == "keep_labor_faq":
        return "background-color:#E2F0D9; color:#375623; font-weight:bold;"

    if value == "review_labor_related":
        return "background-color:#FFF2CC; color:#7F6000; font-weight:bold;"

    if value == "exclude_non_labor_faq":
        return "background-color:#FCE4D6; color:#9C0006; font-weight:bold;"

    return ""

# ---------------------------------------------------------
# 3) Keyword dictionaries
# ---------------------------------------------------------

# Manual exclusions for questions that are general service/request-management,
# even if their metadata contains labor-related beneficiaries.
MANUAL_EXCLUDE_QUESTIONS = [
    "هل يمكن تعديل الطلب بعد إرساله؟",
    "هل يمكن تعديل الطلب بعد ارساله؟",
    "هل يمكن تعديل الطلب بعد إرساله",
    "هل يمكن تعديل الطلب بعد ارساله",
]

# Hard exclusions must be checked only against question/answer,
# not against metadata such as subsite or beneficiaries.
HARD_EXCLUDE_TOPICS = [
    "مؤتمر العمل الدولي",
    "المؤتمر",
    "مؤتمر",
    "رؤية المملكة 2030",
    "رؤية 2030",
    "وسائل الإعلام",
    "الإعلانات",
    "الحوار الدولي",
    "مكانة الرياض",
    "فعاليات مؤتمر",
    "النتائج المتوقعة من مؤتمر",
    "مهمًا عالميًا",
]

# Strong direct labor-law / labor-service signals.
# Professional classification terms were intentionally removed from this list
# because they may belong to social-development professional licensing services.
STRONG_LABOR_QA_KEYWORDS = [
    "نظام العمل",
    "عقد العمل",
    "عقود العمل",
    "توثيق عقد",
    "توثيق عقود",
    "منصة قوى",
    "قوى",
    "صاحب العمل",
    "أصحاب العمل",
    "العامل",
    "العاملين",
    "الموظف",
    "الموظفين",
    "الأجر",
    "الأجور",
    "الراتب",
    "الرواتب",
    "حماية الأجور",
    "مكتب العمل",
    "شكوى عمالية",
    "الشكاوى العمالية",
    "التسوية الودية",
    "رخصة العمل",
    "رخص العمل",
    "نقل خدمات",
    "نقل الخدمة",
    "تأشيرة العمل",
    "التأشيرات المهنية",
    "عامل غير سعودي",
    "العمالة المنزلية",
    "عامل منزلي",
    "مساند",
    "ساعات العمل",
    "الإجازة السنوية",
    "الإجازة المرضية",
    "مكافأة نهاية الخدمة",
    "نهاية الخدمة",
    "فترة التجربة",
    "الاستقالة",
    "إنهاء العقد",
    "إنهاء عقد",
    "الفصل",
    "العمل الإضافي",
    "بلاغ تغيب",
    "نطاقات",
    "عقد العمل الموحد",
    "سند تنفيذي",
]

# Weak terms are not enough alone for direct RAG/evaluation inclusion.
WEAK_LABOR_QA_KEYWORDS = [
    "العمل",
    "الخدمة",
    "المهنة",
    "المهن",
    "المسمى المهني",
    "الفئة المهنية",
    "التصنيف المهني",
    "التسجيل والتصنيف المهني",
    "التوظيف",
    "منشأة",
    "المنشأة",
    "منشآت",
    "المنشآت",
    "الموارد البشرية",
    "وزارة الموارد البشرية",
]

# General / non-labor FAQ topics.
EXCLUDE_GENERAL_KEYWORDS = [
    "ماذا أفعل إذا تم إرجاع الطلب للمراجعة",
    "ماذا يحدث بعد إرسال الطلب",
    "حالة الطلب",
    "حالات الطلب",
    "إرجاع الطلب",
    "إعادة الإرسال",
    "صيغة وحجم الملفات",
    "حجم الملفات",
    "المرفقات",
    "لم يتم جلب بياناتي تلقائيًا",
    "حق الوصول إلى المعلومات",
    "الحصول على المعلومات",
    "المعلومات المستثناة",
    "الشفافية",
    "التخصصات الاجتماعية",
    "كبار السن",
    "الأشخاص ذوي الإعاقة",
    "ذوي الإعاقة",
    "الإعاقات",
    "المكافأة المالية للطلبة",
    "الطلبة الجامعيين",
    "التنمية الاجتماعية",
]

GENERIC_QUESTION_PATTERNS = [
    "ماذا أفعل",
    "ماذا يحدث",
    "ما هي حالة الطلب",
    "كيف أعرف حالة الطلب",
    "كيف أرفع ملف",
    "كيف أرفق",
    "ما صيغة",
    "ما حجم",
    "هل يمكن تعديل الطلب",
    "هل يمكن إلغاء الطلب",
    "ما المقصود",
]

# Special guard for professional-classification FAQ from social-development sector.
SOCIAL_PROFESSION_TERMS = [
    "التصنيف المهني",
    "الفئة المهنية",
    "التسجيل والتصنيف المهني",
    "الممارس",
    "الممارس الحاصل على ترخيص",
    "التخصصات الاجتماعية",
]

# ---------------------------------------------------------
# 4) Classification logic
# ---------------------------------------------------------

def classify_faq_domain(row):
    question = clean_text(row.get("question", ""))
    answer = clean_text(row.get("answer", ""))

    sector = clean_text(row.get("sector", ""))
    subsite = clean_text(row.get("subsite", ""))
    beneficiaries = clean_text(row.get("beneficiaries", ""))

    qa_text = f"{question} {answer}"
    metadata_text = f"{sector} {subsite} {beneficiaries}"
    all_text = f"{qa_text} {metadata_text}"

    # -----------------------------------------------------
    # Rule 0.0: Manual exclusion for known generic request-management FAQ
    # -----------------------------------------------------
    if question in MANUAL_EXCLUDE_QUESTIONS or contains_any(question, MANUAL_EXCLUDE_QUESTIONS):
        return (
            "exclude_non_labor_faq",
            "Manual exclusion: generic request-management FAQ, not a direct labor-law or labor-service question."
        )

    strong_qa_count = count_keywords(qa_text, STRONG_LABOR_QA_KEYWORDS)
    weak_qa_count = count_keywords(qa_text, WEAK_LABOR_QA_KEYWORDS)
    exclude_qa_count = count_keywords(qa_text, EXCLUDE_GENERAL_KEYWORDS)
    exclude_all_count = count_keywords(all_text, EXCLUDE_GENERAL_KEYWORDS)

    is_sector_work = "قطاع العمل" in sector
    is_sector_social = "قطاع التنمية الاجتماعية" in sector
    is_generic_question = contains_any(question, GENERIC_QUESTION_PATTERNS)

    # -----------------------------------------------------
    # Rule 0.1: Hard exclusion based on question/answer only
    # -----------------------------------------------------
    if contains_any(qa_text, HARD_EXCLUDE_TOPICS):
        return (
            "exclude_non_labor_faq",
            "Hard exclusion: conference, Vision 2030, media, or general ministry topic found in question/answer."
        )

    # -----------------------------------------------------
    # Rule 0.5: Social-development professional classification guard
    # -----------------------------------------------------
    if is_sector_social and contains_any(qa_text, SOCIAL_PROFESSION_TERMS):
        return (
            "review_labor_related",
            "Social-development professional classification FAQ; kept for manual review, not direct labor-law RAG."
        )

    # -----------------------------------------------------
    # Rule 1: Strong labor evidence in question/answer
    # -----------------------------------------------------
    if strong_qa_count >= 1:
        return (
            "keep_labor_faq",
            "Strong labor-law or labor-service evidence found in question/answer."
        )

    # -----------------------------------------------------
    # Rule 2: Work sector with labor-related wording, but not generic
    # -----------------------------------------------------
    if is_sector_work and weak_qa_count >= 1 and not is_generic_question:
        return (
            "keep_labor_faq",
            "Sector is labor and question/answer contains labor-related terms."
        )

    # -----------------------------------------------------
    # Rule 3: Social/general FAQ without strong labor evidence
    # -----------------------------------------------------
    if is_sector_social and strong_qa_count == 0:
        return (
            "exclude_non_labor_faq",
            "Social-development FAQ without strong labor-law evidence."
        )

    if is_generic_question and strong_qa_count == 0:
        return (
            "exclude_non_labor_faq",
            "Generic service-status question without labor-law evidence."
        )

    if exclude_qa_count > 0 and strong_qa_count == 0:
        return (
            "exclude_non_labor_faq",
            "Question/answer matches general or non-labor FAQ keywords."
        )

    if exclude_all_count > 0 and strong_qa_count == 0 and weak_qa_count == 0:
        return (
            "exclude_non_labor_faq",
            "Metadata indicates non-labor, social, or general FAQ."
        )

    # -----------------------------------------------------
    # Rule 4: Weak possible labor relation
    # -----------------------------------------------------
    if weak_qa_count >= 1 or is_sector_work:
        return (
            "review_labor_related",
            "Weak labor signal; requires manual review before indexing or evaluation."
        )

    # -----------------------------------------------------
    # Rule 5: Default exclusion
    # -----------------------------------------------------
    return (
        "exclude_non_labor_faq",
        "No clear labor-law or labor-service evidence in question/answer."
    )

# ---------------------------------------------------------
# 5) Apply classification
# ---------------------------------------------------------

classification = faq_df.apply(classify_faq_domain, axis=1)

faq_df["faq_domain_label"] = classification.apply(lambda x: x[0])
faq_df["faq_filter_reason"] = classification.apply(lambda x: x[1])

# Strict policy:
# Only keep_labor_faq enters RAG and Evaluation.
# review_labor_related is saved for audit only.
faq_df["keep_for_rag"] = faq_df["faq_domain_label"].eq("keep_labor_faq")
faq_df["keep_for_eval"] = faq_df["faq_domain_label"].eq("keep_labor_faq")

# ---------------------------------------------------------
# 6) Build filtered outputs
# ---------------------------------------------------------

df_faq_labor_clean = faq_df[faq_df["faq_domain_label"].eq("keep_labor_faq")].copy()
df_faq_labor_review = faq_df[faq_df["faq_domain_label"].eq("review_labor_related")].copy()
df_faq_excluded = faq_df[faq_df["faq_domain_label"].eq("exclude_non_labor_faq")].copy()

df_faq_for_rag = faq_df[faq_df["keep_for_rag"]].copy()
df_faq_for_eval = faq_df[faq_df["keep_for_eval"]].copy()

# Aliases for later stages
df_faq_domain_filtered = faq_df.copy()
df_faq_indexing_candidates = df_faq_for_rag.copy()
df_faq_evaluation_candidates = df_faq_for_eval.copy()

# ---------------------------------------------------------
# 7) Save outputs
# ---------------------------------------------------------

CLEAN_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)

df_faq_domain_filtered.to_csv(
    CLEAN_DIR / "clean_hrsd_faq_domain_classified.csv",
    index=False,
    encoding="utf-8-sig"
)

df_faq_labor_clean.to_csv(
    CLEAN_DIR / "clean_hrsd_faq_labor_related_strict.csv",
    index=False,
    encoding="utf-8-sig"
)

df_faq_for_rag.to_csv(
    FINAL_DIR / "hrsd_faq_for_rag_labor_filtered.csv",
    index=False,
    encoding="utf-8-sig"
)

df_faq_for_eval.to_csv(
    FINAL_DIR / "hrsd_faq_for_eval_labor_filtered.csv",
    index=False,
    encoding="utf-8-sig"
)

df_faq_labor_review.to_csv(
    FINAL_DIR / "hrsd_faq_manual_review_needed.csv",
    index=False,
    encoding="utf-8-sig"
)

df_faq_excluded.to_csv(
    FINAL_DIR / "hrsd_faq_excluded_non_labor.csv",
    index=False,
    encoding="utf-8-sig"
)

df_faq_labor_review.to_excel(
    FINAL_DIR / "hrsd_faq_manual_review_needed.xlsx",
    index=False
)

df_faq_excluded.to_excel(
    FINAL_DIR / "hrsd_faq_excluded_non_labor.xlsx",
    index=False
)

# ---------------------------------------------------------
# 8) Professional summary display
# ---------------------------------------------------------

display(Markdown("""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:14px;
    margin:12px 0;
    border-radius:6px;
">

<h3 style="color:#1F4E78; margin-top:0;">
Stage 05.5 — FAQ Domain Filtering for Labor-Law RAG
</h3>

<p>
This stage applies strict but safe domain filtering to the HRSD FAQ dataset. Only FAQ records with clear labor-law or labor-service evidence in the question or answer are allowed into the RAG knowledge base and FAQ evaluation set.
Hard exclusions are applied only to the question and answer text, not to metadata fields, to avoid over-excluding valid labor FAQ records.
</p>

<p>
Generic request-management questions and professional-classification records from the social-development sector are excluded from direct RAG indexing and evaluation, then preserved in audit files.
</p>

</div>
"""))

summary = (
    faq_df["faq_domain_label"]
    .value_counts()
    .reset_index()
)

summary.columns = ["FAQ Domain Label", "Records"]
summary["Percentage"] = (summary["Records"] / len(faq_df)).round(4)

summary_style = professional_table(
    summary,
    caption="Table 5.5 — FAQ Domain Filtering Summary",
    width="70%",
    font_size="12px"
)

summary_style = apply_style_compatible(
    summary_style,
    color_domain_label,
    subset=["FAQ Domain Label"]
)

display(summary_style)

print("FAQ records before domain filtering:", before_domain_filter)
print("FAQ records kept as strict labor FAQ:", len(df_faq_labor_clean))
print("FAQ records kept for RAG:", len(df_faq_for_rag))
print("FAQ records eligible for FAQ evaluation:", len(df_faq_for_eval))
print("FAQ records requiring manual review:", len(df_faq_labor_review))
print("FAQ records excluded as non-labor:", len(df_faq_excluded))

# ---------------------------------------------------------
# 9) Preview kept records
# ---------------------------------------------------------

preview_cols = [
    "question",
    "answer",
    "beneficiaries",
    "sector",
    "subsite",
    "faq_domain_label",
    "faq_filter_reason",
    "keep_for_rag",
    "keep_for_eval"
]

preview_cols = [c for c in preview_cols if c in faq_df.columns]

kept_preview = df_faq_for_rag[preview_cols].head(20).copy()

for col in ["question", "answer", "beneficiaries", "sector", "subsite", "faq_filter_reason"]:
    if col in kept_preview.columns:
        kept_preview[col] = kept_preview[col].apply(lambda x: short_text(x, 130))

kept_style = professional_table(
    kept_preview,
    caption="Table 5.6 — Preview of FAQ Records Kept for Labor-Law RAG",
    width="100%",
    font_size="11.5px"
)

if len(kept_preview) > 0:
    kept_style = apply_style_compatible(
        kept_style,
        color_domain_label,
        subset=["faq_domain_label"]
    )

    kept_rtl_cols = [
        c for c in ["question", "answer", "beneficiaries", "sector", "subsite", "faq_filter_reason"]
        if c in kept_preview.columns
    ]

    if kept_rtl_cols:
        kept_style = kept_style.set_properties(
            subset=kept_rtl_cols,
            **{
                "text-align": "right",
                "direction": "rtl",
                "line-height": "1.7"
            }
        )

display(kept_style)

# ---------------------------------------------------------
# 10) Preview manual-review records
# ---------------------------------------------------------

review_preview = df_faq_labor_review[preview_cols].head(20).copy()

for col in ["question", "answer", "beneficiaries", "sector", "subsite", "faq_filter_reason"]:
    if col in review_preview.columns:
        review_preview[col] = review_preview[col].apply(lambda x: short_text(x, 130))

review_style = professional_table(
    review_preview,
    caption="Table 5.7 — FAQ Records Requiring Manual Review",
    width="100%",
    font_size="11.5px"
)

if len(review_preview) > 0:
    review_style = apply_style_compatible(
        review_style,
        color_domain_label,
        subset=["faq_domain_label"]
    )

    review_rtl_cols = [
        c for c in ["question", "answer", "beneficiaries", "sector", "subsite", "faq_filter_reason"]
        if c in review_preview.columns
    ]

    if review_rtl_cols:
        review_style = review_style.set_properties(
            subset=review_rtl_cols,
            **{
                "text-align": "right",
                "direction": "rtl",
                "line-height": "1.7"
            }
        )

display(review_style)

# ---------------------------------------------------------
# 11) Preview excluded records
# ---------------------------------------------------------

excluded_preview = df_faq_excluded[preview_cols].head(20).copy()

for col in ["question", "answer", "beneficiaries", "sector", "subsite", "faq_filter_reason"]:
    if col in excluded_preview.columns:
        excluded_preview[col] = excluded_preview[col].apply(lambda x: short_text(x, 130))

excluded_style = professional_table(
    excluded_preview,
    caption="Table 5.8 — Preview of FAQ Records Excluded from Labor-Law RAG",
    width="100%",
    font_size="11.5px"
)

if len(excluded_preview) > 0:
    excluded_style = apply_style_compatible(
        excluded_style,
        color_domain_label,
        subset=["faq_domain_label"]
    )

    excluded_rtl_cols = [
        c for c in ["question", "answer", "beneficiaries", "sector", "subsite", "faq_filter_reason"]
        if c in excluded_preview.columns
    ]

    if excluded_rtl_cols:
        excluded_style = excluded_style.set_properties(
            subset=excluded_rtl_cols,
            **{
                "text-align": "right",
                "direction": "rtl",
                "line-height": "1.7"
            }
        )

display(excluded_style)

# ---------------------------------------------------------
# 12) Final confirmation
# ---------------------------------------------------------

display(Markdown(f"""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#E2F0D9;
    border-right:5px solid #375623;
    padding:14px;
    margin:12px 0;
    border-radius:6px;
">

<b>FAQ domain filtering completed successfully.</b><br>

FAQ records before filtering: <b>{before_domain_filter}</b><br>
FAQ records kept for RAG: <b>{len(df_faq_for_rag)}</b><br>
FAQ records kept for evaluation: <b>{len(df_faq_for_eval)}</b><br>
FAQ records requiring manual review: <b>{len(df_faq_labor_review)}</b><br>
FAQ records excluded: <b>{len(df_faq_excluded)}</b><br>

The filtered FAQ files were saved under <code>02_clean</code> and <code>03_final</code>.

</div>
"""))

print("Stage 05.5 completed successfully.")


<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:14px;
    margin:12px 0;
    border-radius:6px;
">

<h3 style="color:#1F4E78; margin-top:0;">
Stage 05.5 — FAQ Domain Filtering for Labor-Law RAG
</h3>

<p>
This stage applies strict but safe domain filtering to the HRSD FAQ dataset. Only FAQ records with clear labor-law or labor-service evidence in the question or answer are allowed into the RAG knowledge base and FAQ evaluation set.
Hard exclusions are applied only to the question and answer text, not to metadata fields, to avoid over-excluding valid labor FAQ records.
</p>

<p>
Generic request-management questions and professional-classification records from the social-development sector are excluded from direct RAG indexing and evaluation, then preserved in audit files.
</p>

</div>


FAQ Domain Label,Records,Percentage
keep_labor_faq,402,0.571800
exclude_non_labor_faq,247,0.351400
review_labor_related,54,0.076800


FAQ records before domain filtering: 703
FAQ records kept as strict labor FAQ: 402
FAQ records kept for RAG: 402
FAQ records eligible for FAQ evaluation: 402
FAQ records requiring manual review: 54
FAQ records excluded as non-labor: 247


question,answer,beneficiaries,sector,subsite,faq_domain_label,faq_filter_reason,keep_for_rag,keep_for_eval
هل تستخدم البوابة والانظمة التابعة لها الحلول الوطنية eSignature؟,"تقدم الوزارة خدمة التوقيع الالكتروني في اطار ادارة وتوثيق عقود العمل عبر منصة ""قوى"" يمكن للمنشآت انشاء وتوثيق وانهاء عقود العمل لل...",أصحاب عمل | أفراد | العمالة المنزلية | كبار السن,قطاع العمل,مركز الرياض للسياسات السلوكية,keep_labor_faq,Strong labor-law or labor-service evidence found in question/answer.,True,True
هل يطبق قرار توطين المهن بالتوازي مع نطاقات؟,نعم، يُطبق قرار توطين المهن على المهن المستهدفة بالقرار داخل المنشأة، وتُطبق العقوبات المنصوص عليها نظامًا بغض النظر عن نطاق المنش...,العمالة المنزلية | كبار السن,,مركز الرياض للسياسات السلوكية,keep_labor_faq,Strong labor-law or labor-service evidence found in question/answer.,True,True
هل يحق صاحب العمل في الاعتراض في عقد العمل الموثق سندًا تنفيذيًا؟,يحق لصاحب العمل الاعتراض على طلب التنفيذ عن طريق منصة ( ناجز )، حيث يتم إشعار صاحب العمل بطلب التنفيذ ويتم إعطاءه (5) أيام للاعترا...,أصحاب عمل | أفراد | العمالة المنزلية | كبار السن,قطاع العمل,مركز الرياض للسياسات السلوكية,keep_labor_faq,Strong labor-law or labor-service evidence found in question/answer.,True,True
كيف يتم التحقق من استلام العامل للأجر الذي سيتم التنفيذ عليه في عقد العمل الموثق سندًا تنفيذيًا؟,يتم التحقق إلكترونيا وفق بيانات برنامج حماية الأجور ( منصة مدد ) والذي يوثق دفع أجور العاملين بالمنشآت حسب تاريخ الاستحقاق ومقدار ...,أصحاب عمل | أفراد | العمالة المنزلية | كبار السن,قطاع العمل,مركز الرياض للسياسات السلوكية,keep_labor_faq,Strong labor-law or labor-service evidence found in question/answer.,True,True
كيف يتم رفع طلب التنفيذ في حال عدم التزام صاحب العمل بسداد الأجر في عقد العمل الموثق سندًا تنفيذيًا؟,يتم وفق التالي: الدخول على منصة ناجز التابعة لوزارة العدل. الدخول على منصة ناجز التابعة لوزارة العدل. عن طريق خيار التنفيذ (تقديم ...,أصحاب عمل | أفراد | العمالة المنزلية | كبار السن,قطاع العمل,مركز الرياض للسياسات السلوكية,keep_labor_faq,Strong labor-law or labor-service evidence found in question/answer.,True,True
متى يمكن للعامل التنفيذ على صاحب العمل في حال عدم التزامه بدفع الأجر لعقد العمل الموثق سندًا تنفيذيًا؟,في حال عدم السداد لكامل الاجر سيُمكن العامل من التنفيذ على صاحب العمل بعد مرور (30 يوماً) من تاريخ استحقاق الأجر. في حال عدم السدا...,أصحاب عمل | أفراد | العمالة المنزلية | كبار السن,قطاع العمل,مركز الرياض للسياسات السلوكية,keep_labor_faq,Strong labor-law or labor-service evidence found in question/answer.,True,True
ماهو بند العقد الخاضع للتنفيذ لعقد العمل الموثق سندًا تنفيذيًا؟,البند الخاضع للتنفيذ هو بند الأجر ويشمل ما يلي: الأجر الأساسي. بدل السكن حال وجوده. بدل النقل حال وجوده. إجمالي بدلات نقدية أخرى ح...,أصحاب عمل | أفراد | العمالة المنزلية | كبار السن,قطاع العمل,مركز الرياض للسياسات السلوكية,keep_labor_faq,Strong labor-law or labor-service evidence found in question/answer.,True,True
كيفية توثيق عقد العمل على نموذج عقد العمل الجديد (التنفيذي)؟,يتم التوثيق عبر منصة ( قوى ) حيث تتقدم المنشأة بطلب توثيق العقد أو تحديث العقود القائمة. يتم التوثيق عبر منصة ( قوى ) حيث تتقدم ال...,أصحاب عمل | أفراد | العمالة المنزلية | كبار السن,قطاع العمل,مركز الرياض للسياسات السلوكية,keep_labor_faq,Strong labor-law or labor-service evidence found in question/answer.,True,True
ماهي مراحل التطبيق على العقود الموثقة لعقد العمل الموثق سندًا تنفيذيًا؟,سيتم تطبيق نموذج عقد العمل الموحد الجديد (سند تنفيذي) بمنصة قوى على ثلاث مراحل: المرحلة الأولى: تشمل العقود الجديدة (العلاقة التعا...,أصحاب عمل | أفراد | العمالة المنزلية | كبار السن,قطاع العمل,مركز الرياض للسياسات السلوكية,keep_labor_faq,Strong labor-law or labor-service evidence found in question/answer.,True,True
ماهي متطلبات الاستفادة من مبادرة عقد العمل الموثق سندًا تنفيذيًا؟,وجود عقد عمل موثق بمنصة (قوى) وفق نموذج عقد العمل التنفيذي وجود عقد عمل موثق بمنصة (قوى) وفق نموذج عقد العمل التنفيذي أن يكون لعقد...,أصحاب عمل | أفراد | العمالة المنزلية | كبار السن,قطاع العمل,مركز الرياض للسياسات السلوكية,keep_labor_faq,Strong labor-law or labor-service evidence found in question/answer.,True,Tru

question,answer,beneficiaries,sector,subsite,faq_domain_label,faq_filter_reason,keep_for_rag,keep_for_eval
كيف أقدم طلب تعديل فئة مهنية؟,من خلال تسجيل الدخول ← الخدمات ← اختيار “طلب تعديل الفئة المهنية” ← تعبئة النموذج ← إرسال الطلب.,التخصصات الاجتماعية | العمالة المنزلية | كبار السن,قطاع التنمية الاجتماعية,مركز الرياض للسياسات السلوكية,review_labor_related,"Social-development professional classification FAQ; kept for manual review, not direct labor-law RAG.",False,False
من يمكنه التقديم على خدمة تعديل الفئة المهنية؟,الممارس الحاصل على ترخيص من خلال خدمة التسجيل والتصنيف المهني.,التخصصات الاجتماعية | العمالة المنزلية | كبار السن,,مركز الرياض للسياسات السلوكية,review_labor_related,Weak labor signal; requires manual review before indexing or evaluation.,False,False
هل الخدمة (التسجيل والتصنيف المهني) فورية؟,تقديم الطلب فوري، أما الطلب فيخضع للمراجعة.,التخصصات الاجتماعية | العمالة المنزلية | كبار السن,قطاع التنمية الاجتماعية,مركز الرياض للسياسات السلوكية,review_labor_related,"Social-development professional classification FAQ; kept for manual review, not direct labor-law RAG.",False,False
من يمكنه التقديم على الخدمة (التسجيل والتصنيف المهني)؟,يمكن لأي ممارس حاصل على مؤهل علمي أو دورة تدريبية في مجال التخصصات الاجتماعية التقديم على الخدمة عبر المنتج.,التخصصات الاجتماعية | العمالة المنزلية | كبار السن,قطاع التنمية الاجتماعية,مركز الرياض للسياسات السلوكية,review_labor_related,"Social-development professional classification FAQ; kept for manual review, not direct labor-law RAG.",False,False
ما هي خدمة التسجيل والتصنيف المهني؟,هي خدمة تتيح للممارس الاجتماعي التقديم إلكترونيًا للحصول على التسجيل والتصنيف المهني عبر منتج التخصصات الاجتماعية.,التخصصات الاجتماعية | العمالة المنزلية | كبار السن,قطاع التنمية الاجتماعية,مركز الرياض للسياسات السلوكية,review_labor_related,"Social-development professional classification FAQ; kept for manual review, not direct labor-law RAG.",False,False
متى ستبدأ الرقابة على منافذ الخدمة للمهنة الواردة في القرار؟,في التاريخ المحدد لتطبيق وتنفيذ القرار.,العمالة المنزلية | كبار السن,,مركز الرياض للسياسات السلوكية,review_labor_related,Weak labor signal; requires manual review before indexing or evaluation.,False,False
هل يطبق القرار على مسمى المهنة فقط، أو على العمل الفعلي للعامل؟,يتم تطبيق القرار على المسميات المهنية والعمل الفعلي للعامل.,العمالة المنزلية | كبار السن,,مركز الرياض للسياسات السلوكية,review_labor_related,Weak labor signal; requires manual review before indexing or evaluation.,False,False
ماهي أوقات العمل الرسمية للفروع؟,يمكنكم معرفة أوقات العمل الرسمية للفروع عن طريق دليل الفروع اضغط هنا.,العمالة المنزلية | كبار السن,,مركز الرياض للسياسات السلوكية,review_labor_related,Weak labor signal; requires manual review before indexing or evaluation.,False,False
هل توجد وظائف لايمكن التعاقد عليها لغير السعوديين؟,نعم و التي صدرت الأنظمة بحصر التوظيف عليها للسعوديين,أفراد | العمالة المنزلية | كبار السن,قطاع الخدمة المدنية (العام),مركز الرياض للسياسات السلوكية,review_labor_related,Weak labor signal; requires manual review before indexing or evaluation.,False,False
ماهي الإجراءات لطلب تحديث بيانات موظف؟,من خلال الدخول على الخدمة وتقديم طلب تحديث بيانات,أفراد | العمالة المنزلية | كبار السن,قطاع الخدمة المدنية (العام),مركز الرياض للسياسات السلوكية,review_labor_related,Weak labor signal; requires manual review before indexing or evaluation.,False,False


question,answer,beneficiaries,sector,subsite,faq_domain_label,faq_filter_reason,keep_for_rag,keep_for_eval
ماذا أفعل إذا تم إرجاع الطلب للمراجعة؟,الدخول إلى “مهامي” ← اختيار مهمة طلب مراجعة ← تعديل البيانات ← إعادة الإرسال.,التخصصات الاجتماعية | العمالة المنزلية | كبار السن,قطاع التنمية الاجتماعية,مركز الرياض للسياسات السلوكية,exclude_non_labor_faq,Social-development FAQ without strong labor-law evidence.,False,False
ماذا يحدث بعد إرسال الطلب؟,يتم تحويل الطلب إلى اللجنة للمراجعة ويتم إشعار المتقدم بحالة الطلب.,التخصصات الاجتماعية | العمالة المنزلية | كبار السن,قطاع التنمية الاجتماعية,مركز الرياض للسياسات السلوكية,exclude_non_labor_faq,Social-development FAQ without strong labor-law evidence.,False,False
ما متطلبات التقديم؟,أن يكون المتقدم حاصل على ترخيص سابق.,التخصصات الاجتماعية | العمالة المنزلية | كبار السن,قطاع التنمية الاجتماعية,مركز الرياض للسياسات السلوكية,exclude_non_labor_faq,Social-development FAQ without strong labor-law evidence.,False,False
هل يمكن للمتقدم أن يحصل على أكثر من تصنيف وبالتالي أكثر من رخصة؟,نعم، يمكن للمتقدم الحصول على أكثر من رخصة إذا كان يحمل مؤهلات أو دورات أو خبرات تؤهله لذلك.,التخصصات الاجتماعية | العمالة المنزلية | كبار السن,قطاع التنمية الاجتماعية,مركز الرياض للسياسات السلوكية,exclude_non_labor_faq,Social-development FAQ without strong labor-law evidence.,False,False
إذا رفض طلب التسجيل والتصنيف هل يحق لي استرداد رسوم الخدمة؟,نعم، يحق لك استرداد رسوم خدمة التسجيل والتصنيف إذا رفض الطلب.,التخصصات الاجتماعية | العمالة المنزلية | كبار السن,قطاع التنمية الاجتماعية,مركز الرياض للسياسات السلوكية,exclude_non_labor_faq,Social-development FAQ without strong labor-law evidence.,False,False
ماذا يحدث في حال رفض الطلب؟,يمكنك الاطلاع على سبب الرفض من خلال تفاصيل الطلب، ثم التقديم بطلب جديد بعد معالجة الأسباب.,التخصصات الاجتماعية | العمالة المنزلية | كبار السن,,مركز الرياض للسياسات السلوكية,exclude_non_labor_faq,Generic service-status question without labor-law evidence.,False,False
هل يمكن إلغاء الفاتورة؟,نعم، يمكن إلغاء الفاتورة قبل السداد، مع ضرورة إدخال سبب الإلغاء.,التخصصات الاجتماعية | العمالة المنزلية | كبار السن,قطاع التنمية الاجتماعية,مركز الرياض للسياسات السلوكية,exclude_non_labor_faq,Social-development FAQ without strong labor-law evidence.,False,False
كيف أعرف حالة طلبي؟,يمكنك متابعة حالة الطلب من خلال صفحة “طلباتي” في المنتج، كما تصلك إشعارات عبر: رسالة نصية (SMS) رسالة نصية (SMS) بريد إلكتروني بري...,التخصصات الاجتماعية | العمالة المنزلية | كبار السن,,مركز الرياض للسياسات السلوكية,exclude_non_labor_faq,Question/answer matches general or non-labor FAQ keywords.,False,False
ماذا يحدث بعد قبول الطلب؟,قد يتم: قبول مباشر وإصدار الرخصة قبول مباشر وإصدار الرخصة قبول مع اختبار قبول مع اختبار قبول مع اختبار ومقابلة قبول مع اختبار ومقا...,التخصصات الاجتماعية | العمالة المنزلية | كبار السن,قطاع التنمية الاجتماعية,مركز الرياض للسياسات السلوكية,exclude_non_labor_faq,Social-development FAQ without strong labor-law evidence.,False,False
ما هي حالات الطلب؟,تحت الإجراء تحت الإجراء مرفوض مرفوض مكتمل مكتمل,التخصصات الاجتماعية | العمالة المنزلية | كبار السن,قطاع التنمية الاجتماعية,مركز الرياض للسياسات السلوكية,exclude_non_labor_faq,Social-development FAQ without strong labor-law evidence.,False,False



<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#E2F0D9;
    border-right:5px solid #375623;
    padding:14px;
    margin:12px 0;
    border-radius:6px;
">

<b>FAQ domain filtering completed successfully.</b><br>

FAQ records before filtering: <b>703</b><br>
FAQ records kept for RAG: <b>402</b><br>
FAQ records kept for evaluation: <b>402</b><br>
FAQ records requiring manual review: <b>54</b><br>
FAQ records excluded: <b>247</b><br>

The filtered FAQ files were saved under <code>02_clean</code> and <code>03_final</code>.

</div>


Stage 05.5 completed successfully.


In [9]:
# =========================================================
# Locate FAQ Filtering Output Files
# تحديد أماكن ملفات FAQ الناتجة من الفلترة
# =========================================================

from pathlib import Path

files_to_check = [
    FINAL_DIR / "hrsd_faq_manual_review_needed.csv",
    FINAL_DIR / "hrsd_faq_excluded_non_labor.csv",
    FINAL_DIR / "hrsd_faq_for_rag_labor_filtered.csv",
    FINAL_DIR / "hrsd_faq_for_eval_labor_filtered.csv",
    CLEAN_DIR / "clean_hrsd_faq_labor_related_strict.csv",
]

for f in files_to_check:
    print("=" * 100)
    print(f.name)
    print("Exists:", f.exists())
    print("Full path:", f.resolve())

hrsd_faq_manual_review_needed.csv
Exists: True
Full path: C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\03_final\hrsd_faq_manual_review_needed.csv
hrsd_faq_excluded_non_labor.csv
Exists: True
Full path: C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\03_final\hrsd_faq_excluded_non_labor.csv
hrsd_faq_for_rag_labor_filtered.csv
Exists: True
Full path: C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\03_final\hrsd_faq_for_rag_labor_filtered.csv
hrsd_faq_for_eval_labor_filtered.csv
Exists: True
Full path: C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\03_final\hrsd_faq_for_eval_labor_filtered.csv
clean_hrsd_faq_labor_related_strict.csv
Exists: True
Full path: C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\02_clean\clean_hrsd_faq_labor_related_strict.csv


## Stage 06 — FAQ Indexing/Evaluation Split

This stage splits the cleaned FAQ dataset into two non-overlapping subsets:

1. `hrsd_faq_indexing_set.csv` — FAQ records allowed inside the experimental RAG knowledge base.
2. `hrsd_faq_evaluation_set.csv` — FAQ records held out exclusively for retrieval evaluation.

The split is deterministic and based on a hash of the document identifier. This makes the experiment reproducible across notebook runs while preventing the same FAQ question from appearing in both indexing and evaluation.


In [10]:
# =========================================================
# Stage 06 - FAQ Indexing/Evaluation Split to Avoid Leakage
# Safe split after Stage 05.5 domain filtering
# =========================================================

from IPython.display import display, Markdown
import pandas as pd
import numpy as np
import re
import hashlib
from pathlib import Path

# ---------------------------------------------------------
# 1) Configuration safety
# ---------------------------------------------------------

if "FAQ_EVAL_RATIO" not in globals():
    FAQ_EVAL_RATIO = 0.30

if "RANDOM_SEED" not in globals():
    RANDOM_SEED = 42

if "FINAL_DIR" not in globals():
    FINAL_DIR = Path("03_final")

FINAL_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# 2) Professional display helpers
# ---------------------------------------------------------

def clean_display_value(value):
    if value is None:
        return "N/A"
    try:
        if pd.isna(value):
            return "N/A"
    except Exception:
        pass

    value = str(value).strip()
    if value.lower() in ["", "nan", "none", "<na>", "n/a"]:
        return "N/A"

    return value


def short_text(text, max_len=150):
    text = clean_display_value(text)
    if text == "N/A":
        return "N/A"

    text = re.sub(r"\s+", " ", text).strip()
    return text[:max_len] + ("..." if len(text) > max_len else "")


def hide_index_safe(styler):
    try:
        return styler.hide(axis="index")
    except Exception:
        try:
            return styler.hide_index()
        except Exception:
            return styler


def apply_style_compatible(styler, func, subset):
    if hasattr(styler, "map"):
        return styler.map(func, subset=subset)
    return styler.applymap(func, subset=subset)


def professional_table(df, caption="", width="100%", font_size="12px"):
    styler = (
        df.style
        .set_caption(caption)
        .set_table_styles([
            {
                "selector": "caption",
                "props": [
                    ("caption-side", "top"),
                    ("font-size", "17px"),
                    ("font-weight", "bold"),
                    ("color", "#1F4E78"),
                    ("text-align", "center"),
                    ("padding", "10px")
                ]
            },
            {
                "selector": "th",
                "props": [
                    ("background-color", "#1F4E78"),
                    ("color", "white"),
                    ("font-weight", "bold"),
                    ("text-align", "center"),
                    ("border", "1px solid #D9E2F3"),
                    ("padding", "8px"),
                    ("white-space", "normal")
                ]
            },
            {
                "selector": "td",
                "props": [
                    ("text-align", "center"),
                    ("border", "1px solid #D9E2F3"),
                    ("padding", "7px"),
                    ("vertical-align", "middle"),
                    ("white-space", "normal")
                ]
            },
            {
                "selector": "tbody tr:nth-child(even)",
                "props": [("background-color", "#F8FBFD")]
            },
            {
                "selector": "tbody tr:nth-child(odd)",
                "props": [("background-color", "#FFFFFF")]
            },
            {
                "selector": "table",
                "props": [
                    ("border-collapse", "collapse"),
                    ("width", width),
                    ("font-family", "Arial, Tahoma, sans-serif"),
                    ("font-size", font_size),
                    ("table-layout", "fixed"),
                    ("margin", "12px 0")
                ]
            }
        ])
    )

    return hide_index_safe(styler)


def color_status(value):
    value = str(value)

    if value == "PASS":
        return "background-color:#E2F0D9; color:#375623; font-weight:bold;"

    if value == "WARN":
        return "background-color:#FFF2CC; color:#7F6000; font-weight:bold;"

    if value == "FAIL":
        return "background-color:#FCE4D6; color:#9C0006; font-weight:bold;"

    return ""


# ---------------------------------------------------------
# 3) Stage explanation
# ---------------------------------------------------------

display(Markdown("""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:14px;
    margin:12px 0;
    border-radius:6px;
">

<h3 style="color:#1F4E78; margin-top:0;">
Stage 06 — FAQ Indexing/Evaluation Split
</h3>

<p>
This stage splits the labor-filtered FAQ records into two non-overlapping subsets:
one subset for indexing inside the experimental RAG knowledge base, and one held-out subset for evaluation.
The split is performed after Stage 05.5 domain filtering, so no additional strict domain guard is applied here.
</p>

<p>
This prevents over-filtering and ensures that valid labor FAQ records are not accidentally removed from the evaluation set.
</p>

</div>
"""))

# ---------------------------------------------------------
# 4) Select the correct FAQ source from Stage 05.5
# ---------------------------------------------------------

if "df_faq_for_eval" in globals() and len(df_faq_for_eval) > 0:
    strict_faq = df_faq_for_eval.copy().reset_index(drop=True)
    split_source_name = "df_faq_for_eval from Stage 05.5"

elif "df_faq_labor_clean" in globals() and len(df_faq_labor_clean) > 0:
    strict_faq = df_faq_labor_clean.copy().reset_index(drop=True)
    split_source_name = "df_faq_labor_clean from Stage 05.5"

elif "df_faq_domain_filtered" in globals():
    strict_faq = df_faq_domain_filtered[
        df_faq_domain_filtered.get("keep_for_eval", False) == True
    ].copy().reset_index(drop=True)
    split_source_name = "df_faq_domain_filtered where keep_for_eval=True"

else:
    raise NameError(
        "لم أجد مخرجات Stage 05.5. شغّل Stage 05.5 أولًا حتى تتوفر df_faq_for_eval أو df_faq_labor_clean."
    )

assert len(strict_faq) > 0, (
    "No FAQ records are available for splitting after Stage 05.5. "
    "Review Stage 05.5 domain-filtering rules."
)

# ---------------------------------------------------------
# 5) Ensure required identifiers exist
# ---------------------------------------------------------

required_cols = ["question", "answer"]
missing_cols = [c for c in required_cols if c not in strict_faq.columns]

if missing_cols:
    raise KeyError(f"الأعمدة التالية غير موجودة في FAQ split source: {missing_cols}")

def make_document_unit_id(row):
    question = str(row.get("question", "")).strip()
    answer = str(row.get("answer", "")).strip()
    raw = f"{question}||{answer}"
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()

if "document_unit_id" not in strict_faq.columns:
    strict_faq["document_unit_id"] = strict_faq.apply(make_document_unit_id, axis=1)

strict_faq["document_unit_id"] = strict_faq["document_unit_id"].fillna("").astype(str)

missing_id_mask = strict_faq["document_unit_id"].str.strip().eq("")
if missing_id_mask.any():
    strict_faq.loc[missing_id_mask, "document_unit_id"] = strict_faq[missing_id_mask].apply(
        make_document_unit_id,
        axis=1
    )

# ---------------------------------------------------------
# 6) Exact duplicate removal before split
# ---------------------------------------------------------

before_dedup = len(strict_faq)

strict_faq["_question_answer_hash"] = strict_faq.apply(
    lambda row: hashlib.sha256(
        f"{str(row.get('question', '')).strip()}||{str(row.get('answer', '')).strip()}".encode("utf-8")
    ).hexdigest(),
    axis=1
)

duplicates_mask = strict_faq.duplicated(
    subset=["_question_answer_hash"],
    keep="first"
)

df_faq_duplicates_removed_stage06 = strict_faq[duplicates_mask].copy()

strict_faq = (
    strict_faq[~duplicates_mask]
    .drop(columns=["_question_answer_hash"], errors="ignore")
    .reset_index(drop=True)
)

after_dedup = len(strict_faq)
duplicates_removed = before_dedup - after_dedup

# ---------------------------------------------------------
# 7) Deterministic hash-based split
# ---------------------------------------------------------

def deterministic_eval_split_flag(document_unit_id, eval_ratio=0.30):
    """
    Returns True if record should be assigned to evaluation split.
    Deterministic across reruns.
    """
    document_unit_id = str(document_unit_id)
    h = hashlib.sha256(document_unit_id.encode("utf-8")).hexdigest()
    score = int(h[:12], 16) / float(16 ** 12)
    return score < eval_ratio


strict_faq["_is_eval_split"] = strict_faq["document_unit_id"].apply(
    lambda x: deterministic_eval_split_flag(x, FAQ_EVAL_RATIO)
)

df_faq_evaluation = strict_faq[strict_faq["_is_eval_split"]].copy()
df_faq_indexing = strict_faq[~strict_faq["_is_eval_split"]].copy()

# ---------------------------------------------------------
# 8) Safe fallback if the deterministic split becomes empty
# ---------------------------------------------------------

if len(df_faq_evaluation) == 0 and len(strict_faq) > 1:
    eval_n = max(1, int(round(len(strict_faq) * FAQ_EVAL_RATIO)))

    df_faq_evaluation = strict_faq.sample(
        n=eval_n,
        random_state=RANDOM_SEED
    ).copy()

    df_faq_indexing = strict_faq.drop(
        index=df_faq_evaluation.index
    ).copy()

if len(df_faq_indexing) == 0 and len(strict_faq) > 1:
    index_n = max(1, len(strict_faq) - 1)

    df_faq_indexing = strict_faq.sample(
        n=index_n,
        random_state=RANDOM_SEED
    ).copy()

    df_faq_evaluation = strict_faq.drop(
        index=df_faq_indexing.index
    ).copy()

df_faq_indexing = (
    df_faq_indexing
    .drop(columns=["_is_eval_split"], errors="ignore")
    .reset_index(drop=True)
)

df_faq_evaluation = (
    df_faq_evaluation
    .drop(columns=["_is_eval_split"], errors="ignore")
    .reset_index(drop=True)
)

assert len(df_faq_indexing) > 0, "FAQ indexing set is empty after split."
assert len(df_faq_evaluation) > 0, "FAQ evaluation set is empty after split."

# ---------------------------------------------------------
# 9) Mark split role
# ---------------------------------------------------------

df_faq_indexing["is_eval_record"] = False
df_faq_evaluation["is_eval_record"] = True

df_faq_indexing["faq_split"] = "indexing"
df_faq_evaluation["faq_split"] = "evaluation"

# ---------------------------------------------------------
# 10) Leakage checks
# ---------------------------------------------------------

indexing_ids = set(df_faq_indexing["document_unit_id"].astype(str))
evaluation_ids = set(df_faq_evaluation["document_unit_id"].astype(str))

overlap_ids = indexing_ids.intersection(evaluation_ids)
faq_id_leakage_count = len(overlap_ids)

assert faq_id_leakage_count == 0, (
    f"FAQ leakage detected between indexing and evaluation sets: {faq_id_leakage_count}"
)

actual_eval_ratio = len(df_faq_evaluation) / max(len(strict_faq), 1)

# ---------------------------------------------------------
# 11) Save output files
# ---------------------------------------------------------

df_faq_indexing.to_csv(
    FINAL_DIR / "hrsd_faq_indexing_set.csv",
    index=False,
    encoding="utf-8-sig"
)

df_faq_evaluation.to_csv(
    FINAL_DIR / "hrsd_faq_evaluation_set.csv",
    index=False,
    encoding="utf-8-sig"
)

df_faq_indexing.to_excel(
    FINAL_DIR / "hrsd_faq_indexing_set.xlsx",
    index=False
)

df_faq_evaluation.to_excel(
    FINAL_DIR / "hrsd_faq_evaluation_set.xlsx",
    index=False
)

if len(df_faq_duplicates_removed_stage06) > 0:
    df_faq_duplicates_removed_stage06.to_csv(
        FINAL_DIR / "hrsd_faq_duplicates_removed_stage06.csv",
        index=False,
        encoding="utf-8-sig"
    )

    df_faq_duplicates_removed_stage06.to_excel(
        FINAL_DIR / "hrsd_faq_duplicates_removed_stage06.xlsx",
        index=False
    )

# ---------------------------------------------------------
# 12) Summary tables
# ---------------------------------------------------------

split_summary = pd.DataFrame([
    {
        "Component": "FAQ source used for split",
        "Value": split_source_name,
        "Status": "PASS",
        "Interpretation": "Stage 06 uses the already domain-filtered FAQ records from Stage 05.5."
    },
    {
        "Component": "FAQ records before duplicate removal",
        "Value": before_dedup,
        "Status": "PASS",
        "Interpretation": "Labor-filtered FAQ records available before exact duplicate removal."
    },
    {
        "Component": "Exact duplicates removed before split",
        "Value": duplicates_removed,
        "Status": "PASS" if duplicates_removed >= 0 else "FAIL",
        "Interpretation": "Exact duplicate FAQ records were removed using question-answer hashing."
    },
    {
        "Component": "FAQ indexing records",
        "Value": len(df_faq_indexing),
        "Status": "PASS" if len(df_faq_indexing) > 0 else "FAIL",
        "Interpretation": "FAQ records allowed inside the experimental RAG knowledge base."
    },
    {
        "Component": "FAQ evaluation records",
        "Value": len(df_faq_evaluation),
        "Status": "PASS" if len(df_faq_evaluation) > 0 else "FAIL",
        "Interpretation": "Held-out FAQ records reserved only for retrieval evaluation."
    },
    {
        "Component": "Actual FAQ evaluation ratio",
        "Value": round(actual_eval_ratio, 4),
        "Status": "PASS",
        "Interpretation": "Actual split ratio after deterministic hashing and safety fallback."
    },
    {
        "Component": "FAQ ID overlap between indexing and evaluation",
        "Value": faq_id_leakage_count,
        "Status": "PASS" if faq_id_leakage_count == 0 else "FAIL",
        "Interpretation": "Must be zero to prevent exact ID leakage."
    }
])

summary_style = professional_table(
    split_summary,
    caption="Table 6.1 — FAQ Split Summary and Leakage Safety Checks",
    width="100%",
    font_size="12px"
)

summary_style = apply_style_compatible(
    summary_style,
    color_status,
    subset=["Status"]
)

display(summary_style)

# ---------------------------------------------------------
# 13) Split distribution table
# ---------------------------------------------------------

distribution_df = pd.DataFrame([
    {
        "FAQ Split": "Indexing Set",
        "Records": len(df_faq_indexing),
        "Percentage": round(len(df_faq_indexing) / max(len(strict_faq), 1) * 100, 2),
        "Usage": "Included in experimental RAG knowledge base"
    },
    {
        "FAQ Split": "Evaluation Set",
        "Records": len(df_faq_evaluation),
        "Percentage": round(len(df_faq_evaluation) / max(len(strict_faq), 1) * 100, 2),
        "Usage": "Held out for retrieval evaluation only"
    }
])

display(
    professional_table(
        distribution_df,
        caption="Table 6.2 — FAQ Indexing/Evaluation Split Distribution",
        width="80%",
        font_size="12px"
    )
)

# ---------------------------------------------------------
# 14) Preview tables
# ---------------------------------------------------------

preview_cols = [
    "question",
    "answer",
    "beneficiaries",
    "sector",
    "subsite",
    "document_unit_id",
    "faq_split",
    "is_eval_record"
]

preview_cols = [c for c in preview_cols if c in df_faq_indexing.columns]

indexing_preview = df_faq_indexing[preview_cols].head(8).copy()
evaluation_preview = df_faq_evaluation[preview_cols].head(8).copy()

for df_preview in [indexing_preview, evaluation_preview]:
    for col in ["question", "answer", "beneficiaries", "sector", "subsite"]:
        if col in df_preview.columns:
            df_preview[col] = df_preview[col].apply(lambda x: short_text(x, 120))

indexing_style = professional_table(
    indexing_preview,
    caption="Table 6.3 — Preview of FAQ Indexing Set",
    width="100%",
    font_size="11.5px"
)

evaluation_style = professional_table(
    evaluation_preview,
    caption="Table 6.4 — Preview of FAQ Evaluation Set",
    width="100%",
    font_size="11.5px"
)

for style_obj, df_preview in [(indexing_style, indexing_preview), (evaluation_style, evaluation_preview)]:
    rtl_cols = [
        c for c in ["question", "answer", "beneficiaries", "sector", "subsite"]
        if c in df_preview.columns
    ]

    if rtl_cols:
        style_obj = style_obj.set_properties(
            subset=rtl_cols,
            **{
                "text-align": "right",
                "direction": "rtl",
                "line-height": "1.7"
            }
        )

    if "document_unit_id" in df_preview.columns:
        style_obj = style_obj.set_properties(
            subset=["document_unit_id"],
            **{
                "text-align": "left",
                "direction": "ltr",
                "font-size": "10px"
            }
        )

    display(style_obj)

# ---------------------------------------------------------
# 15) Final confirmation
# ---------------------------------------------------------

display(Markdown(f"""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#E2F0D9;
    border-right:5px solid #375623;
    padding:14px;
    margin:12px 0;
    border-radius:6px;
">

<b>Stage 06 completed successfully.</b><br>

Strict labor FAQ records used for split: <b>{len(strict_faq)}</b><br>
FAQ indexing set: <b>{len(df_faq_indexing)}</b><br>
FAQ evaluation set: <b>{len(df_faq_evaluation)}</b><br>
Actual FAQ evaluation ratio: <b>{round(actual_eval_ratio, 4)}</b><br>
Exact FAQ leakage between indexing and evaluation: <b>{faq_id_leakage_count}</b><br>

Files saved:<br>
<code>{FINAL_DIR / "hrsd_faq_indexing_set.csv"}</code><br>
<code>{FINAL_DIR / "hrsd_faq_evaluation_set.csv"}</code>

</div>
"""))

print("Stage 06 completed successfully.")
print("FAQ indexing set:", len(df_faq_indexing))
print("FAQ evaluation set:", len(df_faq_evaluation))
print("Actual FAQ evaluation ratio:", round(actual_eval_ratio, 4))
print("FAQ ID leakage:", faq_id_leakage_count)
print("Saved:", FINAL_DIR / "hrsd_faq_indexing_set.csv")
print("Saved:", FINAL_DIR / "hrsd_faq_evaluation_set.csv")


<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:14px;
    margin:12px 0;
    border-radius:6px;
">

<h3 style="color:#1F4E78; margin-top:0;">
Stage 06 — FAQ Indexing/Evaluation Split
</h3>

<p>
This stage splits the labor-filtered FAQ records into two non-overlapping subsets:
one subset for indexing inside the experimental RAG knowledge base, and one held-out subset for evaluation.
The split is performed after Stage 05.5 domain filtering, so no additional strict domain guard is applied here.
</p>

<p>
This prevents over-filtering and ensures that valid labor FAQ records are not accidentally removed from the evaluation set.
</p>

</div>


Component,Value,Status,Interpretation
FAQ source used for split,df_faq_for_eval from Stage 05.5,PASS,Stage 06 uses the already domain-filtered FAQ records from Stage 05.5.
FAQ records before duplicate removal,402,PASS,Labor-filtered FAQ records available before exact duplicate removal.
Exact duplicates removed before split,0,PASS,Exact duplicate FAQ records were removed using question-answer hashing.
FAQ indexing records,278,PASS,FAQ records allowed inside the experimental RAG knowledge base.
FAQ evaluation records,124,PASS,Held-out FAQ records reserved only for retrieval evaluation.
Actual FAQ evaluation ratio,0.308500,PASS,Actual split ratio after deterministic hashing and safety fallback.
FAQ ID overlap between indexing and evaluation,0,PASS,Must be zero to prevent exact ID leakage.


FAQ Split,Records,Percentage,Usage
Indexing Set,278,69.150000,Included in experimental RAG knowledge base
Evaluation Set,124,30.850000,Held out for retrieval evaluation only


question,answer,beneficiaries,sector,subsite,document_unit_id,faq_split,is_eval_record
هل تستخدم البوابة والانظمة التابعة لها الحلول الوطنية eSignature؟,"تقدم الوزارة خدمة التوقيع الالكتروني في اطار ادارة وتوثيق عقود العمل عبر منصة ""قوى"" يمكن للمنشآت انشاء وتوثيق وانهاء عقو...",أصحاب عمل | أفراد | العمالة المنزلية | كبار السن,قطاع العمل,مركز الرياض للسياسات السلوكية,87948be8261685e657502e8c023d087a27fb13927373f98568f379a8d20ff856,indexing,False
هل يطبق قرار توطين المهن بالتوازي مع نطاقات؟,نعم، يُطبق قرار توطين المهن على المهن المستهدفة بالقرار داخل المنشأة، وتُطبق العقوبات المنصوص عليها نظامًا بغض النظر عن ...,العمالة المنزلية | كبار السن,N/A,مركز الرياض للسياسات السلوكية,f6cb1c912ccbb571bd042480fb270bb4a2a348d19b19998dafe25511b00601ef,indexing,False
كيف يتم التحقق من استلام العامل للأجر الذي سيتم التنفيذ عليه في عقد العمل الموثق سندًا تنفيذيًا؟,يتم التحقق إلكترونيا وفق بيانات برنامج حماية الأجور ( منصة مدد ) والذي يوثق دفع أجور العاملين بالمنشآت حسب تاريخ الاستحق...,أصحاب عمل | أفراد | العمالة المنزلية | كبار السن,قطاع العمل,مركز الرياض للسياسات السلوكية,529eb529660f41db8b40d522f7914d1ea87fc055ca2c0bd6017d77fdd2d80aa3,indexing,False
متى يمكن للعامل التنفيذ على صاحب العمل في حال عدم التزامه بدفع الأجر لعقد العمل الموثق سندًا تنفيذيًا؟,في حال عدم السداد لكامل الاجر سيُمكن العامل من التنفيذ على صاحب العمل بعد مرور (30 يوماً) من تاريخ استحقاق الأجر. في حال...,أصحاب عمل | أفراد | العمالة المنزلية | كبار السن,قطاع العمل,مركز الرياض للسياسات السلوكية,17d98bd155a20df1f783c2bdece8e48a0c9691c9d26a364b52e3622d27c99673,indexing,False
كيفية توثيق عقد العمل على نموذج عقد العمل الجديد (التنفيذي)؟,يتم التوثيق عبر منصة ( قوى ) حيث تتقدم المنشأة بطلب توثيق العقد أو تحديث العقود القائمة. يتم التوثيق عبر منصة ( قوى ) حي...,أصحاب عمل | أفراد | العمالة المنزلية | كبار السن,قطاع العمل,مركز الرياض للسياسات السلوكية,7ea405a8ffadf308f00ac9d24e5af0365a1509267866717162d256c66dc9e5ca,indexing,False
كيف يمكن الاستفادة من الخدمة؟,بواسطة قناة تقديم الخدمة من خلال الضغط على زر ابدأ الخدمة,أفراد | العمالة المنزلية | المواطنين | كبار السن | مقيمين,قطاع التنمية الاجتماعية | قطاع العمل,مركز الرياض للسياسات السلوكية,25e5c8ff51519e02c26e10002f1d90ba6b0ea02062d05961e861366f1865274b,indexing,False
كيف أعرف ما إذا كانت منشأتي ملزمة بتطبيق برنامج حماية الأجور؟,يمكنك معرفة ذلك عبر متابعة المصادر والحسابات الرسمية لوزارة الموارد البشرية والتنمية الاجتماعية.,العمالة المنزلية | كبار السن,N/A,مركز الرياض للسياسات السلوكية,2edd67f62b6108dad9a62ceaeac4e7fcc5c617562e84f22a3d81d0fe89edfd64,indexing,False
هل أستطيع رفع ملفات حماية الأجور عبر موقع وزارة الموارد البشرية والتنمية الاجتماعية للخدمات الإلكترونية؟,يدخل المستخدم إلى برنامج حماية الأجور فقط عن طريق منصة مُدد، بعد أن كان موقع الخدمات الإلكترونية لوزارة الموارد البشرية ...,العمالة المنزلية | كبار السن,N/A,مركز الرياض للسياسات السلوكية,53612f8f879579506fe457514d3af7812925a3583886242c3d2fcf62fefd19a9,indexing,False


question,answer,beneficiaries,sector,subsite,document_unit_id,faq_split,is_eval_record
هل يحق صاحب العمل في الاعتراض في عقد العمل الموثق سندًا تنفيذيًا؟,يحق لصاحب العمل الاعتراض على طلب التنفيذ عن طريق منصة ( ناجز )، حيث يتم إشعار صاحب العمل بطلب التنفيذ ويتم إعطاءه (5) أي...,أصحاب عمل | أفراد | العمالة المنزلية | كبار السن,قطاع العمل,مركز الرياض للسياسات السلوكية,004ea50cd2e3b50e605b6d0f769d2fb418a83490050e46372e0a4ce04446c378,evaluation,True
كيف يتم رفع طلب التنفيذ في حال عدم التزام صاحب العمل بسداد الأجر في عقد العمل الموثق سندًا تنفيذيًا؟,يتم وفق التالي: الدخول على منصة ناجز التابعة لوزارة العدل. الدخول على منصة ناجز التابعة لوزارة العدل. عن طريق خيار التنف...,أصحاب عمل | أفراد | العمالة المنزلية | كبار السن,قطاع العمل,مركز الرياض للسياسات السلوكية,001715202ff48d357db0ee5b5dc2f8992b5a1cec7389155a11cad6d876f37fc6,evaluation,True
ماهو بند العقد الخاضع للتنفيذ لعقد العمل الموثق سندًا تنفيذيًا؟,البند الخاضع للتنفيذ هو بند الأجر ويشمل ما يلي: الأجر الأساسي. بدل السكن حال وجوده. بدل النقل حال وجوده. إجمالي بدلات نق...,أصحاب عمل | أفراد | العمالة المنزلية | كبار السن,قطاع العمل,مركز الرياض للسياسات السلوكية,1f0a32b43d1d888523da1db089e2b5f6310977c6d4065dff65bf39462c298fd2,evaluation,True
ماهي مراحل التطبيق على العقود الموثقة لعقد العمل الموثق سندًا تنفيذيًا؟,سيتم تطبيق نموذج عقد العمل الموحد الجديد (سند تنفيذي) بمنصة قوى على ثلاث مراحل: المرحلة الأولى: تشمل العقود الجديدة (الع...,أصحاب عمل | أفراد | العمالة المنزلية | كبار السن,قطاع العمل,مركز الرياض للسياسات السلوكية,c37bc793ca18cedb26c8bc4e19db4b07f676501c117336fea4847f8c92bb7e80,evaluation,True
ماهي متطلبات الاستفادة من مبادرة عقد العمل الموثق سندًا تنفيذيًا؟,وجود عقد عمل موثق بمنصة (قوى) وفق نموذج عقد العمل التنفيذي وجود عقد عمل موثق بمنصة (قوى) وفق نموذج عقد العمل التنفيذي أن...,أصحاب عمل | أفراد | العمالة المنزلية | كبار السن,قطاع العمل,مركز الرياض للسياسات السلوكية,46001ae4bf6cc21f527a78e9a6a46b5913de3ecff09d71a32feb88f8a6ccaaa9,evaluation,True
ما هي الجهة المشرعة لبرنامج حماية الأجور؟,وزارة الموارد البشرية والتنمية الاجتماعية.,العمالة المنزلية | كبار السن,N/A,مركز الرياض للسياسات السلوكية,78f881c6ac6d11831761761dbf7216c07b31b6dee228ccdb6e04ad4debb0bb6c,evaluation,True
ما هي أهداف برنامج حماية الأجور؟,تحسين العلاقة التعاقدية بين صاحب العمل والعامل، وضمان حقوق العمال والموظفين ودفع رواتبهم وفقاً للشروط المتفق عليها في عق...,العمالة المنزلية | كبار السن,N/A,مركز الرياض للسياسات السلوكية,b0ebcab351b4739f0566a90080daa23d875b475bac36659a0088539697b8a266,evaluation,True
ما هو تاريخ بداية العقد لمن أكمل 5 سنوات وكيف تحسب فترة الخدمة السابقة لتسجيل العقد؟,يجب إدخال تاريخ المباشرة كما هو مذكور في أول عقد للموظف مع المنشأة.,أصحاب عمل | أفراد | العمالة المنزلية | كبار السن,قطاع العمل,مركز الرياض للسياسات السلوكية,0d597df1b233e5d84c1e7ddfd9353da00659d88fb934af35eaa2b25a57140b2c,evaluation,True



<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#E2F0D9;
    border-right:5px solid #375623;
    padding:14px;
    margin:12px 0;
    border-radius:6px;
">

<b>Stage 06 completed successfully.</b><br>

Strict labor FAQ records used for split: <b>402</b><br>
FAQ indexing set: <b>278</b><br>
FAQ evaluation set: <b>124</b><br>
Actual FAQ evaluation ratio: <b>0.3085</b><br>
Exact FAQ leakage between indexing and evaluation: <b>0</b><br>

Files saved:<br>
<code>saudi_labor_law_voice_agent_project\03_final\hrsd_faq_indexing_set.csv</code><br>
<code>saudi_labor_law_voice_agent_project\03_final\hrsd_faq_evaluation_set.csv</code>

</div>


Stage 06 completed successfully.
FAQ indexing set: 278
FAQ evaluation set: 124
Actual FAQ evaluation ratio: 0.3085
FAQ ID leakage: 0
Saved: saudi_labor_law_voice_agent_project\03_final\hrsd_faq_indexing_set.csv
Saved: saudi_labor_law_voice_agent_project\03_final\hrsd_faq_evaluation_set.csv


In [11]:
from IPython.display import display, Markdown

display(Markdown("""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:16px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:16px;
    margin:14px 0;
    border-radius:6px;
">

<h3 style="color:#1F4E78; margin-top:0;">
Interpretation — FAQ Indexing/Evaluation Split
</h3>

<p>
تعرض هذه المرحلة نتيجة تقسيم سجلات FAQ المرتبطة بقطاع العمل إلى مجموعتين منفصلتين: مجموعة مخصصة للفهرسة داخل قاعدة المعرفة التجريبية، ومجموعة محجوزة للتقييم فقط. تم تنفيذ هذا التقسيم بعد فلترة المجال في Stage 05.5، لذلك لم يتم تطبيق فلترة مجال إضافية في هذه المرحلة لتجنب حذف أسئلة صحيحة من مجموعة التقييم.
</p>

<p>
بلغ عدد سجلات FAQ الصالحة بعد فلترة المجال <b>402</b> سجلًا. تم تخصيص <b>278</b> سجلًا لمجموعة الفهرسة، بينما تم حجز <b>124</b> سجلًا لمجموعة التقييم. وبذلك بلغت نسبة التقييم الفعلية <b>30.85%</b>، وهي قريبة جدًا من النسبة المستهدفة البالغة تقريبًا <b>30%</b>.
</p>

<p>
أظهرت اختبارات السلامة أن عدد السجلات المكررة المحذوفة قبل التقسيم يساوي <b>0</b>، كما أن التداخل بين معرفات سجلات الفهرسة وسجلات التقييم يساوي <b>0</b>. وهذا يعني عدم وجود تسريب مباشر بين مجموعة الفهرسة ومجموعة التقييم على مستوى <code>document_unit_id</code>.
</p>

<div style="
    background-color:#E2F0D9;
    border-right:4px solid #375623;
    padding:12px;
    margin-top:12px;
    border-radius:5px;
">
<b>Conclusion:</b><br>
The FAQ split is valid and methodologically sound. The evaluation subset is held out from the experimental RAG knowledge base, which supports fair retrieval evaluation in the next stage.
</div>

</div>
"""))


<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:16px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:16px;
    margin:14px 0;
    border-radius:6px;
">

<h3 style="color:#1F4E78; margin-top:0;">
Interpretation — FAQ Indexing/Evaluation Split
</h3>

<p>
تعرض هذه المرحلة نتيجة تقسيم سجلات FAQ المرتبطة بقطاع العمل إلى مجموعتين منفصلتين: مجموعة مخصصة للفهرسة داخل قاعدة المعرفة التجريبية، ومجموعة محجوزة للتقييم فقط. تم تنفيذ هذا التقسيم بعد فلترة المجال في Stage 05.5، لذلك لم يتم تطبيق فلترة مجال إضافية في هذه المرحلة لتجنب حذف أسئلة صحيحة من مجموعة التقييم.
</p>

<p>
بلغ عدد سجلات FAQ الصالحة بعد فلترة المجال <b>402</b> سجلًا. تم تخصيص <b>278</b> سجلًا لمجموعة الفهرسة، بينما تم حجز <b>124</b> سجلًا لمجموعة التقييم. وبذلك بلغت نسبة التقييم الفعلية <b>30.85%</b>، وهي قريبة جدًا من النسبة المستهدفة البالغة تقريبًا <b>30%</b>.
</p>

<p>
أظهرت اختبارات السلامة أن عدد السجلات المكررة المحذوفة قبل التقسيم يساوي <b>0</b>، كما أن التداخل بين معرفات سجلات الفهرسة وسجلات التقييم يساوي <b>0</b>. وهذا يعني عدم وجود تسريب مباشر بين مجموعة الفهرسة ومجموعة التقييم على مستوى <code>document_unit_id</code>.
</p>

<div style="
    background-color:#E2F0D9;
    border-right:4px solid #375623;
    padding:12px;
    margin-top:12px;
    border-radius:5px;
">
<b>Conclusion:</b><br>
The FAQ split is valid and methodologically sound. The evaluation subset is held out from the experimental RAG knowledge base, which supports fair retrieval evaluation in the next stage.
</div>

</div>


### Interpretation — FAQ Split and Leakage Prevention after Domain Filtering

The FAQ split is performed only after domain filtering. This means that the retrieval experiment does not evaluate general ministry or social-development FAQ records that are outside the labor-law voice-agent scope.

Strict labor-related FAQ records are divided into indexing and held-out evaluation subsets using a deterministic hash-based split. The held-out FAQ evaluation records are excluded from the experimental RAG knowledge base, which prevents data leakage and makes the retrieval evaluation more reliable.

FAQ records marked for manual review may be included in the indexing set to improve coverage, but they are excluded from the quantitative FAQ evaluation until they are manually approved.

## Stage 06.5 — Content-Level Leakage Detection (Near-Duplicate FAQ)

<div dir="rtl" style="text-align:right; line-height:2; font-size:16px; font-family:Arial, Tahoma, sans-serif;">

يكمّل فحص المعرّفات في المرحلة السابقة بفحص أعمق على مستوى المحتوى.

الهدف هو منع تسرّب الأسئلة المُعاد صياغتها أو المتشابهة جدًا بين مجموعة التقييم ومجموعة الفهرسة.

يُقاس التشابه باستخدام تمثيل TF-IDF على مستوى الأحرف، وهو متين أمام اختلاف التشكيل والأخطاء الإملائية في العربية.

كل سؤال تقييمي يتجاوز عتبة التشابه يُعتبر تسريبًا ويُحذف من مجموعة التقييم.

</div>

In [12]:
# =========================================================
# Stage 06.5 - Content-Level Leakage Detection
# Near-Duplicate FAQ Detection between Indexing and Evaluation Sets
# =========================================================

from IPython.display import display, Markdown
import pandas as pd
import numpy as np
import re
from pathlib import Path

try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
except Exception as e:
    raise ImportError(
        "Stage 06.5 requires scikit-learn. Install it using: pip install scikit-learn"
    ) from e

# ---------------------------------------------------------
# 1) Settings
# ---------------------------------------------------------

# Strong leakage: automatically removed from evaluation
LEAK_SIM_THRESHOLD = 0.90

# Review zone: flagged for audit, but not automatically removed
REVIEW_SIM_THRESHOLD = 0.85

if "FINAL_DIR" not in globals():
    FINAL_DIR = Path("03_final")

FINAL_DIR.mkdir(parents=True, exist_ok=True)

assert "df_faq_indexing" in globals(), "df_faq_indexing غير موجود. شغّل Stage 06 أولًا."
assert "df_faq_evaluation" in globals(), "df_faq_evaluation غير موجود. شغّل Stage 06 أولًا."

assert len(df_faq_indexing) > 0, "FAQ indexing set is empty."
assert len(df_faq_evaluation) > 0, "FAQ evaluation set is empty."

# ---------------------------------------------------------
# 2) Helper functions
# ---------------------------------------------------------

def fallback_normalize_arabic_for_search(text):
    text = "" if text is None else str(text)

    # Normalize common Arabic variants
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ؤ", "و", text)
    text = re.sub(r"ئ", "ي", text)
    text = re.sub(r"ة", "ه", text)

    # Remove tashkeel
    text = re.sub(r"[\u064B-\u065F\u0670]", "", text)

    # Keep Arabic, digits, English, and spaces
    text = re.sub(r"[^\u0600-\u06FFa-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


if "normalize_arabic_for_search" not in globals():
    normalize_arabic_for_search = fallback_normalize_arabic_for_search


def clean_text(value):
    if value is None:
        return ""

    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass

    value = str(value)
    value = re.sub(r"\s+", " ", value).strip()
    return value


def build_faq_compare_text(row, mode="question"):
    question = clean_text(row.get("question", ""))
    answer = clean_text(row.get("answer", ""))

    if mode == "question_answer":
        return normalize_arabic_for_search(f"{question} {answer}")

    return normalize_arabic_for_search(question)


def short_text(text, max_len=160):
    text = clean_text(text)
    return text[:max_len] + ("..." if len(text) > max_len else "")


def hide_index_safe(styler):
    try:
        return styler.hide(axis="index")
    except Exception:
        try:
            return styler.hide_index()
        except Exception:
            return styler


def apply_style_compatible(styler, func, subset):
    if hasattr(styler, "map"):
        return styler.map(func, subset=subset)
    return styler.applymap(func, subset=subset)


def color_leak_status(value):
    value = str(value)

    if value == "remove_strong_leak":
        return "background-color:#FCE4D6; color:#9C0006; font-weight:bold;"

    if value == "review_possible_leak":
        return "background-color:#FFF2CC; color:#7F6000; font-weight:bold;"

    if value == "clean":
        return "background-color:#E2F0D9; color:#375623; font-weight:bold;"

    return ""


def default_professional_table(df, caption="", width="100%", font_size="12px"):
    styler = (
        df.style
        .set_caption(caption)
        .set_table_styles([
            {
                "selector": "caption",
                "props": [
                    ("caption-side", "top"),
                    ("font-size", "17px"),
                    ("font-weight", "bold"),
                    ("color", "#1F4E78"),
                    ("text-align", "center"),
                    ("padding", "10px")
                ]
            },
            {
                "selector": "th",
                "props": [
                    ("background-color", "#1F4E78"),
                    ("color", "white"),
                    ("font-weight", "bold"),
                    ("text-align", "center"),
                    ("border", "1px solid #D9E2F3"),
                    ("padding", "8px"),
                    ("white-space", "normal")
                ]
            },
            {
                "selector": "td",
                "props": [
                    ("text-align", "center"),
                    ("border", "1px solid #D9E2F3"),
                    ("padding", "7px"),
                    ("vertical-align", "middle"),
                    ("white-space", "normal")
                ]
            },
            {
                "selector": "tbody tr:nth-child(even)",
                "props": [("background-color", "#F8FBFD")]
            },
            {
                "selector": "tbody tr:nth-child(odd)",
                "props": [("background-color", "#FFFFFF")]
            },
            {
                "selector": "table",
                "props": [
                    ("border-collapse", "collapse"),
                    ("width", width),
                    ("font-family", "Arial, Tahoma, sans-serif"),
                    ("font-size", font_size),
                    ("table-layout", "fixed"),
                    ("margin", "12px 0")
                ]
            }
        ])
    )

    return hide_index_safe(styler)


if "professional_table" not in globals():
    professional_table = default_professional_table

# ---------------------------------------------------------
# 3) Stage explanation
# ---------------------------------------------------------

display(Markdown("""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:14px;
    margin:12px 0;
    border-radius:6px;
">

<h3 style="color:#1F4E78; margin-top:0;">
Stage 06.5 — Content-Level Leakage Detection
</h3>

<p>
This stage checks whether held-out FAQ evaluation questions are semantically too similar to FAQ records used for indexing.
It complements the exact <code>document_unit_id</code> leakage check from Stage 06 by detecting near-duplicate or paraphrased records.
</p>

<p>
The method uses character-level TF-IDF n-grams with cosine similarity. Questions with similarity above the strong leakage threshold are removed from the evaluation set, while borderline cases are kept but flagged for audit.
</p>

</div>
"""))

# ---------------------------------------------------------
# 4) Prepare comparison texts
# ---------------------------------------------------------

idx_q_texts = df_faq_indexing.apply(
    lambda row: build_faq_compare_text(row, mode="question"),
    axis=1
).tolist()

ev_q_texts = df_faq_evaluation.apply(
    lambda row: build_faq_compare_text(row, mode="question"),
    axis=1
).tolist()

idx_qa_texts = df_faq_indexing.apply(
    lambda row: build_faq_compare_text(row, mode="question_answer"),
    axis=1
).tolist()

ev_qa_texts = df_faq_evaluation.apply(
    lambda row: build_faq_compare_text(row, mode="question_answer"),
    axis=1
).tolist()

# ---------------------------------------------------------
# 5) Similarity function
# ---------------------------------------------------------

def compute_nearest_similarity(eval_texts, index_texts):
    vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=1
    )

    vectorizer.fit(index_texts + eval_texts)

    matrix_index = vectorizer.transform(index_texts)
    matrix_eval = vectorizer.transform(eval_texts)

    sim_matrix = cosine_similarity(matrix_eval, matrix_index)

    max_sim = sim_matrix.max(axis=1)
    nearest_idx = sim_matrix.argmax(axis=1)

    return max_sim, nearest_idx

q_max_sim, q_nearest = compute_nearest_similarity(ev_q_texts, idx_q_texts)
qa_max_sim, qa_nearest = compute_nearest_similarity(ev_qa_texts, idx_qa_texts)

# ---------------------------------------------------------
# 6) Build leakage audit table
# ---------------------------------------------------------

audit = df_faq_evaluation.copy().reset_index(drop=True)

audit["question_similarity_to_indexing"] = np.round(q_max_sim, 4)
audit["qa_similarity_to_indexing"] = np.round(qa_max_sim, 4)

audit["max_similarity_to_indexing"] = audit[
    ["question_similarity_to_indexing", "qa_similarity_to_indexing"]
].max(axis=1)

audit["nearest_indexing_question_by_question"] = [
    df_faq_indexing["question"].iloc[i] for i in q_nearest
]

audit["nearest_indexing_question_by_qa"] = [
    df_faq_indexing["question"].iloc[i] for i in qa_nearest
]

audit["nearest_indexing_document_unit_id_by_question"] = [
    df_faq_indexing["document_unit_id"].iloc[i] for i in q_nearest
]

audit["nearest_indexing_document_unit_id_by_qa"] = [
    df_faq_indexing["document_unit_id"].iloc[i] for i in qa_nearest
]

def leakage_status(score):
    if score >= LEAK_SIM_THRESHOLD:
        return "remove_strong_leak"

    if score >= REVIEW_SIM_THRESHOLD:
        return "review_possible_leak"

    return "clean"

audit["content_leakage_status"] = audit["max_similarity_to_indexing"].apply(leakage_status)

strong_leaks = audit[audit["content_leakage_status"].eq("remove_strong_leak")].copy()
review_leaks = audit[audit["content_leakage_status"].eq("review_possible_leak")].copy()

# ---------------------------------------------------------
# 7) Remove strong leaks from evaluation set
# ---------------------------------------------------------

before_eval_count = len(df_faq_evaluation)

strong_leak_ids = set(str(x) for x in strong_leaks["document_unit_id"].tolist())

df_faq_evaluation_clean = (
    df_faq_evaluation[
        ~df_faq_evaluation["document_unit_id"].astype(str).isin(strong_leak_ids)
    ]
    .copy()
    .reset_index(drop=True)
)

after_eval_count = len(df_faq_evaluation_clean)

assert after_eval_count > 0, (
    "FAQ evaluation set became empty after removing strong content leaks. "
    "Increase LEAK_SIM_THRESHOLD or review the leakage audit manually."
)

# Keep global variable updated for later stages
df_faq_evaluation = df_faq_evaluation_clean.copy()

# ---------------------------------------------------------
# 8) Re-check exact ID overlap
# ---------------------------------------------------------

index_ids = set(df_faq_indexing["document_unit_id"].astype(str))
eval_ids = set(df_faq_evaluation["document_unit_id"].astype(str))

id_overlap_after_cleaning = len(index_ids.intersection(eval_ids))

assert id_overlap_after_cleaning == 0, "Exact ID leakage detected after Stage 06.5 cleaning."

# ---------------------------------------------------------
# 9) Save outputs
# ---------------------------------------------------------

audit_path_csv = FINAL_DIR / "faq_eval_content_leakage_audit.csv"
audit_path_xlsx = FINAL_DIR / "faq_eval_content_leakage_audit.xlsx"

strong_leaks_path_csv = FINAL_DIR / "faq_eval_content_leaks_removed.csv"
strong_leaks_path_xlsx = FINAL_DIR / "faq_eval_content_leaks_removed.xlsx"

review_leaks_path_csv = FINAL_DIR / "faq_eval_possible_leaks_review.csv"
review_leaks_path_xlsx = FINAL_DIR / "faq_eval_possible_leaks_review.xlsx"

clean_eval_path_csv = FINAL_DIR / "hrsd_faq_evaluation_set.csv"
clean_eval_path_xlsx = FINAL_DIR / "hrsd_faq_evaluation_set.xlsx"

audit.to_csv(audit_path_csv, index=False, encoding="utf-8-sig")
audit.to_excel(audit_path_xlsx, index=False)

strong_leaks.to_csv(strong_leaks_path_csv, index=False, encoding="utf-8-sig")
strong_leaks.to_excel(strong_leaks_path_xlsx, index=False)

review_leaks.to_csv(review_leaks_path_csv, index=False, encoding="utf-8-sig")
review_leaks.to_excel(review_leaks_path_xlsx, index=False)

df_faq_evaluation.to_csv(clean_eval_path_csv, index=False, encoding="utf-8-sig")
df_faq_evaluation.to_excel(clean_eval_path_xlsx, index=False)

# ---------------------------------------------------------
# 10) Professional summary table
# ---------------------------------------------------------

summary_df = pd.DataFrame([
    {
        "Metric": "FAQ indexing records",
        "Value": len(df_faq_indexing),
        "Interpretation": "Records used inside the experimental RAG knowledge base."
    },
    {
        "Metric": "FAQ evaluation records before content-leakage cleaning",
        "Value": before_eval_count,
        "Interpretation": "Held-out FAQ records before near-duplicate removal."
    },
    {
        "Metric": "Strong content leaks removed",
        "Value": len(strong_leaks),
        "Interpretation": f"Evaluation records with similarity >= {LEAK_SIM_THRESHOLD}."
    },
    {
        "Metric": "Possible leaks flagged for review",
        "Value": len(review_leaks),
        "Interpretation": f"Evaluation records with similarity between {REVIEW_SIM_THRESHOLD} and {LEAK_SIM_THRESHOLD}."
    },
    {
        "Metric": "FAQ evaluation records after cleaning",
        "Value": after_eval_count,
        "Interpretation": "Final cleaned FAQ evaluation records used in later stages."
    },
    {
        "Metric": "Exact ID overlap after cleaning",
        "Value": id_overlap_after_cleaning,
        "Interpretation": "Must remain zero after content-level leakage cleaning."
    }
])

summary_style = professional_table(
    summary_df,
    caption="Table 6.5 — Content-Level FAQ Leakage Detection Summary",
    width="100%",
    font_size="12px"
)

display(summary_style)

# ---------------------------------------------------------
# 11) Top similarity audit preview
# ---------------------------------------------------------

preview_cols = [
    "question",
    "max_similarity_to_indexing",
    "question_similarity_to_indexing",
    "qa_similarity_to_indexing",
    "nearest_indexing_question_by_question",
    "nearest_indexing_question_by_qa",
    "content_leakage_status"
]

preview_cols = [c for c in preview_cols if c in audit.columns]

audit_preview = (
    audit[preview_cols]
    .sort_values("max_similarity_to_indexing", ascending=False)
    .head(20)
    .copy()
)

for col in [
    "question",
    "nearest_indexing_question_by_question",
    "nearest_indexing_question_by_qa"
]:
    if col in audit_preview.columns:
        audit_preview[col] = audit_preview[col].apply(lambda x: short_text(x, 140))

audit_style = professional_table(
    audit_preview,
    caption="Table 6.6 — Top FAQ Evaluation/Indexing Similarity Pairs",
    width="100%",
    font_size="11.5px"
)

audit_style = apply_style_compatible(
    audit_style,
    color_leak_status,
    subset=["content_leakage_status"]
)

rtl_cols = [
    c for c in [
        "question",
        "nearest_indexing_question_by_question",
        "nearest_indexing_question_by_qa"
    ]
    if c in audit_preview.columns
]

if rtl_cols:
    audit_style = audit_style.set_properties(
        subset=rtl_cols,
        **{
            "text-align": "right",
            "direction": "rtl",
            "line-height": "1.7"
        }
    )

display(audit_style)

# ---------------------------------------------------------
# 12) Final interpretation
# ---------------------------------------------------------

display(Markdown(f"""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#E2F0D9;
    border-right:5px solid #375623;
    padding:14px;
    margin:12px 0;
    border-radius:6px;
">

<b>Stage 06.5 completed successfully.</b><br>

تم فحص التشابه التقريبي بين FAQ التقييم وFAQ الفهرسة باستخدام 
<code>TF-IDF char n-gram</code> و <code>cosine similarity</code>.<br>

عدد أسئلة FAQ التقييم قبل التنظيف: <b>{before_eval_count}</b><br>
عدد الأسئلة ذات التسريب القوي المحذوفة: <b>{len(strong_leaks)}</b><br>
عدد الأسئلة المشكوك فيها والمحفوظة للمراجعة: <b>{len(review_leaks)}</b><br>
عدد أسئلة FAQ التقييم بعد التنظيف: <b>{after_eval_count}</b><br>
التداخل بالمعرّفات بعد التنظيف: <b>{id_overlap_after_cleaning}</b><br>

تم حفظ ملفات التدقيق والتنظيف داخل مجلد <code>03_final</code>.

</div>
"""))

print("Stage 06.5 completed successfully.")
print("FAQ evaluation before cleaning:", before_eval_count)
print("Strong content leaks removed:", len(strong_leaks))
print("Possible leaks flagged for review:", len(review_leaks))
print("FAQ evaluation after cleaning:", after_eval_count)
print("Exact ID overlap after cleaning:", id_overlap_after_cleaning)
print("Saved audit:", audit_path_csv)
print("Saved cleaned evaluation set:", clean_eval_path_csv)


<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:14px;
    margin:12px 0;
    border-radius:6px;
">

<h3 style="color:#1F4E78; margin-top:0;">
Stage 06.5 — Content-Level Leakage Detection
</h3>

<p>
This stage checks whether held-out FAQ evaluation questions are semantically too similar to FAQ records used for indexing.
It complements the exact <code>document_unit_id</code> leakage check from Stage 06 by detecting near-duplicate or paraphrased records.
</p>

<p>
The method uses character-level TF-IDF n-grams with cosine similarity. Questions with similarity above the strong leakage threshold are removed from the evaluation set, while borderline cases are kept but flagged for audit.
</p>

</div>


Metric,Value,Interpretation
FAQ indexing records,278,Records used inside the experimental RAG knowledge base.
FAQ evaluation records before content-leakage cleaning,124,Held-out FAQ records before near-duplicate removal.
Strong content leaks removed,2,Evaluation records with similarity >= 0.9.
Possible leaks flagged for review,0,Evaluation records with similarity between 0.85 and 0.9.
FAQ evaluation records after cleaning,122,Final cleaned FAQ evaluation records used in later stages.
Exact ID overlap after cleaning,0,Must remain zero after content-level leakage cleaning.


question,max_similarity_to_indexing,question_similarity_to_indexing,qa_similarity_to_indexing,nearest_indexing_question_by_question,nearest_indexing_question_by_qa,content_leakage_status
ما هو نظام حماية الأجور؟,1.000000,1.000000,0.959600,ما هو نظام حماية الأجور؟,ما هو نظام حماية الأجور؟,remove_strong_leak
متى يتم تجديد عقد العمل؟,0.951400,0.635500,0.951400,متى يتاح تجديد رخصة العمل؟,مدة عمل عقد العامل السعودي,remove_strong_leak
هل تمكن البوابة والانظمة التابعة لها المستخدم الاعتباري (الشركات) من تعديل بياناتها,0.829500,0.732400,0.829500,هل تمكن البوابة والانظمة التابعة لها المستخدم من تعديل بياناته.,هل تمكن البوابة والانظمة التابعة لها المستخدم من تعديل بياناته.,clean
هل تمكن البوابة والانظمة التابعة لها المستخدم من الوصول إلى البيانات الشخصية؟,0.822100,0.666800,0.822100,هل تمكن البوابة والانظمة التابعة لها المستخدم من تعديل بياناته.,هل تمكن البوابة والانظمة التابعة لها المستخدم من تعديل بياناته.,clean
متى سيبدأ تطبيق نطاقات المطور؟,0.744700,0.744700,0.521900,متى تم تطبيق قرار نطاقات المطور؟,متى تم تطبيق قرار نطاقات المطور؟,clean
كيف يتم رفع طلب التنفيذ في حال عدم التزام صاحب العمل بسداد الأجر في عقد العمل الموثق سندًا تنفيذيًا؟,0.719800,0.719800,0.259400,متى يمكن للعامل التنفيذ على صاحب العمل في حال عدم التزامه بدفع الأجر لعقد العمل الموثق سندًا تنفيذيًا؟,كيف يتم التحقق من استلام العامل للأجر الذي سيتم التنفيذ عليه في عقد العمل الموثق سندًا تنفيذيًا؟,clean
ما هو الشرط الجزائي للعامل؟,0.685800,0.659000,0.685800,هل ينطبق الشرط الجزائي على العامل أو صاحب العمل؟,هل ينطبق الشرط الجزائي على العامل أو صاحب العمل؟,clean
هل عدد ساعات العمل للعامل غير المسلم أكثر من العامل المسلم في شهر رمضان المبارك؟,0.685500,0.325000,0.685500,كم عدد ساعات العمل ؟,كم عدد ساعات العمل ؟,clean
ماهي أنواع الأجور؟,0.675700,0.675700,0.200900,ماهي الأجور؟,متى يكون إنهاء العقد؟,clean
ما هو إجراء تنفيذ طلب خدمة التنقل الوظيفي؟,0.671600,0.671600,0.317000,ماهي خدمة التنقل الوظيفي؟,كيف يتم الاستفادة من خدمة التنقل الوظيفي للعمال الوافدين بين المنشآت؟,clean



<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#E2F0D9;
    border-right:5px solid #375623;
    padding:14px;
    margin:12px 0;
    border-radius:6px;
">

<b>Stage 06.5 completed successfully.</b><br>

تم فحص التشابه التقريبي بين FAQ التقييم وFAQ الفهرسة باستخدام 
<code>TF-IDF char n-gram</code> و <code>cosine similarity</code>.<br>

عدد أسئلة FAQ التقييم قبل التنظيف: <b>124</b><br>
عدد الأسئلة ذات التسريب القوي المحذوفة: <b>2</b><br>
عدد الأسئلة المشكوك فيها والمحفوظة للمراجعة: <b>0</b><br>
عدد أسئلة FAQ التقييم بعد التنظيف: <b>122</b><br>
التداخل بالمعرّفات بعد التنظيف: <b>0</b><br>

تم حفظ ملفات التدقيق والتنظيف داخل مجلد <code>03_final</code>.

</div>


Stage 06.5 completed successfully.
FAQ evaluation before cleaning: 124
Strong content leaks removed: 2
Possible leaks flagged for review: 0
FAQ evaluation after cleaning: 122
Exact ID overlap after cleaning: 0
Saved audit: saudi_labor_law_voice_agent_project\03_final\faq_eval_content_leakage_audit.csv
Saved cleaned evaluation set: saudi_labor_law_voice_agent_project\03_final\hrsd_faq_evaluation_set.csv


In [13]:
from IPython.display import display, Markdown

display(Markdown("""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:16px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:16px;
    margin:14px 0;
    border-radius:6px;
">

<h3 style="color:#1F4E78; margin-top:0;">
Interpretation — Content-Level FAQ Leakage Detection
</h3>

<p>
توضح هذه المرحلة فحص التسريب التقريبي بين أسئلة FAQ المخصصة للتقييم وأسئلة FAQ المستخدمة في الفهرسة. 
لا يكتفي هذا الفحص بمقارنة المعرّفات، بل يقيس التشابه النصي بين الأسئلة باستخدام 
<code>TF-IDF</code> على مستوى <code>character n-grams</code> مع حساب <code>cosine similarity</code>.
</p>

<p>
بلغ عدد سجلات FAQ المستخدمة في الفهرسة <b>278</b> سجلًا، بينما بلغ عدد سجلات FAQ المخصصة للتقييم قبل التنظيف <b>124</b> سجلًا.
كشف الفحص عن <b>2</b> سجلين في مجموعة التقييم لديهما تشابه مرتفع جدًا مع سجلات موجودة في الفهرسة، ولذلك تم حذفهما من مجموعة التقييم.
</p>

<p>
بعد إزالة حالات التسريب القوي، أصبح عدد سجلات FAQ النظيفة في مجموعة التقييم <b>122</b> سجلًا. 
كما بقي التداخل المباشر بالمعرّفات بين الفهرسة والتقييم مساويًا <b>0</b>، مما يؤكد سلامة الفصل بين المجموعتين.
</p>

<div style="
    background-color:#E2F0D9;
    border-right:4px solid #375623;
    padding:12px;
    margin-top:12px;
    border-radius:5px;
">
<b>Conclusion:</b><br>
The content-level leakage check successfully removed near-duplicate FAQ evaluation questions while preserving a clean held-out evaluation set. 
This improves the reliability of the retrieval evaluation and reduces the risk of inflated FAQ retrieval scores.
</div>

</div>
"""))


<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:16px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:16px;
    margin:14px 0;
    border-radius:6px;
">

<h3 style="color:#1F4E78; margin-top:0;">
Interpretation — Content-Level FAQ Leakage Detection
</h3>

<p>
توضح هذه المرحلة فحص التسريب التقريبي بين أسئلة FAQ المخصصة للتقييم وأسئلة FAQ المستخدمة في الفهرسة. 
لا يكتفي هذا الفحص بمقارنة المعرّفات، بل يقيس التشابه النصي بين الأسئلة باستخدام 
<code>TF-IDF</code> على مستوى <code>character n-grams</code> مع حساب <code>cosine similarity</code>.
</p>

<p>
بلغ عدد سجلات FAQ المستخدمة في الفهرسة <b>278</b> سجلًا، بينما بلغ عدد سجلات FAQ المخصصة للتقييم قبل التنظيف <b>124</b> سجلًا.
كشف الفحص عن <b>2</b> سجلين في مجموعة التقييم لديهما تشابه مرتفع جدًا مع سجلات موجودة في الفهرسة، ولذلك تم حذفهما من مجموعة التقييم.
</p>

<p>
بعد إزالة حالات التسريب القوي، أصبح عدد سجلات FAQ النظيفة في مجموعة التقييم <b>122</b> سجلًا. 
كما بقي التداخل المباشر بالمعرّفات بين الفهرسة والتقييم مساويًا <b>0</b>، مما يؤكد سلامة الفصل بين المجموعتين.
</p>

<div style="
    background-color:#E2F0D9;
    border-right:4px solid #375623;
    padding:12px;
    margin-top:12px;
    border-radius:5px;
">
<b>Conclusion:</b><br>
The content-level leakage check successfully removed near-duplicate FAQ evaluation questions while preserving a clean held-out evaluation set. 
This improves the reliability of the retrieval evaluation and reduces the risk of inflated FAQ retrieval scores.
</div>

</div>


### Interpretation — Content-Level Leakage Detection

<div dir="rtl" style="text-align:right; line-height:2; font-size:16px; font-family:Arial, Tahoma, sans-serif;
     background-color:#F8FBFD; border-right:5px solid #1F4E78; padding:16px; margin:12px 0; border-radius:6px;">

<h3 style="color:#1F4E78; margin-top:0;">لماذا هذه الخطوة مهمة بحثيًا</h3>

<p>
فحص المعرّفات في المرحلة السابقة يمنع التطابق التام فقط.
</p>

<p>
لكن السؤال المُعاد صياغته يبقى تسريبًا خفيًا يرفع نتائج الاسترجاع بشكل غير واقعي.
</p>

<p>
هذه الخطوة تقيس التشابه على مستوى الأحرف، فتكتشف إعادة الصياغة والأخطاء الإملائية واختلاف التشكيل.
</p>

<p>
بحذف الأسئلة المتسرّبة، تصبح أرقام تقييم استرجاع الأسئلة الشائعة في المرحلة الثالثة أكثر مصداقية وقابلية للدفاع في المناقشة.
</p>

<p>
يُحفظ ملف تدقيق مستقل يوثّق كل زوج ودرجة تشابهه، لإثبات أن ضبط التسريب تم على مستوى المحتوى لا المعرّفات فقط.
</p>

</div>

## Stage 07 — Data Quality Report

This stage generates a data quality report before constructing the RAG knowledge base.

The report checks whether the cleaned datasets are complete, whether critical fields are populated, whether duplicated identifiers exist, and whether leakage risks are present. These checks provide an academic quality-assurance layer before the data is used for retrieval experiments.


In [14]:
# =========================================================
# Stage 07 - Data Quality Report
# Professional Data Quality and RAG Safety Report
# =========================================================

from IPython.display import display, Markdown
import pandas as pd
import numpy as np
import re
from pathlib import Path

# ---------------------------------------------------------
# 1) Directory safety
# ---------------------------------------------------------

if "REPORTS_DIR" not in globals():
    REPORTS_DIR = Path("05_reports")

if "FINAL_DIR" not in globals():
    FINAL_DIR = Path("03_final")

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# 2) Helper functions
# ---------------------------------------------------------

quality_rows = []

def add_quality_check(category, check, value, expected, status, interpretation=""):
    quality_rows.append({
        "Category": category,
        "Check": check,
        "Value": value,
        "Expected": expected,
        "Status": status,
        "Interpretation": interpretation,
    })


def safe_len(obj_name):
    if obj_name in globals() and isinstance(globals()[obj_name], pd.DataFrame):
        return len(globals()[obj_name])
    return "Not available"


def has_columns(df, cols):
    return isinstance(df, pd.DataFrame) and set(cols).issubset(df.columns)


def clean_text(value):
    if value is None:
        return ""

    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass

    return re.sub(r"\s+", " ", str(value)).strip()


def status_from_bool(condition):
    return "PASS" if condition else "FAIL"


def status_warn_if_missing(value):
    return "WARN" if value == "Not available" else "PASS"


def hide_index_safe(styler):
    try:
        return styler.hide(axis="index")
    except Exception:
        try:
            return styler.hide_index()
        except Exception:
            return styler


def apply_style_compatible(styler, func, subset):
    if hasattr(styler, "map"):
        return styler.map(func, subset=subset)
    return styler.applymap(func, subset=subset)


def color_status(value):
    value = str(value)

    if value == "PASS":
        return "background-color:#E2F0D9; color:#375623; font-weight:bold;"

    if value == "WARN":
        return "background-color:#FFF2CC; color:#7F6000; font-weight:bold;"

    if value == "FAIL":
        return "background-color:#FCE4D6; color:#9C0006; font-weight:bold;"

    return ""


def professional_table(df, caption="", width="100%", font_size="12px"):
    styler = (
        df.style
        .set_caption(caption)
        .set_table_styles([
            {
                "selector": "caption",
                "props": [
                    ("caption-side", "top"),
                    ("font-size", "17px"),
                    ("font-weight", "bold"),
                    ("color", "#1F4E78"),
                    ("text-align", "center"),
                    ("padding", "10px")
                ]
            },
            {
                "selector": "th",
                "props": [
                    ("background-color", "#1F4E78"),
                    ("color", "white"),
                    ("font-weight", "bold"),
                    ("text-align", "center"),
                    ("border", "1px solid #D9E2F3"),
                    ("padding", "8px"),
                    ("white-space", "normal")
                ]
            },
            {
                "selector": "td",
                "props": [
                    ("text-align", "center"),
                    ("border", "1px solid #D9E2F3"),
                    ("padding", "7px"),
                    ("vertical-align", "middle"),
                    ("white-space", "normal")
                ]
            },
            {
                "selector": "tbody tr:nth-child(even)",
                "props": [("background-color", "#F8FBFD")]
            },
            {
                "selector": "tbody tr:nth-child(odd)",
                "props": [("background-color", "#FFFFFF")]
            },
            {
                "selector": "table",
                "props": [
                    ("border-collapse", "collapse"),
                    ("width", width),
                    ("font-family", "Arial, Tahoma, sans-serif"),
                    ("font-size", font_size),
                    ("table-layout", "fixed"),
                    ("margin", "12px 0")
                ]
            }
        ])
    )

    return hide_index_safe(styler)


# ---------------------------------------------------------
# 3) Stage explanation
# ---------------------------------------------------------

display(Markdown("""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:14px;
    margin:12px 0;
    border-radius:6px;
">

<h3 style="color:#1F4E78; margin-top:0;">
Stage 07 — Data Quality and RAG Safety Report
</h3>

<p>
This stage summarizes the quality of the cleaned legal articles, the filtered FAQ records, and the safe FAQ indexing/evaluation split.
It also verifies exact leakage, content-level leakage outputs from Stage 06.5, duplicate records, missing text fields, and repealed article handling.
</p>

<p>
The goal is to confirm that the dataset is ready for building the experimental RAG knowledge base and for running retrieval evaluation in later stages.
</p>

</div>
"""))

# ---------------------------------------------------------
# 4) Resolve active datasets
# ---------------------------------------------------------

df_articles_available = "df_articles" in globals() and isinstance(df_articles, pd.DataFrame)
df_faq_available = "df_faq" in globals() and isinstance(df_faq, pd.DataFrame)

df_faq_indexing_available = (
    "df_faq_indexing" in globals()
    and isinstance(df_faq_indexing, pd.DataFrame)
)

df_faq_evaluation_available = (
    "df_faq_evaluation" in globals()
    and isinstance(df_faq_evaluation, pd.DataFrame)
)

# ---------------------------------------------------------
# 5) Basic dataset size checks
# ---------------------------------------------------------

add_quality_check(
    "Dataset Size",
    "Legal article records",
    len(df_articles) if df_articles_available else "Not available",
    "> 0",
    "PASS" if df_articles_available and len(df_articles) > 0 else "FAIL",
    "Legal article records after cleaning and status classification."
)

if df_articles_available and "article_number_int" in df_articles.columns:
    unique_numeric_articles = df_articles["article_number_int"].dropna().astype(float).astype(int).nunique()
else:
    unique_numeric_articles = "Not available"

add_quality_check(
    "Dataset Size",
    "Unique numeric legal articles",
    unique_numeric_articles,
    "> 0",
    "PASS" if isinstance(unique_numeric_articles, int) and unique_numeric_articles > 0 else "WARN",
    "Coverage of numeric legal article sequence. Repeated legal labels such as مكرر are handled separately."
)

add_quality_check(
    "Dataset Size",
    "Raw/clean FAQ records",
    len(df_faq) if df_faq_available else "Not available",
    "> 0",
    "PASS" if df_faq_available and len(df_faq) > 0 else "WARN",
    "Original cleaned FAQ records before final domain-filtered split."
)

add_quality_check(
    "Dataset Size",
    "FAQ indexing records",
    len(df_faq_indexing) if df_faq_indexing_available else "Not available",
    "> 0",
    "PASS" if df_faq_indexing_available and len(df_faq_indexing) > 0 else "FAIL",
    "FAQ records used inside the experimental RAG knowledge base."
)

add_quality_check(
    "Dataset Size",
    "FAQ evaluation records after Stage 06.5",
    len(df_faq_evaluation) if df_faq_evaluation_available else "Not available",
    "> 0",
    "PASS" if df_faq_evaluation_available and len(df_faq_evaluation) > 0 else "FAIL",
    "Held-out FAQ records after exact and content-level leakage checks."
)

# ---------------------------------------------------------
# 6) Empty field checks
# ---------------------------------------------------------

if df_articles_available and "article_text" in df_articles.columns:
    empty_article_text = int(df_articles["article_text"].fillna("").astype(str).str.strip().eq("").sum())
else:
    empty_article_text = "Not available"

add_quality_check(
    "Completeness",
    "Empty article_text",
    empty_article_text,
    "0",
    "PASS" if empty_article_text == 0 else ("WARN" if empty_article_text == "Not available" else "FAIL"),
    "No legal article should have an empty text field."
)

if df_faq_available and "answer" in df_faq.columns:
    empty_faq_answer = int(df_faq["answer"].fillna("").astype(str).str.strip().eq("").sum())
else:
    empty_faq_answer = "Not available"

add_quality_check(
    "Completeness",
    "Empty FAQ answers",
    empty_faq_answer,
    "0",
    "PASS" if empty_faq_answer == 0 else ("WARN" if empty_faq_answer == "Not available" else "FAIL"),
    "No FAQ record should have an empty answer."
)

if df_faq_indexing_available and "question" in df_faq_indexing.columns:
    empty_index_questions = int(df_faq_indexing["question"].fillna("").astype(str).str.strip().eq("").sum())
else:
    empty_index_questions = "Not available"

add_quality_check(
    "Completeness",
    "Empty FAQ indexing questions",
    empty_index_questions,
    "0",
    "PASS" if empty_index_questions == 0 else ("WARN" if empty_index_questions == "Not available" else "FAIL"),
    "FAQ indexing questions must not be empty."
)

if df_faq_evaluation_available and "question" in df_faq_evaluation.columns:
    empty_eval_questions = int(df_faq_evaluation["question"].fillna("").astype(str).str.strip().eq("").sum())
else:
    empty_eval_questions = "Not available"

add_quality_check(
    "Completeness",
    "Empty FAQ evaluation questions",
    empty_eval_questions,
    "0",
    "PASS" if empty_eval_questions == 0 else ("WARN" if empty_eval_questions == "Not available" else "FAIL"),
    "FAQ evaluation questions must not be empty."
)

# ---------------------------------------------------------
# 7) Duplicate checks
# ---------------------------------------------------------

if df_articles_available and "article_number_label" in df_articles.columns:
    dup_article_labels = int(df_articles["article_number_label"].astype(str).duplicated().sum())
else:
    dup_article_labels = "Not available"

add_quality_check(
    "Duplicates",
    "Duplicated legal article labels",
    dup_article_labels,
    "0",
    "PASS" if dup_article_labels == 0 else ("WARN" if dup_article_labels == "Not available" else "FAIL"),
    "True duplicate legal article labels should not appear after cleaning."
)

if df_articles_available and "document_unit_id" in df_articles.columns:
    dup_articles_doc_id = int(df_articles["document_unit_id"].astype(str).duplicated().sum())
else:
    dup_articles_doc_id = "Not available"

add_quality_check(
    "Duplicates",
    "Duplicated legal document IDs",
    dup_articles_doc_id,
    "0",
    "PASS" if dup_articles_doc_id == 0 else ("WARN" if dup_articles_doc_id == "Not available" else "FAIL"),
    "Duplicate legal document IDs indicate repeated legal records."
)

if df_faq_indexing_available and "document_unit_id" in df_faq_indexing.columns:
    dup_faq_index = int(df_faq_indexing["document_unit_id"].astype(str).duplicated().sum())
else:
    dup_faq_index = "Not available"

add_quality_check(
    "Duplicates",
    "Duplicated FAQ indexing document IDs",
    dup_faq_index,
    "0",
    "PASS" if dup_faq_index == 0 else ("WARN" if dup_faq_index == "Not available" else "FAIL"),
    "FAQ indexing set should not contain repeated question-answer records."
)

if df_faq_evaluation_available and "document_unit_id" in df_faq_evaluation.columns:
    dup_faq_eval = int(df_faq_evaluation["document_unit_id"].astype(str).duplicated().sum())
else:
    dup_faq_eval = "Not available"

add_quality_check(
    "Duplicates",
    "Duplicated FAQ evaluation document IDs",
    dup_faq_eval,
    "0",
    "PASS" if dup_faq_eval == 0 else ("WARN" if dup_faq_eval == "Not available" else "FAIL"),
    "FAQ evaluation set should not contain repeated question-answer records."
)

# ---------------------------------------------------------
# 8) Exact FAQ leakage check
# ---------------------------------------------------------

if (
    df_faq_indexing_available
    and df_faq_evaluation_available
    and "document_unit_id" in df_faq_indexing.columns
    and "document_unit_id" in df_faq_evaluation.columns
):
    faq_exact_leakage = len(
        set(df_faq_indexing["document_unit_id"].astype(str))
        &
        set(df_faq_evaluation["document_unit_id"].astype(str))
    )
else:
    faq_exact_leakage = "Not available"

add_quality_check(
    "Leakage",
    "Exact FAQ leakage between indexing and evaluation",
    faq_exact_leakage,
    "0",
    "PASS" if faq_exact_leakage == 0 else ("WARN" if faq_exact_leakage == "Not available" else "FAIL"),
    "Must be zero to avoid inflated retrieval scores caused by exact split leakage."
)

# ---------------------------------------------------------
# 9) Content-level leakage audit from Stage 06.5
# ---------------------------------------------------------

content_audit_path = FINAL_DIR / "faq_eval_content_leakage_audit.csv"
strong_leaks_path = FINAL_DIR / "faq_eval_content_leaks_removed.csv"
possible_leaks_path = FINAL_DIR / "faq_eval_possible_leaks_review.csv"

if content_audit_path.exists():
    df_content_audit = pd.read_csv(content_audit_path, encoding="utf-8-sig")
    content_audit_records = len(df_content_audit)
else:
    df_content_audit = pd.DataFrame()
    content_audit_records = "Not available"

if strong_leaks_path.exists():
    df_strong_leaks = pd.read_csv(strong_leaks_path, encoding="utf-8-sig")
    strong_content_leaks_removed = len(df_strong_leaks)
else:
    df_strong_leaks = pd.DataFrame()
    strong_content_leaks_removed = "Not available"

if possible_leaks_path.exists():
    df_possible_leaks = pd.read_csv(possible_leaks_path, encoding="utf-8-sig")
    possible_content_leaks_review = len(df_possible_leaks)
else:
    df_possible_leaks = pd.DataFrame()
    possible_content_leaks_review = "Not available"

add_quality_check(
    "Leakage",
    "Content-level leakage audit records",
    content_audit_records,
    "> 0",
    "PASS" if isinstance(content_audit_records, int) and content_audit_records > 0 else "WARN",
    "Stage 06.5 audit file generated using TF-IDF character n-grams and cosine similarity."
)

add_quality_check(
    "Leakage",
    "Strong content leaks removed from FAQ evaluation",
    strong_content_leaks_removed,
    ">= 0",
    "PASS" if isinstance(strong_content_leaks_removed, int) else "WARN",
    "Strong near-duplicate FAQ evaluation records removed after content-level leakage detection."
)

add_quality_check(
    "Leakage",
    "Possible content leaks flagged for review",
    possible_content_leaks_review,
    ">= 0",
    "PASS" if isinstance(possible_content_leaks_review, int) else "WARN",
    "Borderline FAQ similarities retained but saved for manual review."
)

# ---------------------------------------------------------
# 10) Repealed articles and RAG safety checks
# ---------------------------------------------------------

if df_articles_available and "article_status" in df_articles.columns:
    repealed_count = int(df_articles["article_status"].astype(str).eq("repealed").sum())
    active_count = int(df_articles["article_status"].astype(str).eq("active").sum())
else:
    repealed_count = "Not available"
    active_count = "Not available"

add_quality_check(
    "Legal Status",
    "Active legal articles retained",
    active_count,
    "> 0",
    "PASS" if isinstance(active_count, int) and active_count > 0 else "WARN",
    "Active legal articles are retained and allowed for RAG indexing."
)

add_quality_check(
    "Legal Status",
    "Repealed legal articles retained for audit",
    repealed_count,
    ">= 0",
    "PASS" if isinstance(repealed_count, int) else "WARN",
    "Repealed articles should be retained only as audit/archive records and not indexed for retrieval."
)

if "df_articles_active" in globals() and isinstance(df_articles_active, pd.DataFrame):
    active_rag_count = len(df_articles_active)
else:
    active_rag_count = active_count

add_quality_check(
    "Legal Status",
    "Active legal articles used for RAG",
    active_rag_count,
    "> 0",
    "PASS" if isinstance(active_rag_count, int) and active_rag_count > 0 else "WARN",
    "Only active legal articles should be used in the RAG legal source."
)

if "df_articles_repealed" in globals() and isinstance(df_articles_repealed, pd.DataFrame):
    repealed_archive_count = len(df_articles_repealed)
else:
    repealed_archive_count = repealed_count

add_quality_check(
    "Legal Status",
    "Repealed legal articles archived only",
    repealed_archive_count,
    "Same as repealed count",
    "PASS" if isinstance(repealed_archive_count, int) else "WARN",
    "Repealed legal articles are preserved for transparency and excluded from retrieval indexing."
)

if (
    "kb_experimental" in globals()
    and isinstance(kb_experimental, pd.DataFrame)
    and {"source_type", "article_status"}.issubset(kb_experimental.columns)
):
    repealed_indexed_count = int(
        kb_experimental[
            (kb_experimental["source_type"].astype(str) == "legal_article")
            &
            (kb_experimental["article_status"].astype(str) == "repealed")
        ].shape[0]
    )
    repealed_index_check_source = "Verified using kb_experimental."

elif (
    "df_articles_active" in globals()
    and isinstance(df_articles_active, pd.DataFrame)
    and "article_status" in df_articles_active.columns
):
    repealed_indexed_count = int(
        df_articles_active["article_status"].astype(str).eq("repealed").sum()
    )
    repealed_index_check_source = "Verified using df_articles_active."

else:
    repealed_indexed_count = "Not available"
    repealed_index_check_source = "kb_experimental is not available yet. This is normal if Stage 08 has not been executed."

add_quality_check(
    "RAG Safety",
    "Repealed legal articles indexed in RAG",
    repealed_indexed_count,
    "0",
    "PASS" if repealed_indexed_count == 0 else ("WARN" if repealed_indexed_count == "Not available" else "FAIL"),
    repealed_index_check_source + " Repealed legal articles must be zero inside RAG indexing sources."
)

# ---------------------------------------------------------
# 11) Build and display final quality report
# ---------------------------------------------------------

quality_report = pd.DataFrame(quality_rows)

quality_style = professional_table(
    quality_report,
    caption="Table 7.1 — Data Quality and RAG Safety Report",
    width="100%",
    font_size="11.5px"
)

quality_style = apply_style_compatible(
    quality_style,
    color_status,
    subset=["Status"]
)

quality_style = quality_style.set_properties(
    subset=["Interpretation"],
    **{
        "text-align": "left",
        "direction": "ltr",
        "line-height": "1.6"
    }
)

display(quality_style)

# ---------------------------------------------------------
# 12) Compact summary
# ---------------------------------------------------------

status_summary = (
    quality_report["Status"]
    .value_counts()
    .reset_index()
)

status_summary.columns = ["Status", "Checks"]

status_style = professional_table(
    status_summary,
    caption="Table 7.2 — Quality Check Status Summary",
    width="45%",
    font_size="12px"
)

status_style = apply_style_compatible(
    status_style,
    color_status,
    subset=["Status"]
)

display(status_style)

# ---------------------------------------------------------
# 13) Save reports
# ---------------------------------------------------------

quality_report.to_csv(
    REPORTS_DIR / "data_quality_report.csv",
    index=False,
    encoding="utf-8-sig"
)

quality_report.to_excel(
    REPORTS_DIR / "data_quality_report.xlsx",
    index=False
)

status_summary.to_csv(
    REPORTS_DIR / "data_quality_status_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

status_summary.to_excel(
    REPORTS_DIR / "data_quality_status_summary.xlsx",
    index=False
)

# ---------------------------------------------------------
# 14) Final interpretation
# ---------------------------------------------------------

n_pass = int((quality_report["Status"] == "PASS").sum())
n_warn = int((quality_report["Status"] == "WARN").sum())
n_fail = int((quality_report["Status"] == "FAIL").sum())

display(Markdown(f"""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:{'#E2F0D9' if n_fail == 0 else '#FCE4D6'};
    border-right:5px solid {'#375623' if n_fail == 0 else '#9C0006'};
    padding:14px;
    margin:12px 0;
    border-radius:6px;
">

<b>Stage 07 completed successfully.</b><br>

عدد فحوصات PASS: <b>{n_pass}</b><br>
عدد فحوصات WARN: <b>{n_warn}</b><br>
عدد فحوصات FAIL: <b>{n_fail}</b><br><br>

تم حفظ تقرير جودة البيانات في:<br>
<code>{REPORTS_DIR / "data_quality_report.csv"}</code><br>
<code>{REPORTS_DIR / "data_quality_report.xlsx"}</code>

</div>
"""))

print("Stage 07 completed successfully.")
print("PASS checks:", n_pass)
print("WARN checks:", n_warn)
print("FAIL checks:", n_fail)
print("Saved:", REPORTS_DIR / "data_quality_report.csv")
print("Saved:", REPORTS_DIR / "data_quality_report.xlsx")


<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:14px;
    margin:12px 0;
    border-radius:6px;
">

<h3 style="color:#1F4E78; margin-top:0;">
Stage 07 — Data Quality and RAG Safety Report
</h3>

<p>
This stage summarizes the quality of the cleaned legal articles, the filtered FAQ records, and the safe FAQ indexing/evaluation split.
It also verifies exact leakage, content-level leakage outputs from Stage 06.5, duplicate records, missing text fields, and repealed article handling.
</p>

<p>
The goal is to confirm that the dataset is ready for building the experimental RAG knowledge base and for running retrieval evaluation in later stages.
</p>

</div>


Category,Check,Value,Expected,Status,Interpretation
Dataset Size,Legal article records,249,> 0,PASS,Legal article records after cleaning and status classification.
Dataset Size,Unique numeric legal articles,245,> 0,PASS,Coverage of numeric legal article sequence. Repeated legal labels such as مكرر are handled separately.
Dataset Size,Raw/clean FAQ records,703,> 0,PASS,Original cleaned FAQ records before final domain-filtered split.
Dataset Size,FAQ indexing records,278,> 0,PASS,FAQ records used inside the experimental RAG knowledge base.
Dataset Size,FAQ evaluation records after Stage 06.5,122,> 0,PASS,Held-out FAQ records after exact and content-level leakage checks.
Completeness,Empty article_text,0,0,PASS,No legal article should have an empty text field.
Completeness,Empty FAQ answers,0,0,PASS,No FAQ record should have an empty answer.
Completeness,Empty FAQ indexing questions,0,0,PASS,FAQ indexing questions must not be empty.
Completeness,Empty FAQ evaluation questions,0,0,PASS,FAQ evaluation questions must not be empty.
Duplicates,Duplicated legal article labels,0,0,PASS,True duplicate legal article labels should not appear after cleaning.


Status,Checks
PASS,22



<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#E2F0D9;
    border-right:5px solid #375623;
    padding:14px;
    margin:12px 0;
    border-radius:6px;
">

<b>Stage 07 completed successfully.</b><br>

عدد فحوصات PASS: <b>22</b><br>
عدد فحوصات WARN: <b>0</b><br>
عدد فحوصات FAIL: <b>0</b><br><br>

تم حفظ تقرير جودة البيانات في:<br>
<code>saudi_labor_law_voice_agent_project\05_reports\data_quality_report.csv</code><br>
<code>saudi_labor_law_voice_agent_project\05_reports\data_quality_report.xlsx</code>

</div>


Stage 07 completed successfully.
PASS checks: 22
WARN checks: 0
FAIL checks: 0
Saved: saudi_labor_law_voice_agent_project\05_reports\data_quality_report.csv
Saved: saudi_labor_law_voice_agent_project\05_reports\data_quality_report.xlsx


In [15]:
from IPython.display import display, Markdown

display(Markdown("""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:16px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:16px;
    margin:14px 0;
    border-radius:6px;
">

<h3 style="color:#1F4E78; margin-top:0;">
Interpretation — Data Quality and RAG Safety Report
</h3>

<p>
توضح نتائج تقرير جودة البيانات أن قاعدة البيانات أصبحت جاهزة للانتقال إلى مراحل بناء قاعدة المعرفة وتجارب الاسترجاع. 
أظهرت جميع فحوصات الجودة والسلامة نتيجة <b>PASS</b>، حيث بلغ عدد الفحوصات الناجحة <b>22</b> فحصًا، بينما لم تظهر أي تحذيرات أو أخطاء.
</p>

<p>
تؤكد النتائج أن سجلات المواد القانونية موجودة وسليمة، وأن عدد المواد القانونية الفعالة المستخدمة في قاعدة RAG بلغ <b>211</b> مادة. 
كما تم الاحتفاظ بالمواد الملغاة وعددها <b>38</b> مادة في الأرشيف فقط لأغراض التوثيق والمراجعة، دون إدخالها في الفهرسة أو الاسترجاع.
</p>

<p>
أظهرت فحوصات السلامة أن عدد المواد الملغاة المفهرسة داخل RAG يساوي <b>0</b>، وهذا يؤكد أن قاعدة المعرفة التجريبية لا تحتوي على مواد قانونية غير فعالة أو ملغاة. 
كما تم التحقق من أن تقسيم FAQ إلى فهرسة وتقييم لا يحتوي على تسريب مباشر أو تكرار مؤثر.
</p>

<p>
تم أيضًا حفظ تقرير جودة البيانات بصيغتين، CSV و Excel، داخل مجلد التقارير، مما يوفر سجلًا قابلًا للمراجعة والتوثيق الأكاديمي.
</p>

<div style="
    background-color:#E2F0D9;
    border-right:4px solid #375623;
    padding:12px;
    margin-top:12px;
    border-radius:5px;
">
<b>Conclusion:</b><br>
The data quality validation was completed successfully. All checks passed with no warnings or failures, indicating that the cleaned legal and FAQ datasets are methodologically safe and ready for the next RAG construction and retrieval evaluation stages.
</div>

</div>
"""))


<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:16px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:16px;
    margin:14px 0;
    border-radius:6px;
">

<h3 style="color:#1F4E78; margin-top:0;">
Interpretation — Data Quality and RAG Safety Report
</h3>

<p>
توضح نتائج تقرير جودة البيانات أن قاعدة البيانات أصبحت جاهزة للانتقال إلى مراحل بناء قاعدة المعرفة وتجارب الاسترجاع. 
أظهرت جميع فحوصات الجودة والسلامة نتيجة <b>PASS</b>، حيث بلغ عدد الفحوصات الناجحة <b>22</b> فحصًا، بينما لم تظهر أي تحذيرات أو أخطاء.
</p>

<p>
تؤكد النتائج أن سجلات المواد القانونية موجودة وسليمة، وأن عدد المواد القانونية الفعالة المستخدمة في قاعدة RAG بلغ <b>211</b> مادة. 
كما تم الاحتفاظ بالمواد الملغاة وعددها <b>38</b> مادة في الأرشيف فقط لأغراض التوثيق والمراجعة، دون إدخالها في الفهرسة أو الاسترجاع.
</p>

<p>
أظهرت فحوصات السلامة أن عدد المواد الملغاة المفهرسة داخل RAG يساوي <b>0</b>، وهذا يؤكد أن قاعدة المعرفة التجريبية لا تحتوي على مواد قانونية غير فعالة أو ملغاة. 
كما تم التحقق من أن تقسيم FAQ إلى فهرسة وتقييم لا يحتوي على تسريب مباشر أو تكرار مؤثر.
</p>

<p>
تم أيضًا حفظ تقرير جودة البيانات بصيغتين، CSV و Excel، داخل مجلد التقارير، مما يوفر سجلًا قابلًا للمراجعة والتوثيق الأكاديمي.
</p>

<div style="
    background-color:#E2F0D9;
    border-right:4px solid #375623;
    padding:12px;
    margin-top:12px;
    border-radius:5px;
">
<b>Conclusion:</b><br>
The data quality validation was completed successfully. All checks passed with no warnings or failures, indicating that the cleaned legal and FAQ datasets are methodologically safe and ready for the next RAG construction and retrieval evaluation stages.
</div>

</div>


## Stage 08 — Build a Safe Unified RAG Knowledge Base

This stage constructs the experimental RAG knowledge base from two approved sources:

1. Active Saudi Labor Law articles.
2. FAQ records assigned to the indexing subset.

Repealed legal articles are excluded from the experimental knowledge base and stored separately as an archive. FAQ evaluation records are also excluded to prevent leakage.

Two knowledge-base versions are produced:

- An **experimental no-leakage KB** used for retrieval evaluation.
- A **full production KB** that can be used after evaluation is completed.


In [16]:
# =========================================================
# Stage 08 - Build Safe Unified RAG Knowledge Base Dataset
# Corrected production FAQ source after Stage 05.5 / Stage 06.5
# =========================================================

from IPython.display import display, Markdown
import pandas as pd
import numpy as np
import re
from pathlib import Path

# ---------------------------------------------------------
# 1) Directory safety
# ---------------------------------------------------------

if "FINAL_DIR" not in globals():
    FINAL_DIR = Path("03_final")

if "REPORTS_DIR" not in globals():
    REPORTS_DIR = Path("05_reports")

FINAL_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# 2) Required input validation
# ---------------------------------------------------------

required_objects = [
    "df_articles",
    "df_faq_indexing",
    "df_faq_evaluation",
]

missing_objects = [
    obj for obj in required_objects
    if obj not in globals() or not isinstance(globals()[obj], pd.DataFrame)
]

if missing_objects:
    raise NameError(
        "Missing required objects before Stage 08: "
        + ", ".join(missing_objects)
        + ". Run Stage 05.5, Stage 06, and Stage 06.5 first."
    )

# Active legal articles
if "df_articles_active" in globals() and isinstance(df_articles_active, pd.DataFrame):
    active_articles_source = df_articles_active.copy()
else:
    if "article_status" not in df_articles.columns:
        raise KeyError("df_articles must contain article_status or df_articles_active must exist.")
    active_articles_source = df_articles[
        df_articles["article_status"].astype(str).eq("active")
    ].copy()

# Repealed legal articles
if "df_articles_repealed" in globals() and isinstance(df_articles_repealed, pd.DataFrame):
    repealed_articles_source = df_articles_repealed.copy()
else:
    if "article_status" not in df_articles.columns:
        repealed_articles_source = pd.DataFrame(columns=df_articles.columns)
    else:
        repealed_articles_source = df_articles[
            df_articles["article_status"].astype(str).eq("repealed")
        ].copy()

assert len(active_articles_source) > 0, "No active legal articles available for RAG."
assert len(df_faq_indexing) > 0, "FAQ indexing set is empty. Run Stage 06 first."
assert len(df_faq_evaluation) > 0, "FAQ evaluation set is empty. Run Stage 06.5 first."

# ---------------------------------------------------------
# 3) Helper functions
# ---------------------------------------------------------

def clean_display_value(value):
    if value is None:
        return "N/A"
    try:
        if pd.isna(value):
            return "N/A"
    except Exception:
        pass

    value = str(value).strip()
    if value.lower() in ["", "nan", "none", "<na>", "n/a"]:
        return "N/A"
    return value


def short_text(text, max_len=140):
    text = clean_display_value(text)
    if text == "N/A":
        return "N/A"
    text = re.sub(r"\s+", " ", text).strip()
    return text[:max_len] + ("..." if len(text) > max_len else "")


def hide_index_safe(styler):
    try:
        return styler.hide(axis="index")
    except Exception:
        try:
            return styler.hide_index()
        except Exception:
            return styler


def apply_style_compatible(styler, func, subset):
    if hasattr(styler, "map"):
        return styler.map(func, subset=subset)
    return styler.applymap(func, subset=subset)


def color_status(value):
    value = str(value)
    if value == "PASS":
        return "background-color:#E2F0D9; color:#375623; font-weight:bold;"
    if value == "WARN":
        return "background-color:#FFF2CC; color:#7F6000; font-weight:bold;"
    if value == "FAIL":
        return "background-color:#FCE4D6; color:#9C0006; font-weight:bold;"
    return ""


def professional_table(df, caption="", width="100%", font_size="12px"):
    styler = (
        df.style
        .set_caption(caption)
        .set_table_styles([
            {
                "selector": "caption",
                "props": [
                    ("caption-side", "top"),
                    ("font-size", "17px"),
                    ("font-weight", "bold"),
                    ("color", "#1F4E78"),
                    ("text-align", "center"),
                    ("padding", "10px")
                ]
            },
            {
                "selector": "th",
                "props": [
                    ("background-color", "#1F4E78"),
                    ("color", "white"),
                    ("font-weight", "bold"),
                    ("text-align", "center"),
                    ("border", "1px solid #D9E2F3"),
                    ("padding", "8px"),
                    ("white-space", "normal")
                ]
            },
            {
                "selector": "td",
                "props": [
                    ("text-align", "center"),
                    ("border", "1px solid #D9E2F3"),
                    ("padding", "7px"),
                    ("vertical-align", "middle"),
                    ("white-space", "normal")
                ]
            },
            {
                "selector": "tbody tr:nth-child(even)",
                "props": [("background-color", "#F8FBFD")]
            },
            {
                "selector": "tbody tr:nth-child(odd)",
                "props": [("background-color", "#FFFFFF")]
            },
            {
                "selector": "table",
                "props": [
                    ("border-collapse", "collapse"),
                    ("width", width),
                    ("font-family", "Arial, Tahoma, sans-serif"),
                    ("font-size", font_size),
                    ("table-layout", "fixed"),
                    ("margin", "12px 0")
                ]
            }
        ])
    )
    return hide_index_safe(styler)


def get_col(df, col, default=""):
    """
    Safe column getter.
    If default is a Series, return it aligned to df length.
    """
    if col in df.columns:
        return df[col].reset_index(drop=True)

    if isinstance(default, pd.Series):
        return default.reset_index(drop=True)

    return pd.Series([default] * len(df))


def count_source(df, source_type):
    if not isinstance(df, pd.DataFrame) or "source_type" not in df.columns:
        return 0
    return int((df["source_type"].astype(str) == source_type).sum())


def count_repealed(df):
    if not isinstance(df, pd.DataFrame):
        return 0
    if "source_type" not in df.columns or "article_status" not in df.columns:
        return 0
    return int(
        (
            (df["source_type"].astype(str) == "legal_article")
            &
            (df["article_status"].astype(str) == "repealed")
        ).sum()
    )


def safe_percentage(part, total):
    if total == 0:
        return 0.0
    return round((part / total) * 100, 2)

# ---------------------------------------------------------
# 4) Stage explanation
# ---------------------------------------------------------

display(Markdown("""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:14px;
    margin:12px 0;
    border-radius:6px;
">

<h3 style="color:#1F4E78; margin-top:0;">
Stage 08 — Safe Unified RAG Knowledge Base Dataset
</h3>

<p>
This stage builds the experimental RAG knowledge base using only active Saudi Labor Law articles and the FAQ indexing subset.
The held-out FAQ evaluation records remain excluded from the experimental knowledge base to prevent leakage.
</p>

<p>
The production knowledge base is rebuilt from the latest strict labor-filtered FAQ records, not from old cached variables.
Repealed legal articles are preserved only in an audit archive and are not indexed for retrieval.
</p>

</div>
"""))

# ---------------------------------------------------------
# 5) Build standardized legal article records
# ---------------------------------------------------------

def build_article_records(df_articles_part, usage_label="indexing_allowed_active_law"):
    df = df_articles_part.copy().reset_index(drop=True)

    if len(df) == 0:
        return pd.DataFrame(columns=[
            "source_type", "source_name", "legal_category", "bab_label",
            "bab_title", "fasl", "article_title", "article_number",
            "article_number_label", "article_number_int", "article_status",
            "rag_usage", "question", "answer", "beneficiaries", "sector",
            "subsite", "original_text", "source_url", "document_unit_id",
            "text_for_indexing", "normalized_text"
        ])

    required_article_cols = ["article_text", "document_unit_id"]
    missing = [c for c in required_article_cols if c not in df.columns]
    if missing:
        raise KeyError(f"Missing required legal article columns: {missing}")

    article_text_series = df["article_text"].astype(str).reset_index(drop=True)

    out = pd.DataFrame({
        "source_type": get_col(df, "source_type", "legal_article"),
        "source_name": get_col(df, "source_name", "نظام العمل السعودي"),
        "legal_category": get_col(df, "legal_category", ""),
        "bab_label": get_col(df, "bab_label", ""),
        "bab_title": get_col(df, "bab_title", ""),
        "fasl": get_col(df, "fasl", ""),
        "article_title": get_col(df, "article_title", ""),
        "article_number": get_col(df, "article_number", ""),
        "article_number_label": get_col(df, "article_number_label", ""),
        "article_number_int": get_col(df, "article_number_int", ""),
        "article_status": get_col(df, "article_status", ""),
        "rag_usage": usage_label,
        "question": "",
        "answer": "",
        "beneficiaries": "",
        "sector": "",
        "subsite": "",
        "original_text": get_col(df, "article_text", ""),
        "source_url": get_col(df, "source_url", ""),
        "document_unit_id": get_col(df, "document_unit_id", ""),
        "text_for_indexing": get_col(df, "text_for_indexing", article_text_series),
        "normalized_text": get_col(df, "normalized_text", article_text_series),
    })

    return out


# ---------------------------------------------------------
# 6) Build standardized FAQ records
# ---------------------------------------------------------

def build_faq_records(df_faq_part, usage_label="indexing_allowed_faq"):
    df = df_faq_part.copy().reset_index(drop=True)

    if len(df) == 0:
        return pd.DataFrame(columns=[
            "source_type", "source_name", "legal_category", "bab_label",
            "bab_title", "fasl", "article_title", "article_number",
            "article_number_label", "article_number_int", "article_status",
            "rag_usage", "question", "answer", "beneficiaries", "sector",
            "subsite", "original_text", "source_url", "document_unit_id",
            "text_for_indexing", "normalized_text"
        ])

    required_faq_cols = ["question", "answer", "document_unit_id"]
    missing = [c for c in required_faq_cols if c not in df.columns]
    if missing:
        raise KeyError(f"Missing required FAQ columns: {missing}")

    if "source_type" not in df.columns:
        df["source_type"] = "faq"

    if "source_name" not in df.columns:
        df["source_name"] = "HRSD FAQ"

    if "legal_category" not in df.columns:
        df["legal_category"] = "faq"

    if "text_for_indexing" not in df.columns:
        df["text_for_indexing"] = df.apply(
            lambda r: f"السؤال: {r.get('question', '')}\nالإجابة: {r.get('answer', '')}",
            axis=1
        )

    if "normalized_text" not in df.columns:
        if "normalize_arabic_for_search" in globals():
            df["normalized_text"] = df["text_for_indexing"].apply(normalize_arabic_for_search)
        else:
            df["normalized_text"] = df["text_for_indexing"].astype(str)

    out = pd.DataFrame({
        "source_type": get_col(df, "source_type", "faq"),
        "source_name": get_col(df, "source_name", "HRSD FAQ"),
        "legal_category": get_col(df, "legal_category", "faq"),
        "bab_label": "",
        "bab_title": "",
        "fasl": "",
        "article_title": "",
        "article_number": "",
        "article_number_label": "",
        "article_number_int": "",
        "article_status": "not_applicable",
        "rag_usage": usage_label,
        "question": get_col(df, "question", ""),
        "answer": get_col(df, "answer", ""),
        "beneficiaries": get_col(df, "beneficiaries", ""),
        "sector": get_col(df, "sector", ""),
        "subsite": get_col(df, "subsite", ""),
        "original_text": get_col(df, "answer", ""),
        "source_url": get_col(df, "page_url", get_col(df, "source_url", "")),
        "document_unit_id": get_col(df, "document_unit_id", ""),
        "text_for_indexing": get_col(df, "text_for_indexing", ""),
        "normalized_text": get_col(df, "normalized_text", ""),
    })

    return out

# ---------------------------------------------------------
# 7) Build source records
# ---------------------------------------------------------

article_records_active = build_article_records(
    active_articles_source,
    usage_label="indexing_allowed_active_law"
)

article_records_repealed_archive = build_article_records(
    repealed_articles_source,
    usage_label="archive_only_not_indexed"
)

faq_indexing_records = build_faq_records(
    df_faq_indexing,
    usage_label="indexing_allowed_faq_no_leakage"
)

# ---------------------------------------------------------
# 8) Correct full production FAQ source
# ---------------------------------------------------------
# IMPORTANT:
# Do NOT use old df_faq_full_production.
# Use the latest Stage 05.5 strict domain-filtered FAQ if available.
# Otherwise use indexing + cleaned evaluation after Stage 06.5.

if (
    "df_faq_domain_filtered" in globals()
    and isinstance(df_faq_domain_filtered, pd.DataFrame)
    and "keep_for_rag" in df_faq_domain_filtered.columns
):
    full_faq_source = (
        df_faq_domain_filtered[
            df_faq_domain_filtered["keep_for_rag"] == True
        ]
        .copy()
        .reset_index(drop=True)
    )
    full_faq_source_name = "df_faq_domain_filtered where keep_for_rag=True"

elif (
    "df_faq_for_rag" in globals()
    and isinstance(df_faq_for_rag, pd.DataFrame)
    and len(df_faq_for_rag) > 0
):
    full_faq_source = df_faq_for_rag.copy().reset_index(drop=True)
    full_faq_source_name = "df_faq_for_rag from Stage 05.5"

else:
    full_faq_source = (
        pd.concat([df_faq_indexing, df_faq_evaluation], ignore_index=True)
        .drop_duplicates(subset=["document_unit_id"])
        .reset_index(drop=True)
    )
    full_faq_source_name = "df_faq_indexing + df_faq_evaluation after Stage 06.5"

assert len(full_faq_source) > 0, "Full production FAQ source is empty."

faq_all_records = build_faq_records(
    full_faq_source,
    usage_label="indexing_allowed_faq_full_production_labor_filtered"
)

# ---------------------------------------------------------
# 9) Experimental KB - no leakage
# Active law articles + FAQ indexing subset only
# ---------------------------------------------------------

df_kb_experiment = pd.concat(
    [article_records_active, faq_indexing_records],
    ignore_index=True
)

before_kb_exp_dedup = len(df_kb_experiment)

df_kb_experiment = (
    df_kb_experiment
    .drop_duplicates(subset=["document_unit_id"])
    .reset_index(drop=True)
)

after_kb_exp_dedup = len(df_kb_experiment)
kb_exp_duplicates_removed = before_kb_exp_dedup - after_kb_exp_dedup

df_kb_experiment.insert(
    0,
    "record_id",
    range(1, len(df_kb_experiment) + 1)
)

# ---------------------------------------------------------
# 10) Full production KB
# Active law articles + latest strict labor-filtered FAQ
# ---------------------------------------------------------

df_kb_full_production = pd.concat(
    [article_records_active, faq_all_records],
    ignore_index=True
)

before_kb_prod_dedup = len(df_kb_full_production)

df_kb_full_production = (
    df_kb_full_production
    .drop_duplicates(subset=["document_unit_id"])
    .reset_index(drop=True)
)

after_kb_prod_dedup = len(df_kb_full_production)
kb_prod_duplicates_removed = before_kb_prod_dedup - after_kb_prod_dedup

df_kb_full_production.insert(
    0,
    "record_id",
    range(1, len(df_kb_full_production) + 1)
)

# ---------------------------------------------------------
# 11) Repealed archive only
# ---------------------------------------------------------

df_repealed_archive_for_audit = (
    article_records_repealed_archive
    .copy()
    .reset_index(drop=True)
)

df_repealed_archive_for_audit.insert(
    0,
    "record_id",
    range(1, len(df_repealed_archive_for_audit) + 1)
)

# ---------------------------------------------------------
# 12) Safety checks
# ---------------------------------------------------------

repealed_in_experiment_kb = int(
    (
        (df_kb_experiment["source_type"].astype(str) == "legal_article")
        &
        (df_kb_experiment["article_status"].astype(str) == "repealed")
    ).sum()
)

repealed_in_production_kb = int(
    (
        (df_kb_full_production["source_type"].astype(str) == "legal_article")
        &
        (df_kb_full_production["article_status"].astype(str) == "repealed")
    ).sum()
)

faq_eval_ids = set(df_faq_evaluation["document_unit_id"].astype(str))

faq_leakage_in_experiment = int(
    len(
        set(
            df_kb_experiment[
                df_kb_experiment["source_type"].astype(str) == "faq"
            ]["document_unit_id"].astype(str)
        )
        &
        faq_eval_ids
    )
)

assert repealed_in_experiment_kb == 0, "Repealed articles found in experimental RAG KB."
assert repealed_in_production_kb == 0, "Repealed articles found in production RAG KB."
assert faq_leakage_in_experiment == 0, "FAQ evaluation records found inside experimental KB."

content_leaks_removed_path = FINAL_DIR / "faq_eval_content_leaks_removed.csv"
possible_leaks_review_path = FINAL_DIR / "faq_eval_possible_leaks_review.csv"

content_leaks_removed_count = (
    len(pd.read_csv(content_leaks_removed_path, encoding="utf-8-sig"))
    if content_leaks_removed_path.exists()
    else 0
)

possible_content_leaks_count = (
    len(pd.read_csv(possible_leaks_review_path, encoding="utf-8-sig"))
    if possible_leaks_review_path.exists()
    else 0
)

# ---------------------------------------------------------
# 13) Save outputs
# ---------------------------------------------------------

experiment_path = FINAL_DIR / "rag_knowledge_base_dataset_experiment_no_leakage_active_law_only.csv"
production_path = FINAL_DIR / "rag_knowledge_base_dataset_full_production_active_law_only.csv"
repealed_archive_path = FINAL_DIR / "legal_articles_repealed_archive_not_indexed.csv"

experiment_xlsx_path = FINAL_DIR / "rag_knowledge_base_dataset_experiment_no_leakage_active_law_only.xlsx"
production_xlsx_path = FINAL_DIR / "rag_knowledge_base_dataset_full_production_active_law_only.xlsx"
repealed_archive_xlsx_path = FINAL_DIR / "legal_articles_repealed_archive_not_indexed.xlsx"

legacy_experiment_path = FINAL_DIR / "rag_knowledge_base_dataset_experiment_no_leakage.csv"
legacy_production_path = FINAL_DIR / "rag_knowledge_base_dataset_full_production.csv"

df_kb_experiment.to_csv(experiment_path, index=False, encoding="utf-8-sig")
df_kb_full_production.to_csv(production_path, index=False, encoding="utf-8-sig")
df_repealed_archive_for_audit.to_csv(repealed_archive_path, index=False, encoding="utf-8-sig")

df_kb_experiment.to_csv(legacy_experiment_path, index=False, encoding="utf-8-sig")
df_kb_full_production.to_csv(legacy_production_path, index=False, encoding="utf-8-sig")

df_kb_experiment.to_excel(experiment_xlsx_path, index=False)
df_kb_full_production.to_excel(production_xlsx_path, index=False)
df_repealed_archive_for_audit.to_excel(repealed_archive_xlsx_path, index=False)

# Compatibility aliases for later stages
kb_experimental = df_kb_experiment.copy()
kb_full_production = df_kb_full_production.copy()

# ---------------------------------------------------------
# 14) Output summary
# ---------------------------------------------------------

display(Markdown("""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:14px;
    margin:12px 0;
    border-radius:6px;
">

<h3 style="color:#1F4E78; margin-top:0;">
Stage 08 Output Summary — Safe Unified RAG Knowledge Base
</h3>

<p>
This stage generated the experimental RAG knowledge base, the full production knowledge base, and the repealed legal article archive.
The experimental knowledge base includes active legal articles and FAQ indexing records only, while held-out FAQ evaluation records remain excluded.
</p>

</div>
"""))

kb_summary_df = pd.DataFrame([
    {
        "Dataset": "Experimental RAG Knowledge Base",
        "Purpose": "Used for retrieval experiments without evaluation leakage",
        "Records": len(df_kb_experiment),
        "Columns": df_kb_experiment.shape[1],
        "Legal Articles": count_source(df_kb_experiment, "legal_article"),
        "FAQ Records": count_source(df_kb_experiment, "faq"),
        "Repealed Articles": count_repealed(df_kb_experiment),
        "Duplicates Removed": kb_exp_duplicates_removed,
        "RAG Usage": "Active law + FAQ indexing only",
        "Output File": experiment_path.name
    },
    {
        "Dataset": "Full Production RAG Knowledge Base",
        "Purpose": "Used after evaluation for full production indexing",
        "Records": len(df_kb_full_production),
        "Columns": df_kb_full_production.shape[1],
        "Legal Articles": count_source(df_kb_full_production, "legal_article"),
        "FAQ Records": count_source(df_kb_full_production, "faq"),
        "Repealed Articles": count_repealed(df_kb_full_production),
        "Duplicates Removed": kb_prod_duplicates_removed,
        "RAG Usage": "Active law + latest strict labor-filtered FAQ",
        "Output File": production_path.name
    },
    {
        "Dataset": "Repealed Articles Archive",
        "Purpose": "Audit archive only and excluded from retrieval",
        "Records": len(df_repealed_archive_for_audit),
        "Columns": df_repealed_archive_for_audit.shape[1],
        "Legal Articles": count_source(df_repealed_archive_for_audit, "legal_article"),
        "FAQ Records": count_source(df_repealed_archive_for_audit, "faq"),
        "Repealed Articles": count_repealed(df_repealed_archive_for_audit),
        "Duplicates Removed": 0,
        "RAG Usage": "Archive only - not indexed",
        "Output File": repealed_archive_path.name
    }
])

display(
    professional_table(
        kb_summary_df,
        caption="Table 8.1 — Knowledge Base Datasets Generated in Stage 08",
        width="100%",
        font_size="11.5px"
    )
)

# ---------------------------------------------------------
# 15) Safety and leakage validation summary
# ---------------------------------------------------------

safety_df = pd.DataFrame([
    {
        "Safety Check": "Repealed articles inside experimental KB",
        "Value": repealed_in_experiment_kb,
        "Expected": 0,
        "Status": "PASS" if repealed_in_experiment_kb == 0 else "FAIL",
        "Interpretation": "The experimental KB does not index repealed legal provisions."
    },
    {
        "Safety Check": "Repealed articles inside production KB",
        "Value": repealed_in_production_kb,
        "Expected": 0,
        "Status": "PASS" if repealed_in_production_kb == 0 else "FAIL",
        "Interpretation": "The production KB includes only active legal articles."
    },
    {
        "Safety Check": "FAQ evaluation leakage inside experimental KB",
        "Value": faq_leakage_in_experiment,
        "Expected": 0,
        "Status": "PASS" if faq_leakage_in_experiment == 0 else "FAIL",
        "Interpretation": "Held-out FAQ evaluation records are not included in the experimental KB."
    },
    {
        "Safety Check": "Content-level FAQ leaks removed earlier",
        "Value": content_leaks_removed_count,
        "Expected": ">= 0",
        "Status": "PASS",
        "Interpretation": "Near-duplicate FAQ evaluation records removed in Stage 06.5 before KB construction."
    },
    {
        "Safety Check": "Possible content leaks flagged for review",
        "Value": possible_content_leaks_count,
        "Expected": ">= 0",
        "Status": "PASS",
        "Interpretation": "Borderline FAQ similarities saved for audit and review."
    }
])

safety_style = professional_table(
    safety_df,
    caption="Table 8.2 — RAG Safety and Leakage Validation",
    width="100%",
    font_size="11.5px"
)

safety_style = apply_style_compatible(
    safety_style,
    color_status,
    subset=["Status"]
)

display(safety_style)

# ---------------------------------------------------------
# 16) Experimental KB source composition
# ---------------------------------------------------------

source_composition_df = (
    df_kb_experiment["source_type"]
    .value_counts()
    .rename_axis("Source Type")
    .reset_index(name="Records")
)

total_experimental_records = int(source_composition_df["Records"].sum())

source_composition_df["Percentage"] = source_composition_df["Records"].apply(
    lambda x: safe_percentage(x, total_experimental_records)
)

display(
    professional_table(
        source_composition_df,
        caption="Table 8.3 — Experimental KB Records by Source Type",
        width="65%",
        font_size="12px"
    )
)

# ---------------------------------------------------------
# 17) Saved files manifest
# ---------------------------------------------------------

saved_files_df = pd.DataFrame([
    {
        "File Type": "Experimental KB - CSV",
        "Path": str(experiment_path),
        "Purpose": "Main dataset for retrieval experiments without FAQ leakage"
    },
    {
        "File Type": "Experimental KB - Excel",
        "Path": str(experiment_xlsx_path),
        "Purpose": "Readable Excel version of the experimental RAG knowledge base"
    },
    {
        "File Type": "Production KB - CSV",
        "Path": str(production_path),
        "Purpose": "Full production knowledge base after evaluation"
    },
    {
        "File Type": "Production KB - Excel",
        "Path": str(production_xlsx_path),
        "Purpose": "Readable Excel version of the full production knowledge base"
    },
    {
        "File Type": "Repealed Archive - CSV",
        "Path": str(repealed_archive_path),
        "Purpose": "Audit archive for repealed legal articles, not indexed"
    },
    {
        "File Type": "Repealed Archive - Excel",
        "Path": str(repealed_archive_xlsx_path),
        "Purpose": "Readable Excel version of repealed legal article archive"
    },
    {
        "File Type": "Legacy Experimental KB - CSV",
        "Path": str(legacy_experiment_path),
        "Purpose": "Compatibility file for later RAG stages"
    },
    {
        "File Type": "Legacy Production KB - CSV",
        "Path": str(legacy_production_path),
        "Purpose": "Compatibility file for later RAG stages"
    }
])

display(
    professional_table(
        saved_files_df,
        caption="Table 8.4 — Stage 08 Saved Output Files",
        width="100%",
        font_size="11.5px"
    )
)

stage08_manifest_path = REPORTS_DIR / "stage08_output_manifest.csv"
stage08_manifest_xlsx_path = REPORTS_DIR / "stage08_output_manifest.xlsx"

saved_files_df.to_csv(stage08_manifest_path, index=False, encoding="utf-8-sig")
saved_files_df.to_excel(stage08_manifest_xlsx_path, index=False)

# ---------------------------------------------------------
# 18) Preview of experimental KB
# ---------------------------------------------------------

preview_cols = [
    "record_id",
    "source_type",
    "rag_usage",
    "article_number",
    "article_status",
    "question",
    "source_name",
    "document_unit_id"
]

preview_cols = [
    col for col in preview_cols
    if col in df_kb_experiment.columns
]

preview_df = df_kb_experiment[preview_cols].head(10).copy()

for col in ["question", "source_name"]:
    if col in preview_df.columns:
        preview_df[col] = preview_df[col].apply(lambda x: short_text(x, 120))

preview_style = professional_table(
    preview_df,
    caption="Table 8.5 — First 10 Records from the Experimental KB",
    width="100%",
    font_size="11.5px"
)

rtl_cols = [c for c in ["question", "source_name"] if c in preview_df.columns]

if rtl_cols:
    preview_style = preview_style.set_properties(
        subset=rtl_cols,
        **{
            "text-align": "right",
            "direction": "rtl",
            "line-height": "1.7"
        }
    )

if "document_unit_id" in preview_df.columns:
    preview_style = preview_style.set_properties(
        subset=["document_unit_id"],
        **{
            "text-align": "left",
            "direction": "ltr",
            "font-size": "10px"
        }
    )

display(preview_style)

# ---------------------------------------------------------
# 19) Final completion message
# ---------------------------------------------------------

display(Markdown(f"""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#E2F0D9;
    border-right:5px solid #375623;
    padding:14px;
    margin:12px 0;
    border-radius:6px;
">

<b>Stage 08 completed successfully.</b><br>

Full production FAQ source used: <b>{full_faq_source_name}</b><br>
Full production FAQ records used: <b>{len(full_faq_source)}</b><br><br>

Experimental KB records: <b>{len(df_kb_experiment)}</b><br>
Full production KB records: <b>{len(df_kb_full_production)}</b><br>
Repealed archive records: <b>{len(df_repealed_archive_for_audit)}</b><br>
FAQ evaluation leakage inside experimental KB: <b>{faq_leakage_in_experiment}</b><br>
Repealed legal articles inside experimental KB: <b>{repealed_in_experiment_kb}</b><br>

All safety checks passed.

</div>
"""))

print("Stage 08 completed successfully.")
print("Full production FAQ source used:", full_faq_source_name)
print("Full production FAQ records used:", len(full_faq_source))
print(f"Experimental KB records: {len(df_kb_experiment):,}")
print(f"Full production KB records: {len(df_kb_full_production):,}")
print(f"Repealed archive records: {len(df_repealed_archive_for_audit):,}")
print("FAQ evaluation leakage inside experimental KB:", faq_leakage_in_experiment)
print("Repealed articles inside experimental KB:", repealed_in_experiment_kb)
print("All safety checks passed.")


<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:14px;
    margin:12px 0;
    border-radius:6px;
">

<h3 style="color:#1F4E78; margin-top:0;">
Stage 08 — Safe Unified RAG Knowledge Base Dataset
</h3>

<p>
This stage builds the experimental RAG knowledge base using only active Saudi Labor Law articles and the FAQ indexing subset.
The held-out FAQ evaluation records remain excluded from the experimental knowledge base to prevent leakage.
</p>

<p>
The production knowledge base is rebuilt from the latest strict labor-filtered FAQ records, not from old cached variables.
Repealed legal articles are preserved only in an audit archive and are not indexed for retrieval.
</p>

</div>



<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:14px;
    margin:12px 0;
    border-radius:6px;
">

<h3 style="color:#1F4E78; margin-top:0;">
Stage 08 Output Summary — Safe Unified RAG Knowledge Base
</h3>

<p>
This stage generated the experimental RAG knowledge base, the full production knowledge base, and the repealed legal article archive.
The experimental knowledge base includes active legal articles and FAQ indexing records only, while held-out FAQ evaluation records remain excluded.
</p>

</div>


Dataset,Purpose,Records,Columns,Legal Articles,FAQ Records,Repealed Articles,Duplicates Removed,RAG Usage,Output File
Experimental RAG Knowledge Base,Used for retrieval experiments without evaluation leakage,489,23,211,278,0,0,Active law + FAQ indexing only,rag_knowledge_base_dataset_experiment_no_leakage_active_law_only.csv
Full Production RAG Knowledge Base,Used after evaluation for full production indexing,613,23,211,402,0,0,Active law + latest strict labor-filtered FAQ,rag_knowledge_base_dataset_full_production_active_law_only.csv
Repealed Articles Archive,Audit archive only and excluded from retrieval,38,23,38,0,38,0,Archive only - not indexed,legal_articles_repealed_archive_not_indexed.csv


Safety Check,Value,Expected,Status,Interpretation
Repealed articles inside experimental KB,0,0,PASS,The experimental KB does not index repealed legal provisions.
Repealed articles inside production KB,0,0,PASS,The production KB includes only active legal articles.
FAQ evaluation leakage inside experimental KB,0,0,PASS,Held-out FAQ evaluation records are not included in the experimental KB.
Content-level FAQ leaks removed earlier,2,>= 0,PASS,Near-duplicate FAQ evaluation records removed in Stage 06.5 before KB construction.
Possible content leaks flagged for review,0,>= 0,PASS,Borderline FAQ similarities saved for audit and review.


Source Type,Records,Percentage
faq,278,56.850000
legal_article,211,43.150000


File Type,Path,Purpose
Experimental KB - CSV,saudi_labor_law_voice_agent_project\03_final\rag_knowledge_base_dataset_experiment_no_leakage_active_law_only.csv,Main dataset for retrieval experiments without FAQ leakage
Experimental KB - Excel,saudi_labor_law_voice_agent_project\03_final\rag_knowledge_base_dataset_experiment_no_leakage_active_law_only.xlsx,Readable Excel version of the experimental RAG knowledge base
Production KB - CSV,saudi_labor_law_voice_agent_project\03_final\rag_knowledge_base_dataset_full_production_active_law_only.csv,Full production knowledge base after evaluation
Production KB - Excel,saudi_labor_law_voice_agent_project\03_final\rag_knowledge_base_dataset_full_production_active_law_only.xlsx,Readable Excel version of the full production knowledge base
Repealed Archive - CSV,saudi_labor_law_voice_agent_project\03_final\legal_articles_repealed_archive_not_indexed.csv,"Audit archive for repealed legal articles, not indexed"
Repealed Archive - Excel,saudi_labor_law_voice_agent_project\03_final\legal_articles_repealed_archive_not_indexed.xlsx,Readable Excel version of repealed legal article archive
Legacy Experimental KB - CSV,saudi_labor_law_voice_agent_project\03_final\rag_knowledge_base_dataset_experiment_no_leakage.csv,Compatibility file for later RAG stages
Legacy Production KB - CSV,saudi_labor_law_voice_agent_project\03_final\rag_knowledge_base_dataset_full_production.csv,Compatibility file for later RAG stages


record_id,source_type,rag_usage,article_number,article_status,question,source_name,document_unit_id
1,legal_article,indexing_allowed_active_law,الأولى,active,N/A,نظام العمل السعودي,d28fed1c707fb8a6b354171cec6a310c1164782775f0d91e9d13d8b8198af462
2,legal_article,indexing_allowed_active_law,الثانية,active,N/A,نظام العمل السعودي,4f2961046b5fffdc637e2b7de4dfb1fd862ccc7b9f3069349681b04ff17df638
3,legal_article,indexing_allowed_active_law,الثالثة,active,N/A,نظام العمل السعودي,82f3c9791caa01e20aff6f25a0e169fbd07f8aa138625fe674d6df6824a31fda
4,legal_article,indexing_allowed_active_law,الرابعة,active,N/A,نظام العمل السعودي,3457807712bce4c43b55e497232052e55933f7fde19df543c46153e86ca95090
5,legal_article,indexing_allowed_active_law,الخامسة,active,N/A,نظام العمل السعودي,a03900ffe9b536bc7774b84e1eb41826aacf458ed3920b36e463e08ac8bde4cd
6,legal_article,indexing_allowed_active_law,السادسة,active,N/A,نظام العمل السعودي,1157999dfa071a7320606edb9b747429aa71cb7707dcaf5be88784d6c4e2f7ca
7,legal_article,indexing_allowed_active_law,السابعة,active,N/A,نظام العمل السعودي,c91b6c3872f87c320cac960229321ad90710d58e53361f20475d343cc1d88d2d
8,legal_article,indexing_allowed_active_law,الثامنة,active,N/A,نظام العمل السعودي,01eaca9a4fb7e0f5416045832b295fd64085a920f19266ff5bbc0236deddf41d
9,legal_article,indexing_allowed_active_law,التاسعة,active,N/A,نظام العمل السعودي,a0d5a1431be7e4fa5a4411f0ff8222e86afaf8a6e35b994667aae08b4cdfe963
10,legal_article,indexing_allowed_active_law,العاشرة,active,N/A,نظام العمل السعودي,518e66de899a12d4203d77f4bdd3cdac022db77736feac05d635149e954e8dca



<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#E2F0D9;
    border-right:5px solid #375623;
    padding:14px;
    margin:12px 0;
    border-radius:6px;
">

<b>Stage 08 completed successfully.</b><br>

Full production FAQ source used: <b>df_faq_domain_filtered where keep_for_rag=True</b><br>
Full production FAQ records used: <b>402</b><br><br>

Experimental KB records: <b>489</b><br>
Full production KB records: <b>613</b><br>
Repealed archive records: <b>38</b><br>
FAQ evaluation leakage inside experimental KB: <b>0</b><br>
Repealed legal articles inside experimental KB: <b>0</b><br>

All safety checks passed.

</div>


Stage 08 completed successfully.
Full production FAQ source used: df_faq_domain_filtered where keep_for_rag=True
Full production FAQ records used: 402
Experimental KB records: 489
Full production KB records: 613
Repealed archive records: 38
FAQ evaluation leakage inside experimental KB: 0
Repealed articles inside experimental KB: 0
All safety checks passed.


In [17]:
from IPython.display import display, Markdown

display(Markdown("""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:16px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:16px;
    margin:14px 0;
    border-radius:6px;
">

<h3 style="color:#1F4E78; margin-top:0;">
Interpretation — Safe Unified RAG Knowledge Base
</h3>

<p>
توضح هذه المرحلة بناء قاعدة معرفة موحدة وآمنة لاستخدامها في نظام RAG. 
تم إنشاء قاعدة المعرفة التجريبية باستخدام المواد القانونية الفعالة فقط، بالإضافة إلى سجلات FAQ المخصصة للفهرسة، مع استبعاد سجلات FAQ المحجوزة للتقييم.
</p>

<p>
بلغ حجم قاعدة المعرفة التجريبية <b>489</b> سجلًا، موزعة إلى <b>211</b> مادة قانونية فعالة و <b>278</b> سجل FAQ مخصص للفهرسة. 
وهذا يؤكد أن قاعدة التجارب تحتوي فقط على المصادر المسموح استخدامها أثناء تقييم الاسترجاع.
</p>

<p>
كما تم إنشاء قاعدة معرفة إنتاجية كاملة تحتوي على <b>613</b> سجلًا، وتتكون من <b>211</b> مادة قانونية فعالة و <b>402</b> سجل FAQ بعد الفلترة الصارمة. 
وتُستخدم هذه النسخة لاحقًا بعد انتهاء التقييم، لأنها تحتوي على جميع سجلات FAQ النظيفة وليس فقط مجموعة الفهرسة التجريبية.
</p>

<p>
أظهرت فحوصات السلامة أن عدد المواد الملغاة داخل قاعدة المعرفة التجريبية وقاعدة الإنتاج يساوي <b>0</b>. 
وهذا يؤكد أن المواد الملغاة محفوظة فقط في أرشيف مستقل للمراجعة ولا تدخل في الفهرسة أو الاسترجاع.
</p>

<p>
كما أظهر فحص التسريب أن عدد سجلات FAQ الخاصة بالتقييم داخل قاعدة المعرفة التجريبية يساوي <b>0</b>. 
بالإضافة إلى ذلك، تم سابقًا حذف <b>2</b> من أسئلة FAQ المتشابهة جدًا مع سجلات الفهرسة في Stage 06.5، مما يقلل خطر التسريب النصي أو المعنوي.
</p>

<div style="
    background-color:#E2F0D9;
    border-right:4px solid #375623;
    padding:12px;
    margin-top:12px;
    border-radius:5px;
">
<b>Conclusion:</b><br>
The experimental RAG knowledge base is safe and methodologically valid. 
It contains only active legal articles and non-leaking FAQ indexing records, while evaluation FAQ records and repealed legal articles are excluded from retrieval indexing.
</div>

</div>
"""))


<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:16px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:16px;
    margin:14px 0;
    border-radius:6px;
">

<h3 style="color:#1F4E78; margin-top:0;">
Interpretation — Safe Unified RAG Knowledge Base
</h3>

<p>
توضح هذه المرحلة بناء قاعدة معرفة موحدة وآمنة لاستخدامها في نظام RAG. 
تم إنشاء قاعدة المعرفة التجريبية باستخدام المواد القانونية الفعالة فقط، بالإضافة إلى سجلات FAQ المخصصة للفهرسة، مع استبعاد سجلات FAQ المحجوزة للتقييم.
</p>

<p>
بلغ حجم قاعدة المعرفة التجريبية <b>489</b> سجلًا، موزعة إلى <b>211</b> مادة قانونية فعالة و <b>278</b> سجل FAQ مخصص للفهرسة. 
وهذا يؤكد أن قاعدة التجارب تحتوي فقط على المصادر المسموح استخدامها أثناء تقييم الاسترجاع.
</p>

<p>
كما تم إنشاء قاعدة معرفة إنتاجية كاملة تحتوي على <b>613</b> سجلًا، وتتكون من <b>211</b> مادة قانونية فعالة و <b>402</b> سجل FAQ بعد الفلترة الصارمة. 
وتُستخدم هذه النسخة لاحقًا بعد انتهاء التقييم، لأنها تحتوي على جميع سجلات FAQ النظيفة وليس فقط مجموعة الفهرسة التجريبية.
</p>

<p>
أظهرت فحوصات السلامة أن عدد المواد الملغاة داخل قاعدة المعرفة التجريبية وقاعدة الإنتاج يساوي <b>0</b>. 
وهذا يؤكد أن المواد الملغاة محفوظة فقط في أرشيف مستقل للمراجعة ولا تدخل في الفهرسة أو الاسترجاع.
</p>

<p>
كما أظهر فحص التسريب أن عدد سجلات FAQ الخاصة بالتقييم داخل قاعدة المعرفة التجريبية يساوي <b>0</b>. 
بالإضافة إلى ذلك، تم سابقًا حذف <b>2</b> من أسئلة FAQ المتشابهة جدًا مع سجلات الفهرسة في Stage 06.5، مما يقلل خطر التسريب النصي أو المعنوي.
</p>

<div style="
    background-color:#E2F0D9;
    border-right:4px solid #375623;
    padding:12px;
    margin-top:12px;
    border-radius:5px;
">
<b>Conclusion:</b><br>
The experimental RAG knowledge base is safe and methodologically valid. 
It contains only active legal articles and non-leaking FAQ indexing records, while evaluation FAQ records and repealed legal articles are excluded from retrieval indexing.
</div>

</div>


### Interpretation — Safe Unified RAG Knowledge Base

The unified knowledge base is designed to balance legal coverage, retrieval safety, and evaluation validity.

The experimental version includes only active legal articles and FAQ indexing records. This ensures that the RAG system retrieves from valid legal sources and does not see the held-out FAQ evaluation questions. The production version includes all cleaned FAQ records and can be used after the evaluation phase.

This separation supports fair experimentation and reduces the likelihood of hallucination caused by retrieving invalid or leaked content.


In [18]:
# =========================================================
# Stage 08.1 - Quick Checkpoint: Active Legal Articles in Experimental KB
# التحقق السريع من أن قاعدة التجارب تحتوي مواد قانونية فعالة فقط
# =========================================================

from IPython.display import display, Markdown
import pandas as pd
import numpy as np
import re

# ---------------------------------------------------------
# 1) Validate required object
# ---------------------------------------------------------

assert "df_kb_experiment" in globals(), "df_kb_experiment غير موجود. شغّل Stage 08 أولًا."

required_cols = ["source_type", "article_status", "rag_usage", "document_unit_id"]
missing_cols = [c for c in required_cols if c not in df_kb_experiment.columns]

assert not missing_cols, f"الأعمدة التالية غير موجودة في df_kb_experiment: {missing_cols}"

# ---------------------------------------------------------
# 2) Helper functions
# ---------------------------------------------------------

def short_text(text, max_len=120):
    text = "" if text is None else str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text[:max_len] + ("..." if len(text) > max_len else "")


def hide_index_safe(styler):
    try:
        return styler.hide(axis="index")
    except Exception:
        try:
            return styler.hide_index()
        except Exception:
            return styler


def apply_style_compatible(styler, func, subset):
    if hasattr(styler, "map"):
        return styler.map(func, subset=subset)
    return styler.applymap(func, subset=subset)


def color_status(value):
    value = str(value)

    if value == "PASS":
        return "background-color:#E2F0D9; color:#375623; font-weight:bold;"

    if value == "FAIL":
        return "background-color:#FCE4D6; color:#9C0006; font-weight:bold;"

    if value == "WARN":
        return "background-color:#FFF2CC; color:#7F6000; font-weight:bold;"

    return ""


def professional_table(df, caption="", width="100%", font_size="12px"):
    styler = (
        df.style
        .set_caption(caption)
        .set_table_styles([
            {
                "selector": "caption",
                "props": [
                    ("caption-side", "top"),
                    ("font-size", "17px"),
                    ("font-weight", "bold"),
                    ("color", "#1F4E78"),
                    ("text-align", "center"),
                    ("padding", "10px")
                ]
            },
            {
                "selector": "th",
                "props": [
                    ("background-color", "#1F4E78"),
                    ("color", "white"),
                    ("font-weight", "bold"),
                    ("text-align", "center"),
                    ("border", "1px solid #D9E2F3"),
                    ("padding", "8px"),
                    ("white-space", "normal")
                ]
            },
            {
                "selector": "td",
                "props": [
                    ("text-align", "center"),
                    ("border", "1px solid #D9E2F3"),
                    ("padding", "7px"),
                    ("vertical-align", "middle"),
                    ("white-space", "normal")
                ]
            },
            {
                "selector": "tbody tr:nth-child(even)",
                "props": [("background-color", "#F8FBFD")]
            },
            {
                "selector": "tbody tr:nth-child(odd)",
                "props": [("background-color", "#FFFFFF")]
            },
            {
                "selector": "table",
                "props": [
                    ("border-collapse", "collapse"),
                    ("width", width),
                    ("font-family", "Arial, Tahoma, sans-serif"),
                    ("font-size", font_size),
                    ("table-layout", "fixed"),
                    ("margin", "12px 0")
                ]
            }
        ])
    )

    return hide_index_safe(styler)

# ---------------------------------------------------------
# 3) Filter legal records inside experimental KB
# ---------------------------------------------------------

legal_mask = df_kb_experiment["source_type"].astype(str).eq("legal_article")
legal_kb = df_kb_experiment[legal_mask].copy().reset_index(drop=True)

legal_records_count = len(legal_kb)
active_legal_count = int(legal_kb["article_status"].astype(str).eq("active").sum())
repealed_legal_count = int(legal_kb["article_status"].astype(str).eq("repealed").sum())

status_result = "PASS" if legal_records_count > 0 and repealed_legal_count == 0 else "FAIL"

# ---------------------------------------------------------
# 4) Summary table
# ---------------------------------------------------------

checkpoint_summary = pd.DataFrame([
    {
        "Check": "Legal records inside experimental KB",
        "Value": legal_records_count,
        "Expected": "> 0",
        "Status": "PASS" if legal_records_count > 0 else "FAIL",
        "Interpretation": "The experimental KB contains legal article records."
    },
    {
        "Check": "Active legal articles inside experimental KB",
        "Value": active_legal_count,
        "Expected": legal_records_count,
        "Status": "PASS" if active_legal_count == legal_records_count else "FAIL",
        "Interpretation": "All legal articles inside the experimental KB should be active."
    },
    {
        "Check": "Repealed legal articles inside experimental KB",
        "Value": repealed_legal_count,
        "Expected": 0,
        "Status": "PASS" if repealed_legal_count == 0 else "FAIL",
        "Interpretation": "Repealed articles must not be indexed in the experimental KB."
    }
])

summary_style = professional_table(
    checkpoint_summary,
    caption="Table 8.6 — Quick Checkpoint: Legal Articles inside Experimental KB",
    width="100%",
    font_size="12px"
)

summary_style = apply_style_compatible(
    summary_style,
    color_status,
    subset=["Status"]
)

display(summary_style)

# ---------------------------------------------------------
# 5) Article status distribution
# ---------------------------------------------------------

status_distribution = (
    legal_kb["article_status"]
    .astype(str)
    .value_counts()
    .reset_index()
)

status_distribution.columns = ["Article Status", "Count"]

display(
    professional_table(
        status_distribution,
        caption="Table 8.7 — Legal Article Status Distribution in Experimental KB",
        width="50%",
        font_size="12px"
    )
)

# ---------------------------------------------------------
# 6) Preview legal records
# ---------------------------------------------------------

preview_cols = [
    "source_type",
    "article_number_label",
    "article_number_int",
    "article_status",
    "rag_usage",
    "article_title",
    "source_name",
    "document_unit_id"
]

preview_cols = [c for c in preview_cols if c in legal_kb.columns]

legal_preview = legal_kb[preview_cols].head(10).copy()

for col in ["article_title", "source_name"]:
    if col in legal_preview.columns:
        legal_preview[col] = legal_preview[col].apply(lambda x: short_text(x, 100))

preview_style = professional_table(
    legal_preview,
    caption="Table 8.8 — Preview of Active Legal Articles in Experimental KB",
    width="100%",
    font_size="11.5px"
)

rtl_cols = [c for c in ["article_title", "source_name"] if c in legal_preview.columns]

if rtl_cols:
    preview_style = preview_style.set_properties(
        subset=rtl_cols,
        **{
            "text-align": "right",
            "direction": "rtl",
            "line-height": "1.7"
        }
    )

if "document_unit_id" in legal_preview.columns:
    preview_style = preview_style.set_properties(
        subset=["document_unit_id"],
        **{
            "text-align": "left",
            "direction": "ltr",
            "font-size": "10px"
        }
    )

display(preview_style)

# ---------------------------------------------------------
# 7) Final interpretation
# ---------------------------------------------------------

display(Markdown(f"""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:{'#E2F0D9' if status_result == 'PASS' else '#FCE4D6'};
    border-right:5px solid {'#375623' if status_result == 'PASS' else '#9C0006'};
    padding:14px;
    margin:12px 0;
    border-radius:6px;
">

<b>Stage 08.1 checkpoint completed.</b><br>

Legal records inside experimental KB: <b>{legal_records_count}</b><br>
Active legal records: <b>{active_legal_count}</b><br>
Repealed legal records: <b>{repealed_legal_count}</b><br><br>

<b>Status:</b> {status_result}

</div>
"""))

print("Stage 08.1 checkpoint completed.")
print("Legal records inside experimental KB:", legal_records_count)
print("Active legal records:", active_legal_count)
print("Repealed legal records:", repealed_legal_count)
print("Status:", status_result)

Check,Value,Expected,Status,Interpretation
Legal records inside experimental KB,211,> 0,PASS,The experimental KB contains legal article records.
Active legal articles inside experimental KB,211,211,PASS,All legal articles inside the experimental KB should be active.
Repealed legal articles inside experimental KB,0,0,PASS,Repealed articles must not be indexed in the experimental KB.


Article Status,Count
active,211


source_type,article_number_label,article_number_int,article_status,rag_usage,article_title,source_name,document_unit_id
legal_article,1,1,active,indexing_allowed_active_law,المادة الأولى:,نظام العمل السعودي,d28fed1c707fb8a6b354171cec6a310c1164782775f0d91e9d13d8b8198af462
legal_article,2,2,active,indexing_allowed_active_law,المادة الثانية :,نظام العمل السعودي,4f2961046b5fffdc637e2b7de4dfb1fd862ccc7b9f3069349681b04ff17df638
legal_article,3,3,active,indexing_allowed_active_law,المادة الثالثة:,نظام العمل السعودي,82f3c9791caa01e20aff6f25a0e169fbd07f8aa138625fe674d6df6824a31fda
legal_article,4,4,active,indexing_allowed_active_law,المادة الرابعة:,نظام العمل السعودي,3457807712bce4c43b55e497232052e55933f7fde19df543c46153e86ca95090
legal_article,5,5,active,indexing_allowed_active_law,المادة الخامسة:,نظام العمل السعودي,a03900ffe9b536bc7774b84e1eb41826aacf458ed3920b36e463e08ac8bde4cd
legal_article,6,6,active,indexing_allowed_active_law,المادة السادسة:,نظام العمل السعودي,1157999dfa071a7320606edb9b747429aa71cb7707dcaf5be88784d6c4e2f7ca
legal_article,7,7,active,indexing_allowed_active_law,المادة السابعة :,نظام العمل السعودي,c91b6c3872f87c320cac960229321ad90710d58e53361f20475d343cc1d88d2d
legal_article,8,8,active,indexing_allowed_active_law,المادة الثامنة:,نظام العمل السعودي,01eaca9a4fb7e0f5416045832b295fd64085a920f19266ff5bbc0236deddf41d
legal_article,9,9,active,indexing_allowed_active_law,المادة التاسعة:,نظام العمل السعودي,a0d5a1431be7e4fa5a4411f0ff8222e86afaf8a6e35b994667aae08b4cdfe963
legal_article,10,10,active,indexing_allowed_active_law,المادة العاشرة:,نظام العمل السعودي,518e66de899a12d4203d77f4bdd3cdac022db77736feac05d635149e954e8dca



<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#E2F0D9;
    border-right:5px solid #375623;
    padding:14px;
    margin:12px 0;
    border-radius:6px;
">

<b>Stage 08.1 checkpoint completed.</b><br>

Legal records inside experimental KB: <b>211</b><br>
Active legal records: <b>211</b><br>
Repealed legal records: <b>0</b><br><br>

<b>Status:</b> PASS

</div>


Stage 08.1 checkpoint completed.
Legal records inside experimental KB: 211
Active legal records: 211
Repealed legal records: 0
Status: PASS


In [19]:
from IPython.display import display, Markdown

display(Markdown("""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:16px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:16px;
    margin:14px 0;
    border-radius:6px;
">

<h3 style="color:#1F4E78; margin-top:0;">
Interpretation — Legal Article Safety Check in Experimental KB
</h3>

<p>
توضح هذه المرحلة فحصًا سريعًا للتأكد من أن قاعدة المعرفة التجريبية تحتوي فقط على المواد القانونية الفعالة المسموح باستخدامها في نظام RAG.
</p>

<p>
أظهرت النتائج أن عدد السجلات القانونية داخل قاعدة المعرفة التجريبية بلغ <b>211</b> مادة قانونية، وجميعها مصنفة على أنها <b>active</b>. كما أن عدد المواد الملغاة داخل قاعدة التجارب بلغ <b>0</b>.
</p>

<p>
هذا يؤكد أن المواد الملغاة لم تدخل في الفهرسة أو الاسترجاع، وإنما تم حفظها فقط في أرشيف مستقل لأغراض التوثيق والمراجعة.
</p>

<div style="
    background-color:#E2F0D9;
    border-right:4px solid #375623;
    padding:12px;
    margin-top:12px;
    border-radius:5px;
">
<b>Conclusion:</b><br>
The experimental RAG knowledge base passed the legal article safety check. 
It contains only active Saudi Labor Law articles and excludes repealed legal provisions from retrieval indexing.
</div>

</div>
"""))


<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:16px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:16px;
    margin:14px 0;
    border-radius:6px;
">

<h3 style="color:#1F4E78; margin-top:0;">
Interpretation — Legal Article Safety Check in Experimental KB
</h3>

<p>
توضح هذه المرحلة فحصًا سريعًا للتأكد من أن قاعدة المعرفة التجريبية تحتوي فقط على المواد القانونية الفعالة المسموح باستخدامها في نظام RAG.
</p>

<p>
أظهرت النتائج أن عدد السجلات القانونية داخل قاعدة المعرفة التجريبية بلغ <b>211</b> مادة قانونية، وجميعها مصنفة على أنها <b>active</b>. كما أن عدد المواد الملغاة داخل قاعدة التجارب بلغ <b>0</b>.
</p>

<p>
هذا يؤكد أن المواد الملغاة لم تدخل في الفهرسة أو الاسترجاع، وإنما تم حفظها فقط في أرشيف مستقل لأغراض التوثيق والمراجعة.
</p>

<div style="
    background-color:#E2F0D9;
    border-right:4px solid #375623;
    padding:12px;
    margin-top:12px;
    border-radius:5px;
">
<b>Conclusion:</b><br>
The experimental RAG knowledge base passed the legal article safety check. 
It contains only active Saudi Labor Law articles and excludes repealed legal provisions from retrieval indexing.
</div>

</div>


In [20]:
# =========================================================
# Stage 08.2 - Hard Assertion Checkpoint
# تأكيد برمجي نهائي أن قاعدة التجارب تحتوي مواد فعالة فقط
# =========================================================

assert "df_kb_experiment" in globals(), "df_kb_experiment غير موجود. شغّل Stage 08 أولًا."

required_cols = ["source_type", "article_status"]
missing_cols = [c for c in required_cols if c not in df_kb_experiment.columns]

assert not missing_cols, f"Missing required columns in df_kb_experiment: {missing_cols}"

legal_mask = df_kb_experiment["source_type"].astype(str).eq("legal_article")
legal_status_values = set(
    df_kb_experiment.loc[legal_mask, "article_status"]
    .dropna()
    .astype(str)
    .unique()
)

repealed_count = int(
    df_kb_experiment.loc[legal_mask, "article_status"]
    .astype(str)
    .eq("repealed")
    .sum()
)

assert repealed_count == 0, (
    "Error: Repealed legal articles found inside experimental KB."
)

assert legal_status_values == {"active"}, (
    f"Error: Experimental legal KB must contain active legal articles only. "
    f"Found statuses: {legal_status_values}"
)

print("Checkpoint passed: Experimental KB contains active legal articles only.")
print("Legal article records:", int(legal_mask.sum()))
print("Legal article statuses:", legal_status_values)
print("Repealed legal records:", repealed_count)

Checkpoint passed: Experimental KB contains active legal articles only.
Legal article records: 211
Legal article statuses: {'active'}
Repealed legal records: 0


## Stage 09 — Build the Initial Evaluation Dataset

This stage builds a leakage-safe evaluation dataset from the held-out FAQ records.

At this point, the evaluation set contains FAQ questions only. Manual legal-article questions and out-of-scope questions are added later in Stage 09.5. This staged design makes the evaluation dataset easier to inspect, extend, and validate.

This stage also removes duplicate evaluation questions before saving the dataset. Duplicate questions can bias retrieval metrics because the same query would be counted more than once. Removed duplicates are saved as an audit file so the preprocessing decision remains transparent and reproducible.

The final evaluation dataset is intended to test three behaviours:

1. Retrieving relevant FAQ records.
2. Retrieving the correct legal article for legal questions.
3. Rejecting or redirecting questions outside the Saudi Labor Law scope.


In [21]:
# =========================================================
# Stage 09 - Build Evaluation Dataset (FAQ Held-Out Only)
# Leakage-safe FAQ evaluation set after Stage 06.5
# =========================================================

from IPython.display import display, Markdown
import pandas as pd
import numpy as np
import re
from pathlib import Path

# ---------------------------------------------------------
# 1) Directory safety and required objects
# ---------------------------------------------------------

if "FINAL_DIR" not in globals():
    FINAL_DIR = Path("03_final")

if "REPORTS_DIR" not in globals():
    REPORTS_DIR = Path("05_reports")

FINAL_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

required_objects = [
    "df_faq_evaluation",
    "df_faq_indexing",
    "df_kb_experiment",
    "df_articles_active"
]

missing_objects = [
    obj for obj in required_objects
    if obj not in globals() or not isinstance(globals()[obj], pd.DataFrame)
]

if missing_objects:
    raise NameError(
        "Missing required objects before Stage 09: "
        + ", ".join(missing_objects)
        + ". Run Stage 06, Stage 06.5, and Stage 08 first."
    )

assert len(df_faq_evaluation) > 0, "df_faq_evaluation is empty. Run Stage 06.5 first."
assert len(df_faq_indexing) > 0, "df_faq_indexing is empty. Run Stage 06 first."
assert len(df_kb_experiment) > 0, "df_kb_experiment is empty. Run Stage 08 first."
assert len(df_articles_active) > 0, "df_articles_active is empty. Run legal cleaning stages first."

# ---------------------------------------------------------
# 2) Helper functions
# ---------------------------------------------------------

def clean_text(value):
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass
    return re.sub(r"\s+", " ", str(value)).strip()


def fallback_normalize_arabic_for_search(text):
    text = clean_text(text)
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ؤ", "و", text)
    text = re.sub(r"ئ", "ي", text)
    text = re.sub(r"ة", "ه", text)
    text = re.sub(r"[\u064B-\u065F\u0670]", "", text)
    text = re.sub(r"[^\u0600-\u06FFa-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


if "normalize_arabic_for_search" not in globals():
    normalize_arabic_for_search = fallback_normalize_arabic_for_search


def normalize_eval_question_for_dedup(text):
    text = clean_text(text)
    text = normalize_arabic_for_search(text)
    return text


def get_safe_col(df, col, default=""):
    if col in df.columns:
        return df[col]
    return pd.Series([default] * len(df), index=df.index)


def short_display_text(value, max_len=150):
    value = clean_text(value)
    return value[:max_len] + ("..." if len(value) > max_len else "")


def hide_index_safe(styler):
    try:
        return styler.hide(axis="index")
    except Exception:
        try:
            return styler.hide_index()
        except Exception:
            return styler


def apply_style_compatible(styler, func, subset):
    if hasattr(styler, "map"):
        return styler.map(func, subset=subset)
    return styler.applymap(func, subset=subset)


def highlight_status(value):
    value = str(value)

    if value in ["PASS", "OK"]:
        return "background-color:#E2F0D9; color:#375623; font-weight:bold;"
    if value == "WARN":
        return "background-color:#FFF2CC; color:#7F6000; font-weight:bold;"
    if value == "FAIL":
        return "background-color:#FCE4D6; color:#9C0006; font-weight:bold;"
    return ""


def status_label(condition):
    return "PASS" if condition else "FAIL"


def professional_table(df, caption="", width="100%", font_size="12px"):
    styler = (
        df.style
        .set_caption(caption)
        .set_table_styles([
            {
                "selector": "caption",
                "props": [
                    ("caption-side", "top"),
                    ("font-size", "17px"),
                    ("font-weight", "bold"),
                    ("color", "#1F4E78"),
                    ("text-align", "center"),
                    ("padding", "10px")
                ]
            },
            {
                "selector": "th",
                "props": [
                    ("background-color", "#1F4E78"),
                    ("color", "white"),
                    ("font-weight", "bold"),
                    ("text-align", "center"),
                    ("border", "1px solid #D9E2F3"),
                    ("padding", "8px"),
                    ("white-space", "normal")
                ]
            },
            {
                "selector": "td",
                "props": [
                    ("text-align", "center"),
                    ("border", "1px solid #D9E2F3"),
                    ("padding", "7px"),
                    ("vertical-align", "middle"),
                    ("white-space", "normal")
                ]
            },
            {
                "selector": "tbody tr:nth-child(even)",
                "props": [("background-color", "#F8FBFD")]
            },
            {
                "selector": "tbody tr:nth-child(odd)",
                "props": [("background-color", "#FFFFFF")]
            },
            {
                "selector": "table",
                "props": [
                    ("border-collapse", "collapse"),
                    ("width", width),
                    ("font-family", "Arial, Tahoma, sans-serif"),
                    ("font-size", font_size),
                    ("table-layout", "fixed"),
                    ("margin", "12px 0")
                ]
            }
        ])
    )
    return hide_index_safe(styler)

# ---------------------------------------------------------
# 3) Stage explanation
# ---------------------------------------------------------

display(Markdown("""
<div style="
    text-align:left;
    line-height:1.8;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-left:5px solid #1F4E78;
    padding:14px;
    margin:12px 0;
    border-radius:6px;
">

<h3 style="color:#1F4E78; margin-top:0;">
Stage 09 — Evaluation Dataset Construction
</h3>

<p>
This stage constructs the initial FAQ-only evaluation dataset using held-out FAQ records after exact leakage removal and content-level near-duplicate cleaning.
The experimental RAG knowledge base does not contain these evaluation records.
</p>

<p>
Manual legal questions and out-of-scope questions are intentionally added later in Stage 09.5.
</p>

</div>
"""))

# ---------------------------------------------------------
# 4) Prepare FAQ held-out evaluation set
# ---------------------------------------------------------

faq_eval = df_faq_evaluation.copy().reset_index(drop=True)
faq_eval_before_domain_guard = len(faq_eval)

# ---------------------------------------------------------
# 5) Defensive QA-only domain audit
# ---------------------------------------------------------
# This audit checks question + answer only.
# It does NOT check metadata fields to avoid over-filtering.

_AR_DIGITS_STAGE09 = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")

def _stage09_norm(text):
    return normalize_arabic_for_search(text).translate(_AR_DIGITS_STAGE09)


STAGE09_QA_HARD_EXCLUDE_TERMS = [
    "رؤية 2030",
    "رؤية المملكة",
    "رؤية السعودية",
    "مؤتمر",
    "المؤتمر",
    "معرض",
    "المعرض",
    "منتدى",
    "ملتقى",
    "ندوة",
    "احتفال",
    "تكريم",
    "جائزة",
    "الجائزة",
    "تطوع",
    "التطوع",
    "فعالية",
    "الفعاليات",
    "يوم عالمي",
    "يوم عربي",
    "بيان صحفي",
    "خبر صحفي",
    "وسائل الاعلام",
    "الاعلام والاتصال",
    "حق الوصول الى المعلومات",
    "الحصول على المعلومات",
    "الشفافية",
    "حالة الطلب",
    "ارجاع الطلب",
    "ارسال الطلب",
    "صيغة الملفات",
    "حجم الملفات",
    "هل يمكن تعديل الطلب بعد ارساله",
]

_STAGE09_EXCLUDE_NORM = [_stage09_norm(k) for k in STAGE09_QA_HARD_EXCLUDE_TERMS]

def _stage09_is_obvious_non_labor_qa(row):
    qa_text = _stage09_norm(f"{row.get('question', '')} {row.get('answer', '')}")
    return any(term in qa_text for term in _STAGE09_EXCLUDE_NORM)


domain_removed_mask = faq_eval.apply(_stage09_is_obvious_non_labor_qa, axis=1)

faq_eval_domain_removed = faq_eval[domain_removed_mask].copy()
faq_eval_domain_removed_count = int(domain_removed_mask.sum())

if faq_eval_domain_removed_count > 0:
    faq_eval_domain_removed["removal_reason"] = (
        "Removed by Stage 09 QA-only defensive audit. "
        "The question/answer contains obvious non-labor or generic service terms."
    )

faq_eval = faq_eval[~domain_removed_mask].reset_index(drop=True)
faq_eval_after_domain_guard = len(faq_eval)

assert faq_eval_after_domain_guard > 0, (
    "FAQ evaluation set became empty after Stage 09 QA-only defensive audit. "
    "Review Stage 05.5 and Stage 06.5 outputs."
)

# ---------------------------------------------------------
# 6) Save removed audit records
# ---------------------------------------------------------

faq_eval_domain_removed_csv_path = FINAL_DIR / "rag_evaluation_faq_domain_removed.csv"
faq_eval_domain_removed_xlsx_path = FINAL_DIR / "rag_evaluation_faq_domain_removed.xlsx"

faq_eval_domain_removed.to_csv(
    faq_eval_domain_removed_csv_path,
    index=False,
    encoding="utf-8-sig"
)

faq_eval_domain_removed.to_excel(
    faq_eval_domain_removed_xlsx_path,
    index=False
)

# ---------------------------------------------------------
# 7) Build FAQ evaluation rows
# ---------------------------------------------------------

df_eval_faq = pd.DataFrame({
    "question": get_safe_col(faq_eval, "question", ""),
    "expected_answer": get_safe_col(faq_eval, "answer", ""),
    "beneficiaries": get_safe_col(faq_eval, "beneficiaries", ""),
    "sector": get_safe_col(faq_eval, "sector", ""),
    "subsite": get_safe_col(faq_eval, "subsite", ""),
    "legal_category": get_safe_col(faq_eval, "legal_category", ""),
    "source_name": get_safe_col(faq_eval, "source_name", "HRSD FAQ"),
    "source_url": get_safe_col(faq_eval, "page_url", get_safe_col(faq_eval, "source_url", "")),
    "eval_type": "faq_retrieval",
    "gold_source_type": "faq",
    "gold_document_unit_id": get_safe_col(faq_eval, "document_unit_id", ""),
    "gold_document_unit_ids": get_safe_col(faq_eval, "document_unit_id", ""),
    "gold_article_number": "",
    "gold_article_numbers": "",
    "gold_article_title": "",
    "is_out_of_scope": False,
    "notes": "Held-out FAQ question. Not indexed in experimental KB.",
})

# ---------------------------------------------------------
# 8) Helper function for legal gold articles
# Used later in Stage 09.5
# ---------------------------------------------------------

def get_legal_gold(article_numbers):
    """
    Returns gold metadata for one or multiple active legal articles.
    """
    nums = [int(n) for n in article_numbers]

    matched = (
        df_articles_active[
            df_articles_active["article_number_int"].astype(int).isin(nums)
        ]
        .sort_values("article_number_int")
        .copy()
    )

    if matched.empty:
        return {
            "expected_answer": "",
            "gold_document_unit_id": "",
            "gold_document_unit_ids": "",
            "gold_article_number": "",
            "gold_article_numbers": ",".join(map(str, nums)),
            "gold_article_title": "",
            "source_url": "",
            "notes_extra": "WARNING: gold article was not found in active legal articles."
        }

    return {
        "expected_answer": "\n\n".join(matched["article_text"].astype(str).tolist()),
        "gold_document_unit_id": matched["document_unit_id"].iloc[0],
        "gold_document_unit_ids": "|".join(matched["document_unit_id"].astype(str).tolist()),
        "gold_article_number": str(int(matched["article_number_int"].iloc[0])),
        "gold_article_numbers": ",".join(
            matched["article_number_int"].astype(int).astype(str).tolist()
        ),
        "gold_article_title": " | ".join(matched["article_title"].astype(str).tolist()),
        "source_url": matched["source_url"].iloc[0],
        "notes_extra": ""
    }

# ---------------------------------------------------------
# 9) Manual legal and out-of-scope placeholders
# ---------------------------------------------------------

manual_specs = []
out_of_scope_questions = []
manual_eval_questions = []

df_eval_manual = pd.DataFrame(manual_eval_questions)

# ---------------------------------------------------------
# 10) Build unified evaluation dataset
# ---------------------------------------------------------

if len(df_eval_manual) > 0:
    df_eval = pd.concat([df_eval_faq, df_eval_manual], ignore_index=True)
else:
    df_eval = df_eval_faq.copy()

df_eval = df_eval.reset_index(drop=True)

# ---------------------------------------------------------
# 11) Remove duplicate questions
# ---------------------------------------------------------

df_eval["question_normalized_for_dedup"] = (
    df_eval["question"]
    .fillna("")
    .astype(str)
    .apply(normalize_eval_question_for_dedup)
)

duplicate_mask = df_eval.duplicated(
    subset=["question_normalized_for_dedup"],
    keep="first"
)

df_eval_duplicates_removed = df_eval[duplicate_mask].copy()
duplicates_removed_count = int(duplicate_mask.sum())

df_eval = (
    df_eval[~duplicate_mask]
    .drop(columns=["question_normalized_for_dedup"], errors="ignore")
    .reset_index(drop=True)
)

df_eval_duplicates_removed = df_eval_duplicates_removed.drop(
    columns=["question_normalized_for_dedup"],
    errors="ignore"
)

if "eval_id" in df_eval.columns:
    df_eval = df_eval.drop(columns=["eval_id"])

df_eval.insert(0, "eval_id", range(1, len(df_eval) + 1))

# ---------------------------------------------------------
# 12) Quality checks
# ---------------------------------------------------------

faq_eval_count = int((df_eval["eval_type"] == "faq_retrieval").sum())
legal_eval_count = int((df_eval["eval_type"] == "legal_article_retrieval").sum())
oos_eval_count = int((df_eval["eval_type"] == "out_of_scope").sum())

missing_legal_gold = df_eval[
    (df_eval["eval_type"] == "legal_article_retrieval") &
    (df_eval["gold_document_unit_ids"].astype(str).str.strip() == "")
]

faq_leakage_eval = int(
    len(
        set(df_eval_faq["gold_document_unit_id"].astype(str))
        &
        set(
            df_kb_experiment[
                df_kb_experiment["source_type"].astype(str) == "faq"
            ]["document_unit_id"].astype(str)
        )
    )
)

duplicate_eval_questions = int(
    df_eval["question"]
    .fillna("")
    .astype(str)
    .apply(normalize_eval_question_for_dedup)
    .duplicated()
    .sum()
)

allowed_eval_types = {
    "faq_retrieval",
    "legal_article_retrieval",
    "out_of_scope"
}

unknown_eval_types = sorted(
    set(df_eval["eval_type"].dropna().astype(str).unique()) - allowed_eval_types
)

assert len(missing_legal_gold) == 0, (
    "Error: Some legal evaluation questions have no gold legal document ID."
)

assert faq_leakage_eval == 0, (
    "Error: FAQ evaluation records leaked into experimental KB."
)

assert duplicate_eval_questions == 0, (
    "Error: duplicate evaluation questions remain after deduplication."
)

assert len(unknown_eval_types) == 0, (
    f"Error: Unknown eval_type values found: {unknown_eval_types}"
)

assert len(df_eval) > 0, "Evaluation dataset is empty."

# ---------------------------------------------------------
# 13) Save outputs
# ---------------------------------------------------------

eval_csv_path = FINAL_DIR / "rag_evaluation_dataset.csv"
eval_xlsx_path = FINAL_DIR / "rag_evaluation_dataset.xlsx"

duplicates_removed_csv_path = FINAL_DIR / "rag_evaluation_duplicates_removed.csv"
duplicates_removed_xlsx_path = FINAL_DIR / "rag_evaluation_duplicates_removed.xlsx"

df_eval.to_csv(
    eval_csv_path,
    index=False,
    encoding="utf-8-sig"
)

df_eval.to_excel(
    eval_xlsx_path,
    index=False
)

df_eval_duplicates_removed.to_csv(
    duplicates_removed_csv_path,
    index=False,
    encoding="utf-8-sig"
)

df_eval_duplicates_removed.to_excel(
    duplicates_removed_xlsx_path,
    index=False
)

# ---------------------------------------------------------
# 14) Professional summary tables
# ---------------------------------------------------------

evaluation_summary_df = pd.DataFrame([
    {
        "Component": "Held-out FAQ Evaluation Set",
        "Count": faq_eval_count,
        "Status": "PASS" if faq_eval_count > 0 else "FAIL",
        "Interpretation": "FAQ records reserved for evaluation and excluded from the experimental knowledge base."
    },
    {
        "Component": "Manual Legal Evaluation Questions",
        "Count": legal_eval_count,
        "Status": "OK",
        "Interpretation": "Not added in this stage. Legal questions are added later in Stage 09.5."
    },
    {
        "Component": "Out-of-Scope Evaluation Questions",
        "Count": oos_eval_count,
        "Status": "OK",
        "Interpretation": "Not added in this stage. Out-of-scope tests are added later in Stage 09.5."
    },
    {
        "Component": "FAQ Records Removed by Stage 09 QA Audit",
        "Count": faq_eval_domain_removed_count,
        "Status": "PASS",
        "Interpretation": "Only obvious non-labor records in question/answer are removed here. Metadata is not used."
    },
    {
        "Component": "Duplicate Questions Removed",
        "Count": duplicates_removed_count,
        "Status": "PASS",
        "Interpretation": "Duplicate evaluation questions were removed before saving."
    },
    {
        "Component": "Total Evaluation Questions Saved",
        "Count": len(df_eval),
        "Status": "PASS" if len(df_eval) > 0 else "FAIL",
        "Interpretation": "Initial FAQ-only evaluation dataset after duplicate removal."
    }
])

summary_style = professional_table(
    evaluation_summary_df,
    caption="Table 9.1 — Evaluation Dataset Summary",
    width="100%",
    font_size="12px"
)

summary_style = apply_style_compatible(
    summary_style,
    highlight_status,
    subset=["Status"]
)

display(summary_style)

evaluation_safety_df = pd.DataFrame([
    {
        "Safety Check": "FAQ evaluation leakage inside experimental KB",
        "Value": faq_leakage_eval,
        "Expected": 0,
        "Status": status_label(faq_leakage_eval == 0),
        "Interpretation": "Held-out FAQ records must not appear inside the experimental RAG knowledge base."
    },
    {
        "Safety Check": "FAQ removed by QA-only domain audit",
        "Value": faq_eval_domain_removed_count,
        "Expected": ">= 0",
        "Status": "PASS",
        "Interpretation": "Only question/answer text is checked; metadata is not used to avoid over-filtering."
    },
    {
        "Safety Check": "Missing legal gold document IDs",
        "Value": len(missing_legal_gold),
        "Expected": 0,
        "Status": status_label(len(missing_legal_gold) == 0),
        "Interpretation": "No missing legal references are expected because legal questions have not been added yet."
    },
    {
        "Safety Check": "Duplicate evaluation questions after removal",
        "Value": duplicate_eval_questions,
        "Expected": 0,
        "Status": status_label(duplicate_eval_questions == 0),
        "Interpretation": f"Duplicate questions were removed before saving. Removed duplicates: {duplicates_removed_count}."
    },
    {
        "Safety Check": "Unknown evaluation types",
        "Value": len(unknown_eval_types),
        "Expected": 0,
        "Status": status_label(len(unknown_eval_types) == 0),
        "Interpretation": "Only faq_retrieval, legal_article_retrieval, and out_of_scope are allowed."
    }
])

safety_style = professional_table(
    evaluation_safety_df,
    caption="Table 9.2 — Evaluation Dataset Safety and Leakage Checks",
    width="100%",
    font_size="12px"
)

safety_style = apply_style_compatible(
    safety_style,
    highlight_status,
    subset=["Status"]
)

display(safety_style)

# ---------------------------------------------------------
# 15) Evaluation dataset composition
# ---------------------------------------------------------

eval_type_df = (
    df_eval["eval_type"]
    .value_counts()
    .rename_axis("Evaluation Type")
    .reset_index(name="Records")
)

eval_type_df["Percentage"] = (
    eval_type_df["Records"] / eval_type_df["Records"].sum() * 100
).round(2)

display(
    professional_table(
        eval_type_df,
        caption="Table 9.3 — Evaluation Questions by Type",
        width="65%",
        font_size="12px"
    )
)

# ---------------------------------------------------------
# 16) Metadata coverage table
# ---------------------------------------------------------

metadata_coverage_df = pd.DataFrame([
    {
        "Metadata Field": "beneficiaries",
        "Non-empty Records": int(df_eval["beneficiaries"].fillna("").astype(str).str.strip().ne("").sum()),
        "Unique Values": int(df_eval["beneficiaries"].replace("", pd.NA).dropna().nunique()),
        "Interpretation": "Represents the target beneficiary categories in FAQ evaluation questions."
    },
    {
        "Metadata Field": "sector",
        "Non-empty Records": int(df_eval["sector"].fillna("").astype(str).str.strip().ne("").sum()),
        "Unique Values": int(df_eval["sector"].replace("", pd.NA).dropna().nunique()),
        "Interpretation": "Represents the sector classification provided by the FAQ source."
    },
    {
        "Metadata Field": "subsite",
        "Non-empty Records": int(df_eval["subsite"].fillna("").astype(str).str.strip().ne("").sum()),
        "Unique Values": int(df_eval["subsite"].replace("", pd.NA).dropna().nunique()),
        "Interpretation": "Represents the ministry subsite or service area associated with the FAQ."
    },
    {
        "Metadata Field": "legal_category",
        "Non-empty Records": int(df_eval["legal_category"].fillna("").astype(str).str.strip().ne("").sum()),
        "Unique Values": int(df_eval["legal_category"].replace("", pd.NA).dropna().nunique()),
        "Interpretation": "Represents the legal or service category used for filtering and analysis."
    }
])

display(
    professional_table(
        metadata_coverage_df,
        caption="Table 9.4 — FAQ Evaluation Metadata Coverage",
        width="100%",
        font_size="12px"
    )
)

# ---------------------------------------------------------
# 17) Saved files manifest
# ---------------------------------------------------------

saved_eval_files_df = pd.DataFrame([
    {
        "File Type": "Evaluation Dataset - CSV",
        "Path": str(eval_csv_path),
        "Purpose": "Main evaluation dataset used in retrieval evaluation stages."
    },
    {
        "File Type": "Evaluation Dataset - Excel",
        "Path": str(eval_xlsx_path),
        "Purpose": "Readable Excel version of the evaluation dataset."
    },
    {
        "File Type": "Removed Duplicates Audit - CSV",
        "Path": str(duplicates_removed_csv_path),
        "Purpose": "Audit file containing duplicate evaluation questions removed before saving."
    },
    {
        "File Type": "Removed Duplicates Audit - Excel",
        "Path": str(duplicates_removed_xlsx_path),
        "Purpose": "Readable Excel version of the removed duplicates audit file."
    },
    {
        "File Type": "Removed FAQ QA Audit - CSV",
        "Path": str(faq_eval_domain_removed_csv_path),
        "Purpose": "Audit file containing records removed by the Stage 09 question/answer defensive guard."
    },
    {
        "File Type": "Removed FAQ QA Audit - Excel",
        "Path": str(faq_eval_domain_removed_xlsx_path),
        "Purpose": "Readable Excel version of the Stage 09 QA audit file."
    }
])

saved_style = professional_table(
    saved_eval_files_df,
    caption="Table 9.5 — Stage 09 Saved Output Files",
    width="100%",
    font_size="11.5px"
)

saved_style = saved_style.set_properties(
    subset=["Path"],
    **{
        "text-align": "left",
        "direction": "ltr",
        "font-size": "10.5px"
    }
)

display(saved_style)

# ---------------------------------------------------------
# 18) Preview table
# ---------------------------------------------------------

preview_cols = [
    "eval_id",
    "eval_type",
    "question",
    "gold_source_type",
    "gold_document_unit_id",
    "gold_article_numbers",
    "is_out_of_scope",
    "notes"
]

preview_cols = [col for col in preview_cols if col in df_eval.columns]

preview_df = df_eval[preview_cols].head(10).copy()

if "question" in preview_df.columns:
    preview_df["question"] = preview_df["question"].apply(lambda x: short_display_text(x, 160))

preview_style = professional_table(
    preview_df,
    caption="Table 9.6 — First 10 Records from the Evaluation Dataset",
    width="100%",
    font_size="11.5px"
)

if "question" in preview_df.columns:
    preview_style = preview_style.set_properties(
        subset=["question"],
        **{
            "text-align": "right",
            "direction": "rtl",
            "line-height": "1.7"
        }
    )

display(preview_style)

# ---------------------------------------------------------
# 19) Final completion message
# ---------------------------------------------------------

display(Markdown(f"""
<div style="
    text-align:left;
    line-height:1.8;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#E2F0D9;
    border-left:5px solid #375623;
    padding:14px;
    margin:12px 0;
    border-radius:6px;
">

<b>Stage 09 completed successfully.</b><br>
Evaluation dataset records saved: <b>{len(df_eval):,}</b><br>
FAQ evaluation records: <b>{faq_eval_count:,}</b><br>
FAQ records removed by Stage 09 QA audit: <b>{faq_eval_domain_removed_count:,}</b><br>
Duplicates removed before saving: <b>{duplicates_removed_count:,}</b><br>
FAQ leakage into the experimental KB: <b>{faq_leakage_eval:,}</b><br>

</div>
"""))

print("Stage 09 completed successfully.")
print(f"Evaluation dataset records: {len(df_eval):,}")
print(f"FAQ evaluation records: {faq_eval_count:,}")
print(f"FAQ records removed by Stage 09 QA audit: {faq_eval_domain_removed_count:,}")
print(f"Legal article evaluation records: {legal_eval_count:,} - added later in Stage 09.5")
print(f"Out-of-scope evaluation records: {oos_eval_count:,} - added later in Stage 09.5")
print(f"FAQ leakage into experimental KB: {faq_leakage_eval:,}")
print(f"Missing legal gold records: {len(missing_legal_gold):,}")
print(f"Duplicate evaluation questions after removal: {duplicate_eval_questions:,}")
print(f"Duplicate questions removed before saving: {duplicates_removed_count:,}")
print("Saved:", eval_csv_path)
print("Saved:", eval_xlsx_path)


<div style="
    text-align:left;
    line-height:1.8;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-left:5px solid #1F4E78;
    padding:14px;
    margin:12px 0;
    border-radius:6px;
">

<h3 style="color:#1F4E78; margin-top:0;">
Stage 09 — Evaluation Dataset Construction
</h3>

<p>
This stage constructs the initial FAQ-only evaluation dataset using held-out FAQ records after exact leakage removal and content-level near-duplicate cleaning.
The experimental RAG knowledge base does not contain these evaluation records.
</p>

<p>
Manual legal questions and out-of-scope questions are intentionally added later in Stage 09.5.
</p>

</div>


Component,Count,Status,Interpretation
Held-out FAQ Evaluation Set,121,PASS,FAQ records reserved for evaluation and excluded from the experimental knowledge base.
Manual Legal Evaluation Questions,0,OK,Not added in this stage. Legal questions are added later in Stage 09.5.
Out-of-Scope Evaluation Questions,0,OK,Not added in this stage. Out-of-scope tests are added later in Stage 09.5.
FAQ Records Removed by Stage 09 QA Audit,0,PASS,Only obvious non-labor records in question/answer are removed here. Metadata is not used.
Duplicate Questions Removed,1,PASS,Duplicate evaluation questions were removed before saving.
Total Evaluation Questions Saved,121,PASS,Initial FAQ-only evaluation dataset after duplicate removal.


Safety Check,Value,Expected,Status,Interpretation
FAQ evaluation leakage inside experimental KB,0,0,PASS,Held-out FAQ records must not appear inside the experimental RAG knowledge base.
FAQ removed by QA-only domain audit,0,>= 0,PASS,Only question/answer text is checked; metadata is not used to avoid over-filtering.
Missing legal gold document IDs,0,0,PASS,No missing legal references are expected because legal questions have not been added yet.
Duplicate evaluation questions after removal,0,0,PASS,Duplicate questions were removed before saving. Removed duplicates: 1.
Unknown evaluation types,0,0,PASS,"Only faq_retrieval, legal_article_retrieval, and out_of_scope are allowed."


Evaluation Type,Records,Percentage
faq_retrieval,121,100.000000


Metadata Field,Non-empty Records,Unique Values,Interpretation
beneficiaries,121,7,Represents the target beneficiary categories in FAQ evaluation questions.
sector,112,3,Represents the sector classification provided by the FAQ source.
subsite,121,1,Represents the ministry subsite or service area associated with the FAQ.
legal_category,121,10,Represents the legal or service category used for filtering and analysis.


File Type,Path,Purpose
Evaluation Dataset - CSV,saudi_labor_law_voice_agent_project\03_final\rag_evaluation_dataset.csv,Main evaluation dataset used in retrieval evaluation stages.
Evaluation Dataset - Excel,saudi_labor_law_voice_agent_project\03_final\rag_evaluation_dataset.xlsx,Readable Excel version of the evaluation dataset.
Removed Duplicates Audit - CSV,saudi_labor_law_voice_agent_project\03_final\rag_evaluation_duplicates_removed.csv,Audit file containing duplicate evaluation questions removed before saving.
Removed Duplicates Audit - Excel,saudi_labor_law_voice_agent_project\03_final\rag_evaluation_duplicates_removed.xlsx,Readable Excel version of the removed duplicates audit file.
Removed FAQ QA Audit - CSV,saudi_labor_law_voice_agent_project\03_final\rag_evaluation_faq_domain_removed.csv,Audit file containing records removed by the Stage 09 question/answer defensive guard.
Removed FAQ QA Audit - Excel,saudi_labor_law_voice_agent_project\03_final\rag_evaluation_faq_domain_removed.xlsx,Readable Excel version of the Stage 09 QA audit file.


eval_id,eval_type,question,gold_source_type,gold_document_unit_id,gold_article_numbers,is_out_of_scope,notes
1,faq_retrieval,هل يحق صاحب العمل في الاعتراض في عقد العمل الموثق سندًا تنفيذيًا؟,faq,004ea50cd2e3b50e605b6d0f769d2fb418a83490050e46372e0a4ce04446c378,,False,Held-out FAQ question. Not indexed in experimental KB.
2,faq_retrieval,كيف يتم رفع طلب التنفيذ في حال عدم التزام صاحب العمل بسداد الأجر في عقد العمل الموثق سندًا تنفيذيًا؟,faq,001715202ff48d357db0ee5b5dc2f8992b5a1cec7389155a11cad6d876f37fc6,,False,Held-out FAQ question. Not indexed in experimental KB.
3,faq_retrieval,ماهو بند العقد الخاضع للتنفيذ لعقد العمل الموثق سندًا تنفيذيًا؟,faq,1f0a32b43d1d888523da1db089e2b5f6310977c6d4065dff65bf39462c298fd2,,False,Held-out FAQ question. Not indexed in experimental KB.
4,faq_retrieval,ماهي مراحل التطبيق على العقود الموثقة لعقد العمل الموثق سندًا تنفيذيًا؟,faq,c37bc793ca18cedb26c8bc4e19db4b07f676501c117336fea4847f8c92bb7e80,,False,Held-out FAQ question. Not indexed in experimental KB.
5,faq_retrieval,ماهي متطلبات الاستفادة من مبادرة عقد العمل الموثق سندًا تنفيذيًا؟,faq,46001ae4bf6cc21f527a78e9a6a46b5913de3ecff09d71a32feb88f8a6ccaaa9,,False,Held-out FAQ question. Not indexed in experimental KB.
6,faq_retrieval,ما هي الجهة المشرعة لبرنامج حماية الأجور؟,faq,78f881c6ac6d11831761761dbf7216c07b31b6dee228ccdb6e04ad4debb0bb6c,,False,Held-out FAQ question. Not indexed in experimental KB.
7,faq_retrieval,ما هي أهداف برنامج حماية الأجور؟,faq,b0ebcab351b4739f0566a90080daa23d875b475bac36659a0088539697b8a266,,False,Held-out FAQ question. Not indexed in experimental KB.
8,faq_retrieval,ما هو تاريخ بداية العقد لمن أكمل 5 سنوات وكيف تحسب فترة الخدمة السابقة لتسجيل العقد؟,faq,0d597df1b233e5d84c1e7ddfd9353da00659d88fb934af35eaa2b25a57140b2c,,False,Held-out FAQ question. Not indexed in experimental KB.
9,faq_retrieval,كيف يرسل العقد للموظف؟,faq,c18fece192547ecfc865229426cb86860b0eac8f515d288c0f22cbabc1991928,,False,Held-out FAQ question. Not indexed in experimental KB.
10,faq_retrieval,هل تمكن البوابة والانظمة التابعة لها المستخدم الاعتباري (الشركات) من تعديل بياناتها,faq,3bd22efd40a50fd37fe62428fe33778eb4918ea6a0fa8dd2e47cea459f83d393,,False,Held-out FAQ question. Not indexed in experimental KB.



<div style="
    text-align:left;
    line-height:1.8;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#E2F0D9;
    border-left:5px solid #375623;
    padding:14px;
    margin:12px 0;
    border-radius:6px;
">

<b>Stage 09 completed successfully.</b><br>
Evaluation dataset records saved: <b>121</b><br>
FAQ evaluation records: <b>121</b><br>
FAQ records removed by Stage 09 QA audit: <b>0</b><br>
Duplicates removed before saving: <b>1</b><br>
FAQ leakage into the experimental KB: <b>0</b><br>

</div>


Stage 09 completed successfully.
Evaluation dataset records: 121
FAQ evaluation records: 121
FAQ records removed by Stage 09 QA audit: 0
Legal article evaluation records: 0 - added later in Stage 09.5
Out-of-scope evaluation records: 0 - added later in Stage 09.5
FAQ leakage into experimental KB: 0
Missing legal gold records: 0
Duplicate evaluation questions after removal: 0
Duplicate questions removed before saving: 1
Saved: saudi_labor_law_voice_agent_project\03_final\rag_evaluation_dataset.csv
Saved: saudi_labor_law_voice_agent_project\03_final\rag_evaluation_dataset.xlsx


In [22]:
from IPython.display import display, Markdown

display(Markdown("""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:16px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:16px;
    margin:14px 0;
    border-radius:6px;
">

<h3 style="color:#1F4E78; margin-top:0;">
Interpretation — Evaluation Dataset Construction
</h3>

<p>
توضح هذه المرحلة بناء مجموعة التقييم الأولية الخاصة بنظام RAG اعتمادًا على أسئلة FAQ المحجوزة للتقييم فقط. 
تم استخدام سجلات FAQ التي تم فصلها سابقًا عن مجموعة الفهرسة في Stage 06، ثم تنظيفها من حالات التسريب النصي أو التشابه العالي في Stage 06.5.
</p>

<p>
في هذه المرحلة لا يتم إضافة الأسئلة القانونية اليدوية أو أسئلة خارج النطاق، حيث سيتم إضافتها لاحقًا في Stage 09.5. 
وبذلك تبقى Stage 09 مخصصة لبناء مجموعة تقييم FAQ فقط بطريقة واضحة وقابلة للتدقيق.
</p>

<p>
تم التأكد من أن سجلات FAQ المستخدمة في التقييم غير موجودة داخل قاعدة المعرفة التجريبية، حيث بلغ عدد حالات التسريب داخل قاعدة التجارب <b>0</b>. 
كما تم فحص الأسئلة المكررة وإزالة أي تكرار قبل حفظ ملف التقييم النهائي.
</p>

<p>
تقوم هذه المرحلة أيضًا بحفظ ملفات تدقيق منفصلة للأسئلة المحذوفة بسبب التكرار أو بسبب فحص السؤال والإجابة، مما يعزز الشفافية وقابلية إعادة التحقق من خطوات بناء مجموعة التقييم.
</p>

<div style="
    background-color:#E2F0D9;
    border-right:4px solid #375623;
    padding:12px;
    margin-top:12px;
    border-radius:5px;
">
<b>Conclusion:</b><br>
The FAQ-only evaluation dataset was constructed safely. 
The held-out FAQ records are not indexed in the experimental RAG knowledge base, duplicate questions are removed, and the dataset is ready to be enriched with manual legal and out-of-scope questions in Stage 09.5.
</div>

</div>
"""))


<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:16px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:16px;
    margin:14px 0;
    border-radius:6px;
">

<h3 style="color:#1F4E78; margin-top:0;">
Interpretation — Evaluation Dataset Construction
</h3>

<p>
توضح هذه المرحلة بناء مجموعة التقييم الأولية الخاصة بنظام RAG اعتمادًا على أسئلة FAQ المحجوزة للتقييم فقط. 
تم استخدام سجلات FAQ التي تم فصلها سابقًا عن مجموعة الفهرسة في Stage 06، ثم تنظيفها من حالات التسريب النصي أو التشابه العالي في Stage 06.5.
</p>

<p>
في هذه المرحلة لا يتم إضافة الأسئلة القانونية اليدوية أو أسئلة خارج النطاق، حيث سيتم إضافتها لاحقًا في Stage 09.5. 
وبذلك تبقى Stage 09 مخصصة لبناء مجموعة تقييم FAQ فقط بطريقة واضحة وقابلة للتدقيق.
</p>

<p>
تم التأكد من أن سجلات FAQ المستخدمة في التقييم غير موجودة داخل قاعدة المعرفة التجريبية، حيث بلغ عدد حالات التسريب داخل قاعدة التجارب <b>0</b>. 
كما تم فحص الأسئلة المكررة وإزالة أي تكرار قبل حفظ ملف التقييم النهائي.
</p>

<p>
تقوم هذه المرحلة أيضًا بحفظ ملفات تدقيق منفصلة للأسئلة المحذوفة بسبب التكرار أو بسبب فحص السؤال والإجابة، مما يعزز الشفافية وقابلية إعادة التحقق من خطوات بناء مجموعة التقييم.
</p>

<div style="
    background-color:#E2F0D9;
    border-right:4px solid #375623;
    padding:12px;
    margin-top:12px;
    border-radius:5px;
">
<b>Conclusion:</b><br>
The FAQ-only evaluation dataset was constructed safely. 
The held-out FAQ records are not indexed in the experimental RAG knowledge base, duplicate questions are removed, and the dataset is ready to be enriched with manual legal and out-of-scope questions in Stage 09.5.
</div>

</div>


## Stage 09.5 — Enrich the Evaluation Dataset

This stage enriches the evaluation dataset by adding two manually designed evaluation groups:

1. **Legal article retrieval questions** linked to gold Saudi Labor Law article references.
2. **Out-of-scope questions** used to test refusal behaviour and hallucination control.

The legal questions are not inserted into the knowledge base. They are used only as evaluation queries. This prevents data leakage and avoids unrealistically high retrieval scores.

After enrichment, duplicate questions are removed again using the same normalized-question rule used in Stage 09. Removed duplicates are saved in an audit file so that the preprocessing decisions remain transparent and reproducible.


In [23]:
# =========================================================
# Stage 09.5 - Enrich Evaluation Dataset
# Add Manual Legal Questions and Out-of-Scope Questions
# =========================================================

from IPython.display import display, Markdown
import pandas as pd
import numpy as np
import re
from pathlib import Path

# ---------------------------------------------------------
# 1) Safety checks
# ---------------------------------------------------------

assert "df_eval" in globals(), "df_eval must be created in Stage 09 before this cell."
assert "get_legal_gold" in globals(), "get_legal_gold must be defined in Stage 09 before this cell."
assert "df_articles_active" in globals(), "df_articles_active must exist before Stage 09.5."
assert "df_kb_experiment" in globals(), "df_kb_experiment must exist before Stage 09.5."

if "FINAL_DIR" not in globals():
    FINAL_DIR = Path("03_final")

if "REPORTS_DIR" not in globals():
    REPORTS_DIR = Path("05_reports")

FINAL_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# 2) Helper functions
# ---------------------------------------------------------

def clean_text(value):
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass
    return re.sub(r"\s+", " ", str(value)).strip()


def fallback_normalize_arabic_for_search(text):
    text = clean_text(text)
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ؤ", "و", text)
    text = re.sub(r"ئ", "ي", text)
    text = re.sub(r"ة", "ه", text)
    text = re.sub(r"[\u064B-\u065F\u0670]", "", text)
    text = re.sub(r"[^\u0600-\u06FFa-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


if "normalize_arabic_for_search" not in globals():
    normalize_arabic_for_search = fallback_normalize_arabic_for_search


if "normalize_eval_question_for_dedup" not in globals():
    def normalize_eval_question_for_dedup(text):
        return normalize_arabic_for_search(clean_text(text))


def short_text(value, max_len=150):
    value = clean_text(value)
    return value[:max_len] + ("..." if len(value) > max_len else "")


def hide_index_safe(styler):
    try:
        return styler.hide(axis="index")
    except Exception:
        try:
            return styler.hide_index()
        except Exception:
            return styler


def apply_style_compatible(styler, func, subset):
    if hasattr(styler, "map"):
        return styler.map(func, subset=subset)
    return styler.applymap(func, subset=subset)


def color_status(value):
    value = str(value)

    if value in ["PASS", "OK"]:
        return "background-color:#E2F0D9; color:#375623; font-weight:bold;"

    if value == "WARN":
        return "background-color:#FFF2CC; color:#7F6000; font-weight:bold;"

    if value == "FAIL":
        return "background-color:#FCE4D6; color:#9C0006; font-weight:bold;"

    return ""


def professional_table(df, caption="", width="100%", font_size="12px"):
    styler = (
        df.style
        .set_caption(caption)
        .set_table_styles([
            {
                "selector": "caption",
                "props": [
                    ("caption-side", "top"),
                    ("font-size", "17px"),
                    ("font-weight", "bold"),
                    ("color", "#1F4E78"),
                    ("text-align", "center"),
                    ("padding", "10px")
                ]
            },
            {
                "selector": "th",
                "props": [
                    ("background-color", "#1F4E78"),
                    ("color", "white"),
                    ("font-weight", "bold"),
                    ("text-align", "center"),
                    ("border", "1px solid #D9E2F3"),
                    ("padding", "8px"),
                    ("white-space", "normal")
                ]
            },
            {
                "selector": "td",
                "props": [
                    ("text-align", "center"),
                    ("border", "1px solid #D9E2F3"),
                    ("padding", "7px"),
                    ("vertical-align", "middle"),
                    ("white-space", "normal")
                ]
            },
            {
                "selector": "tbody tr:nth-child(even)",
                "props": [("background-color", "#F8FBFD")]
            },
            {
                "selector": "tbody tr:nth-child(odd)",
                "props": [("background-color", "#FFFFFF")]
            },
            {
                "selector": "table",
                "props": [
                    ("border-collapse", "collapse"),
                    ("width", width),
                    ("font-family", "Arial, Tahoma, sans-serif"),
                    ("font-size", font_size),
                    ("table-layout", "fixed"),
                    ("margin", "12px 0")
                ]
            }
        ])
    )
    return hide_index_safe(styler)


def status_label(condition):
    return "PASS" if condition else "FAIL"

# ---------------------------------------------------------
# 3) Stage explanation
# ---------------------------------------------------------

display(Markdown("""
<div style="
    text-align:left;
    line-height:1.8;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-left:5px solid #1F4E78;
    padding:14px;
    margin:12px 0;
    border-radius:6px;
">

<h3 style="color:#1F4E78; margin-top:0;">
Stage 09.5 — Evaluation Dataset Enrichment
</h3>

<p>
This stage enriches the FAQ-only evaluation dataset by adding manually curated legal article questions and out-of-scope questions.
Manual legal questions are linked to active Saudi Labor Law articles using the <code>get_legal_gold</code> function from Stage 09.
Out-of-scope questions are added to evaluate the system's ability to refuse unrelated requests.
</p>

<p>
The cell is rerun-safe: previously added manual legal and out-of-scope records are removed before adding the updated curated set.
</p>

</div>
"""))

# ---------------------------------------------------------
# 4) Preserve initial status and remove previous manual rows
# ---------------------------------------------------------

df_eval_before_enrichment = df_eval.copy()

if "source_name" in df_eval.columns:
    previous_manual_mask = df_eval["source_name"].astype(str).isin([
        "Manual Legal Evaluation",
        "Manual Evaluation"
    ])
else:
    previous_manual_mask = pd.Series([False] * len(df_eval), index=df_eval.index)

previous_manual_rows_removed = int(previous_manual_mask.sum())

df_eval_base = (
    df_eval[~previous_manual_mask]
    .drop(columns=["eval_id"], errors="ignore")
    .copy()
    .reset_index(drop=True)
)

faq_before_count = int((df_eval_base["eval_type"] == "faq_retrieval").sum())

# ---------------------------------------------------------
# 5) Manual legal evaluation questions
# ---------------------------------------------------------

manual_legal_questions = [
    {"question": "كم مدة فترة التجربة في نظام العمل السعودي؟", "article_numbers": [53], "topic": "Probation period"},
    {"question": "هل يجوز وضع العامل تحت التجربة أكثر من مرة لدى نفس صاحب العمل؟", "article_numbers": [54], "topic": "Probation repetition"},
    {"question": "ما الحد الأقصى لساعات العمل اليومية في نظام العمل؟", "article_numbers": [98], "topic": "Working hours"},
    {"question": "ما الحد الأقصى لساعات العمل الأسبوعية؟", "article_numbers": [98], "topic": "Weekly working hours"},
    {"question": "هل تختلف ساعات العمل في شهر رمضان؟", "article_numbers": [98], "topic": "Ramadan working hours"},
    {"question": "ما مدة الإجازة السنوية المستحقة للعامل؟", "article_numbers": [109], "topic": "Annual leave"},
    {"question": "متى تزيد الإجازة السنوية إلى ثلاثين يومًا؟", "article_numbers": [109], "topic": "Annual leave duration"},
    {"question": "هل يستحق العامل أجرًا عن أيام الإجازة غير المستخدمة عند انتهاء العمل؟", "article_numbers": [111], "topic": "Unused leave compensation"},
    {"question": "هل يستحق العامل إجازة عند الزواج أو وفاة أحد الأقارب؟", "article_numbers": [113], "topic": "Personal leave"},
    {"question": "هل للعامل الحق في إجازة مرضية؟", "article_numbers": [117], "topic": "Sick leave"},
    {"question": "ما حالات انتهاء عقد العمل؟", "article_numbers": [74], "topic": "Contract termination"},
    {"question": "ما مدة الإشعار عند إنهاء العقد غير محدد المدة؟", "article_numbers": [75], "topic": "Notice period"},
    {"question": "متى يستحق العامل تعويضًا عند إنهاء العقد لسبب غير مشروع؟", "article_numbers": [77], "topic": "Unlawful termination"},
    {"question": "متى يجوز لصاحب العمل إنهاء العقد دون مكافأة أو تعويض؟", "article_numbers": [80], "topic": "Termination without reward"},
    {"question": "متى يجوز للعامل ترك العمل دون إشعار مع احتفاظه بحقوقه؟", "article_numbers": [81], "topic": "Leaving without notice"},
    {"question": "متى يستحق العامل مكافأة نهاية الخدمة؟", "article_numbers": [84, 85], "topic": "End-of-service benefit"},
    {"question": "هل يحق لصاحب العمل نقل العامل من مكان عمله الأصلي؟", "article_numbers": [58], "topic": "Worker transfer"},
    {"question": "ما المقصود بعقد العمل في نظام العمل؟", "article_numbers": [50], "topic": "Contract definition"},
    {"question": "كيف يتم دفع أجر العامل؟", "article_numbers": [90], "topic": "Wage payment"},
    {"question": "ما واجبات صاحب العمل تجاه العامل؟", "article_numbers": [61], "topic": "Employer obligations"},
    {"question": "ما التزامات العامل في نظام العمل؟", "article_numbers": [65], "topic": "Worker obligations"},
    {"question": "ما الحد الأقصى لساعات العمل الإضافية وكيف تُحتسب؟", "article_numbers": [107], "topic": "Overtime"},
    {"question": "ما أحكام السلامة والصحة المهنية في بيئة العمل؟", "article_numbers": [121], "topic": "Occupational safety"},
    {"question": "ما حقوق العاملة في إجازة الوضع؟", "article_numbers": [151], "topic": "Maternity leave"},
]

# ---------------------------------------------------------
# 6) Out-of-scope evaluation questions
# ---------------------------------------------------------

manual_oos_questions = [
    "ما حكم الطلاق في الشريعة الإسلامية؟",
    "كم سعر العقار في الرياض؟",
    "ما أفضل سيارة عائلية؟",
    "كيف أفتح مؤسسة تجارية؟",
    "ما علاج الصداع؟",
    "كم سعر الذهب اليوم؟",
    "ما نظام المرور السعودي؟",
    "اكتب لي قصيدة عن الوطن.",
    "ما شروط القرض العقاري؟",
    "ما رسوم الجمارك على السيارات؟",
    "كيف أستثمر في الأسهم؟",
    "ما أفضل مطعم في الرياض؟",
    "ما طريقة استخراج تأشيرة سياحية؟",
    "ما شروط القبول في الجامعة؟",
    "من هو أفضل لاعب كرة قدم؟",
]

# ---------------------------------------------------------
# 7) Build enrichment rows
# ---------------------------------------------------------

extra_rows = []

for item in manual_legal_questions:
    gold = get_legal_gold(item["article_numbers"])

    extra_rows.append({
        "question": item["question"],
        "expected_answer": gold.get("expected_answer", ""),
        "beneficiaries": "",
        "sector": "العمل",
        "subsite": "نظام العمل",
        "legal_category": item["topic"],
        "source_name": "Manual Legal Evaluation",
        "source_url": gold.get("source_url", ""),
        "eval_type": "legal_article_retrieval",
        "gold_source_type": "legal_article",
        "gold_document_unit_id": gold.get("gold_document_unit_id", ""),
        "gold_document_unit_ids": gold.get("gold_document_unit_ids", ""),
        "gold_article_number": gold.get("gold_article_number", ""),
        "gold_article_numbers": gold.get("gold_article_numbers", ""),
        "gold_article_title": gold.get("gold_article_title", ""),
        "is_out_of_scope": False,
        "notes": "Manual legal question. " + (gold.get("notes_extra", "") or ""),
    })

for q in manual_oos_questions:
    extra_rows.append({
        "question": q,
        "expected_answer": "هذا السؤال خارج نطاق نظام العمل السعودي.",
        "beneficiaries": "",
        "sector": "",
        "subsite": "",
        "legal_category": "Out of Scope",
        "source_name": "Manual Evaluation",
        "source_url": "",
        "eval_type": "out_of_scope",
        "gold_source_type": "",
        "gold_document_unit_id": "",
        "gold_document_unit_ids": "",
        "gold_article_number": "",
        "gold_article_numbers": "",
        "gold_article_title": "",
        "is_out_of_scope": True,
        "notes": "Out-of-scope rejection test.",
    })

df_extra_eval = pd.DataFrame(extra_rows)

# ---------------------------------------------------------
# 8) Align columns with Stage 09 schema
# ---------------------------------------------------------

base_cols = list(df_eval_base.columns)

for col in base_cols:
    if col not in df_extra_eval.columns:
        df_extra_eval[col] = ""

df_extra_eval = df_extra_eval[base_cols]

# ---------------------------------------------------------
# 9) Append enrichment records and remove duplicates
# ---------------------------------------------------------

df_eval = pd.concat(
    [df_eval_base, df_extra_eval],
    ignore_index=True
)

df_eval["question"] = df_eval["question"].fillna("").astype(str).str.strip()

df_eval["question_normalized_for_dedup"] = (
    df_eval["question"]
    .apply(normalize_eval_question_for_dedup)
)

enrichment_duplicate_mask = df_eval.duplicated(
    subset=["question_normalized_for_dedup"],
    keep="first"
)

df_eval_enrichment_duplicates_removed = df_eval[enrichment_duplicate_mask].copy()
enrichment_duplicates_removed_count = int(enrichment_duplicate_mask.sum())

df_eval = (
    df_eval[~enrichment_duplicate_mask]
    .drop(columns=["question_normalized_for_dedup"], errors="ignore")
    .reset_index(drop=True)
)

df_eval_enrichment_duplicates_removed = df_eval_enrichment_duplicates_removed.drop(
    columns=["question_normalized_for_dedup"],
    errors="ignore"
)

df_eval.insert(0, "eval_id", range(1, len(df_eval) + 1))

# ---------------------------------------------------------
# 10) Validation checks
# ---------------------------------------------------------

faq_count = int((df_eval["eval_type"] == "faq_retrieval").sum())
legal_count = int((df_eval["eval_type"] == "legal_article_retrieval").sum())
oos_count = int((df_eval["eval_type"] == "out_of_scope").sum())

missing_gold = df_eval[
    (df_eval["eval_type"] == "legal_article_retrieval") &
    (df_eval["gold_document_unit_ids"].astype(str).str.strip().eq(""))
].copy()

duplicate_eval_questions_after = int(
    df_eval["question"]
    .fillna("")
    .astype(str)
    .apply(normalize_eval_question_for_dedup)
    .duplicated()
    .sum()
)

unknown_eval_types = sorted(
    set(df_eval["eval_type"].dropna().astype(str).unique())
    -
    {"faq_retrieval", "legal_article_retrieval", "out_of_scope"}
)

faq_eval_ids = set(
    df_eval[df_eval["eval_type"].eq("faq_retrieval")]["gold_document_unit_id"]
    .astype(str)
)

kb_faq_ids = set(
    df_kb_experiment[
        df_kb_experiment["source_type"].astype(str).eq("faq")
    ]["document_unit_id"].astype(str)
)

faq_leakage_after_enrichment = len(faq_eval_ids & kb_faq_ids)

assert len(missing_gold) == 0, (
    "Some manual legal questions did not match active legal articles. "
    "Review gold article numbers."
)

assert duplicate_eval_questions_after == 0, (
    "Duplicate evaluation questions remain after Stage 09.5 enrichment."
)

assert len(unknown_eval_types) == 0, (
    f"Unknown eval_type values found: {unknown_eval_types}"
)

assert faq_leakage_after_enrichment == 0, (
    "FAQ evaluation records leaked into experimental KB after enrichment."
)

assert legal_count > 0, "No manual legal questions were added."
assert oos_count > 0, "No out-of-scope questions were added."

# ---------------------------------------------------------
# 11) Save updated evaluation dataset and audit files
# ---------------------------------------------------------

eval_csv_path = FINAL_DIR / "rag_evaluation_dataset.csv"
eval_xlsx_path = FINAL_DIR / "rag_evaluation_dataset.xlsx"

enrichment_duplicates_csv_path = FINAL_DIR / "rag_evaluation_duplicates_removed_after_enrichment.csv"
enrichment_duplicates_xlsx_path = FINAL_DIR / "rag_evaluation_duplicates_removed_after_enrichment.xlsx"

df_eval.to_csv(eval_csv_path, index=False, encoding="utf-8-sig")
df_eval.to_excel(eval_xlsx_path, index=False)

df_eval_enrichment_duplicates_removed.to_csv(
    enrichment_duplicates_csv_path,
    index=False,
    encoding="utf-8-sig"
)

df_eval_enrichment_duplicates_removed.to_excel(
    enrichment_duplicates_xlsx_path,
    index=False
)

# ---------------------------------------------------------
# 12) Professional summary tables
# ---------------------------------------------------------

enrichment_summary_df = pd.DataFrame([
    {
        "Component": "FAQ evaluation questions retained from Stage 09",
        "Count": faq_count,
        "Status": "PASS" if faq_count > 0 else "FAIL",
        "Interpretation": "Held-out FAQ questions remain in the final evaluation dataset."
    },
    {
        "Component": "Manual legal questions added",
        "Count": legal_count,
        "Status": "PASS" if legal_count > 0 else "FAIL",
        "Interpretation": "Legal questions are linked to active labor-law articles as gold references."
    },
    {
        "Component": "Out-of-scope questions added",
        "Count": oos_count,
        "Status": "PASS" if oos_count > 0 else "FAIL",
        "Interpretation": "OOS questions test refusal and hallucination control."
    },
    {
        "Component": "Previous manual rows removed before rerun",
        "Count": previous_manual_rows_removed,
        "Status": "OK",
        "Interpretation": "Makes Stage 09.5 rerun-safe and prevents duplicated manual questions."
    },
    {
        "Component": "Duplicates removed after enrichment",
        "Count": enrichment_duplicates_removed_count,
        "Status": "PASS",
        "Interpretation": "Duplicate questions after enrichment were removed and saved in an audit file."
    },
    {
        "Component": "Final evaluation dataset size",
        "Count": len(df_eval),
        "Status": "PASS" if len(df_eval) > 0 else "FAIL",
        "Interpretation": "Final enriched evaluation dataset used in retrieval experiments."
    }
])


# ---------------------------------------------------------
# 12) Professional summary tables
# ---------------------------------------------------------

summary_style = professional_table(
    enrichment_summary_df,
    caption="Table 9.7 — Evaluation Dataset Enrichment Summary",
    width="100%",
    font_size="12px"
)

summary_style = apply_style_compatible(
    summary_style,
    color_status,
    subset=["Status"]
)

display(summary_style)

eval_distribution_df = (
    df_eval["eval_type"]
    .value_counts()
    .rename_axis("Evaluation Type")
    .reset_index(name="Records")
)

eval_distribution_df["Percentage"] = (
    eval_distribution_df["Records"] / eval_distribution_df["Records"].sum() * 100
).round(2)

display(
    professional_table(
        eval_distribution_df,
        caption="Table 9.8 — Final Evaluation Dataset Composition",
        width="70%",
        font_size="12px"
    )
)

# ---------------------------------------------------------
# 13) Gold coverage table for manual legal questions
# ---------------------------------------------------------

legal_gold_preview = df_eval[
    df_eval["eval_type"].eq("legal_article_retrieval")
][
    [
        "question",
        "gold_article_numbers",
        "gold_article_title",
        "gold_document_unit_ids",
        "legal_category"
    ]
].copy()

legal_gold_preview["question"] = legal_gold_preview["question"].apply(lambda x: short_text(x, 130))
legal_gold_preview["gold_article_title"] = legal_gold_preview["gold_article_title"].apply(lambda x: short_text(x, 120))

gold_style = professional_table(
    legal_gold_preview.head(15),
    caption="Table 9.9 — Manual Legal Questions and Gold Article References",
    width="100%",
    font_size="11.5px"
)

gold_style = gold_style.set_properties(
    subset=["question", "gold_article_title"],
    **{
        "text-align": "right",
        "direction": "rtl",
        "line-height": "1.7"
    }
)

gold_style = gold_style.set_properties(
    subset=["gold_document_unit_ids"],
    **{
        "text-align": "left",
        "direction": "ltr",
        "font-size": "10px"
    }
)

display(gold_style)

# ---------------------------------------------------------
# 14) Out-of-scope preview
# ---------------------------------------------------------

oos_preview = df_eval[
    df_eval["eval_type"].eq("out_of_scope")
][["question", "expected_answer", "is_out_of_scope", "notes"]].head(10).copy()

oos_preview["question"] = oos_preview["question"].apply(lambda x: short_text(x, 130))

oos_style = professional_table(
    oos_preview,
    caption="Table 9.10 — Out-of-Scope Evaluation Samples",
    width="100%",
    font_size="11.5px"
)

oos_style = oos_style.set_properties(
    subset=["question", "expected_answer"],
    **{
        "text-align": "right",
        "direction": "rtl",
        "line-height": "1.7"
    }
)

display(oos_style)

# ---------------------------------------------------------
# 15) Final safety table
# ---------------------------------------------------------

safety_df = pd.DataFrame([
    {
        "Safety Check": "Missing legal gold article references",
        "Value": len(missing_gold),
        "Expected": 0,
        "Status": status_label(len(missing_gold) == 0),
        "Interpretation": "Every manual legal question must map to active legal article IDs."
    },
    {
        "Safety Check": "Duplicate evaluation questions after enrichment",
        "Value": duplicate_eval_questions_after,
        "Expected": 0,
        "Status": status_label(duplicate_eval_questions_after == 0),
        "Interpretation": "The final evaluation set should not contain duplicate questions."
    },
    {
        "Safety Check": "FAQ leakage inside experimental KB after enrichment",
        "Value": faq_leakage_after_enrichment,
        "Expected": 0,
        "Status": status_label(faq_leakage_after_enrichment == 0),
        "Interpretation": "Held-out FAQ questions must not appear in the experimental KB."
    },
    {
        "Safety Check": "Unknown eval_type values",
        "Value": len(unknown_eval_types),
        "Expected": 0,
        "Status": status_label(len(unknown_eval_types) == 0),
        "Interpretation": "Only faq_retrieval, legal_article_retrieval, and out_of_scope are allowed."
    }
])

safety_style = professional_table(
    safety_df,
    caption="Table 9.11 — Stage 09.5 Enrichment Safety Checks",
    width="100%",
    font_size="12px"
)

safety_style = apply_style_compatible(
    safety_style,
    color_status,
    subset=["Status"]
)

display(safety_style)

# ---------------------------------------------------------
# 16) Final message
# ---------------------------------------------------------

display(Markdown(f"""
<div style="
    text-align:left;
    line-height:1.8;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#E2F0D9;
    border-left:5px solid #375623;
    padding:14px;
    margin:12px 0;
    border-radius:6px;
">

<b>Stage 09.5 completed successfully.</b><br>
Final evaluation dataset records: <b>{len(df_eval):,}</b><br>
FAQ retrieval records: <b>{faq_count:,}</b><br>
Manual legal retrieval records: <b>{legal_count:,}</b><br>
Out-of-scope records: <b>{oos_count:,}</b><br>
Missing legal gold references: <b>{len(missing_gold):,}</b><br>
Duplicate questions after enrichment: <b>{duplicate_eval_questions_after:,}</b><br>
FAQ leakage after enrichment: <b>{faq_leakage_after_enrichment:,}</b><br>

</div>
"""))

print("Stage 09.5 completed successfully.")
print("Final evaluation dataset records:", len(df_eval))
print("FAQ:", faq_count, "| Legal:", legal_count, "| OOS:", oos_count)
print("Missing legal gold references:", len(missing_gold))
print("Duplicate questions after enrichment:", duplicate_eval_questions_after)
print("FAQ leakage after enrichment:", faq_leakage_after_enrichment)
print("Saved:", eval_csv_path)


<div style="
    text-align:left;
    line-height:1.8;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-left:5px solid #1F4E78;
    padding:14px;
    margin:12px 0;
    border-radius:6px;
">

<h3 style="color:#1F4E78; margin-top:0;">
Stage 09.5 — Evaluation Dataset Enrichment
</h3>

<p>
This stage enriches the FAQ-only evaluation dataset by adding manually curated legal article questions and out-of-scope questions.
Manual legal questions are linked to active Saudi Labor Law articles using the <code>get_legal_gold</code> function from Stage 09.
Out-of-scope questions are added to evaluate the system's ability to refuse unrelated requests.
</p>

<p>
The cell is rerun-safe: previously added manual legal and out-of-scope records are removed before adding the updated curated set.
</p>

</div>


Component,Count,Status,Interpretation
FAQ evaluation questions retained from Stage 09,121,PASS,Held-out FAQ questions remain in the final evaluation dataset.
Manual legal questions added,24,PASS,Legal questions are linked to active labor-law articles as gold references.
Out-of-scope questions added,15,PASS,OOS questions test refusal and hallucination control.
Previous manual rows removed before rerun,0,OK,Makes Stage 09.5 rerun-safe and prevents duplicated manual questions.
Duplicates removed after enrichment,0,PASS,Duplicate questions after enrichment were removed and saved in an audit file.
Final evaluation dataset size,160,PASS,Final enriched evaluation dataset used in retrieval experiments.


Evaluation Type,Records,Percentage
faq_retrieval,121,75.620000
legal_article_retrieval,24,15.000000
out_of_scope,15,9.380000


question,gold_article_numbers,gold_article_title,gold_document_unit_ids,legal_category
كم مدة فترة التجربة في نظام العمل السعودي؟,53,المادة الثالثة والخمسون :,b56e9c99f1c551928c8727dae90850186c14efc78d6272f429f0c448add3f3dc,Probation period
هل يجوز وضع العامل تحت التجربة أكثر من مرة لدى نفس صاحب العمل؟,54,المادة الرابعة والخمسون:,ab8af4e6ecab46b0bf5b588cc55664b9d0a008dc9073912d3417c1f04d083142,Probation repetition
ما الحد الأقصى لساعات العمل اليومية في نظام العمل؟,98,المادة الثامنة والتسعون:,a28d9d1238f829c664612f6e0f62f0152a39351f1f09ec1d7016a52c65a95792,Working hours
ما الحد الأقصى لساعات العمل الأسبوعية؟,98,المادة الثامنة والتسعون:,a28d9d1238f829c664612f6e0f62f0152a39351f1f09ec1d7016a52c65a95792,Weekly working hours
هل تختلف ساعات العمل في شهر رمضان؟,98,المادة الثامنة والتسعون:,a28d9d1238f829c664612f6e0f62f0152a39351f1f09ec1d7016a52c65a95792,Ramadan working hours
ما مدة الإجازة السنوية المستحقة للعامل؟,109,المادة التاسعة بعد المائة:,80b9f2ab3f90233bc03c55e64eb9eaad142f2e0e7b5ea944fb810b1abf71cfed,Annual leave
متى تزيد الإجازة السنوية إلى ثلاثين يومًا؟,109,المادة التاسعة بعد المائة:,80b9f2ab3f90233bc03c55e64eb9eaad142f2e0e7b5ea944fb810b1abf71cfed,Annual leave duration
هل يستحق العامل أجرًا عن أيام الإجازة غير المستخدمة عند انتهاء العمل؟,111,المادة الحادية عشرة بعد المائة:,6e6c8673adaa51e0d5e36d8807155b8b4a11c80dbdacd9d0003d61cadafb2edd,Unused leave compensation
هل يستحق العامل إجازة عند الزواج أو وفاة أحد الأقارب؟,113,المادة الثالثة عشرة بعد المائة :,bf42acecf7d911571f3f1c105f9d51c8e30d781beae5d22afe803a238479a1b7,Personal leave
هل للعامل الحق في إجازة مرضية؟,117,المادة السابعة عشرة بعد المائة:,2183bea86d3647611107df091730d37e79fdbc2b3c9ae4b5ba291c94cd07d19f,Sick leave


question,expected_answer,is_out_of_scope,notes
ما حكم الطلاق في الشريعة الإسلامية؟,هذا السؤال خارج نطاق نظام العمل السعودي.,True,Out-of-scope rejection test.
كم سعر العقار في الرياض؟,هذا السؤال خارج نطاق نظام العمل السعودي.,True,Out-of-scope rejection test.
ما أفضل سيارة عائلية؟,هذا السؤال خارج نطاق نظام العمل السعودي.,True,Out-of-scope rejection test.
كيف أفتح مؤسسة تجارية؟,هذا السؤال خارج نطاق نظام العمل السعودي.,True,Out-of-scope rejection test.
ما علاج الصداع؟,هذا السؤال خارج نطاق نظام العمل السعودي.,True,Out-of-scope rejection test.
كم سعر الذهب اليوم؟,هذا السؤال خارج نطاق نظام العمل السعودي.,True,Out-of-scope rejection test.
ما نظام المرور السعودي؟,هذا السؤال خارج نطاق نظام العمل السعودي.,True,Out-of-scope rejection test.
اكتب لي قصيدة عن الوطن.,هذا السؤال خارج نطاق نظام العمل السعودي.,True,Out-of-scope rejection test.
ما شروط القرض العقاري؟,هذا السؤال خارج نطاق نظام العمل السعودي.,True,Out-of-scope rejection test.
ما رسوم الجمارك على السيارات؟,هذا السؤال خارج نطاق نظام العمل السعودي.,True,Out-of-scope rejection test.


Safety Check,Value,Expected,Status,Interpretation
Missing legal gold article references,0,0,PASS,Every manual legal question must map to active legal article IDs.
Duplicate evaluation questions after enrichment,0,0,PASS,The final evaluation set should not contain duplicate questions.
FAQ leakage inside experimental KB after enrichment,0,0,PASS,Held-out FAQ questions must not appear in the experimental KB.
Unknown eval_type values,0,0,PASS,"Only faq_retrieval, legal_article_retrieval, and out_of_scope are allowed."



<div style="
    text-align:left;
    line-height:1.8;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#E2F0D9;
    border-left:5px solid #375623;
    padding:14px;
    margin:12px 0;
    border-radius:6px;
">

<b>Stage 09.5 completed successfully.</b><br>
Final evaluation dataset records: <b>160</b><br>
FAQ retrieval records: <b>121</b><br>
Manual legal retrieval records: <b>24</b><br>
Out-of-scope records: <b>15</b><br>
Missing legal gold references: <b>0</b><br>
Duplicate questions after enrichment: <b>0</b><br>
FAQ leakage after enrichment: <b>0</b><br>

</div>


Stage 09.5 completed successfully.
Final evaluation dataset records: 160
FAQ: 121 | Legal: 24 | OOS: 15
Missing legal gold references: 0
Duplicate questions after enrichment: 0
FAQ leakage after enrichment: 0
Saved: saudi_labor_law_voice_agent_project\03_final\rag_evaluation_dataset.csv


In [24]:
from IPython.display import display, Markdown

faq_count = int((df_eval["eval_type"] == "faq_retrieval").sum())
legal_count = int((df_eval["eval_type"] == "legal_article_retrieval").sum())
oos_count = int((df_eval["eval_type"] == "out_of_scope").sum())
total_count = len(df_eval)

display(Markdown(f"""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:16px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:16px;
    margin:14px 0;
    border-radius:6px;
">

<h3 style="color:#1F4E78; margin-top:0;">
Interpretation — Final Enriched Evaluation Dataset
</h3>

<p>
توضح هذه المرحلة الإثراء النهائي لمجموعة التقييم المستخدمة في تجارب الاسترجاع. 
بعد بناء مجموعة تقييم FAQ المحجوزة سابقًا، تمت إضافة أسئلة قانونية يدوية مرتبطة بمواد محددة من نظام العمل السعودي، بالإضافة إلى أسئلة خارج النطاق لاختبار قدرة النظام على الرفض وعدم الهلوسة.
</p>

<p>
بلغ الحجم النهائي لمجموعة التقييم <b>{total_count}</b> سؤالًا، موزعة إلى <b>{faq_count}</b> سؤال FAQ، و <b>{legal_count}</b> سؤالًا قانونيًا يدويًا، و <b>{oos_count}</b> سؤالًا خارج النطاق. 
هذا التوزيع يجعل التقييم أكثر شمولًا، لأنه لا يختبر استرجاع FAQ فقط، بل يختبر أيضًا قدرة النظام على استرجاع المواد القانونية الصحيحة والتعامل مع الأسئلة غير المتعلقة بنظام العمل.
</p>

<p>
تم ربط الأسئلة القانونية اليدوية بمواد ذهبية محددة من نظام العمل، مثل مواد فترة التجربة، ساعات العمل، الإجازات، إنهاء العقد، مكافأة نهاية الخدمة، والسلامة المهنية. 
وجود <code>gold_article_numbers</code> و <code>gold_document_unit_ids</code> يسمح بحساب مقاييس الاسترجاع بشكل موضوعي مثل <code>Recall@k</code> و <code>MRR</code> و <code>nDCG</code>.
</p>

<p>
كما أضيفت أسئلة خارج النطاق لاختبار طبقة الحماية ومنع الهلوسة، بحيث يجب على النظام رفض الأسئلة غير المرتبطة بنظام العمل السعودي بدلًا من توليد إجابات غير مدعومة.
</p>

<div style="
    background-color:#E2F0D9;
    border-right:4px solid #375623;
    padding:12px;
    margin-top:12px;
    border-radius:5px;
">
<b>Conclusion:</b><br>
The final enriched evaluation dataset is valid and useful for the next retrieval-evaluation stage. 
It combines held-out FAQ questions, manually curated legal questions with gold article references, and out-of-scope questions for guardrail testing.
</div>

</div>
"""))


<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:16px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:16px;
    margin:14px 0;
    border-radius:6px;
">

<h3 style="color:#1F4E78; margin-top:0;">
Interpretation — Final Enriched Evaluation Dataset
</h3>

<p>
توضح هذه المرحلة الإثراء النهائي لمجموعة التقييم المستخدمة في تجارب الاسترجاع. 
بعد بناء مجموعة تقييم FAQ المحجوزة سابقًا، تمت إضافة أسئلة قانونية يدوية مرتبطة بمواد محددة من نظام العمل السعودي، بالإضافة إلى أسئلة خارج النطاق لاختبار قدرة النظام على الرفض وعدم الهلوسة.
</p>

<p>
بلغ الحجم النهائي لمجموعة التقييم <b>160</b> سؤالًا، موزعة إلى <b>121</b> سؤال FAQ، و <b>24</b> سؤالًا قانونيًا يدويًا، و <b>15</b> سؤالًا خارج النطاق. 
هذا التوزيع يجعل التقييم أكثر شمولًا، لأنه لا يختبر استرجاع FAQ فقط، بل يختبر أيضًا قدرة النظام على استرجاع المواد القانونية الصحيحة والتعامل مع الأسئلة غير المتعلقة بنظام العمل.
</p>

<p>
تم ربط الأسئلة القانونية اليدوية بمواد ذهبية محددة من نظام العمل، مثل مواد فترة التجربة، ساعات العمل، الإجازات، إنهاء العقد، مكافأة نهاية الخدمة، والسلامة المهنية. 
وجود <code>gold_article_numbers</code> و <code>gold_document_unit_ids</code> يسمح بحساب مقاييس الاسترجاع بشكل موضوعي مثل <code>Recall@k</code> و <code>MRR</code> و <code>nDCG</code>.
</p>

<p>
كما أضيفت أسئلة خارج النطاق لاختبار طبقة الحماية ومنع الهلوسة، بحيث يجب على النظام رفض الأسئلة غير المرتبطة بنظام العمل السعودي بدلًا من توليد إجابات غير مدعومة.
</p>

<div style="
    background-color:#E2F0D9;
    border-right:4px solid #375623;
    padding:12px;
    margin-top:12px;
    border-radius:5px;
">
<b>Conclusion:</b><br>
The final enriched evaluation dataset is valid and useful for the next retrieval-evaluation stage. 
It combines held-out FAQ questions, manually curated legal questions with gold article references, and out-of-scope questions for guardrail testing.
</div>

</div>


## Stage 09.6 — Evaluation Dataset Gold Review and Retrieval Eligibility

This stage audits the final evaluation dataset after enrichment.

Article-level retrieval metrics such as Recall@1, Recall@5, MRR, and nDCG require a valid gold article number.  
Any legal question without a gold article is retained for qualitative review but excluded from retrieval metrics.


In [25]:
# =========================================================
# Stage 09.6 - Evaluation Gold Audit and Retrieval Eligibility
# مراجعة أرقام المواد الذهبية وتحديد صلاحية سجلات التقييم للاسترجاع
# =========================================================

from IPython.display import display, Markdown
import pandas as pd
import numpy as np
import re
from pathlib import Path

# ---------------------------------------------------------
# 1) Safety checks
# ---------------------------------------------------------

assert "df_eval" in globals(), "df_eval must exist before Stage 09.6."

if "FINAL_DIR" not in globals():
    FINAL_DIR = Path("03_final")

if "REPORTS_DIR" not in globals():
    REPORTS_DIR = Path("05_reports")

FINAL_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

df_eval_reviewed = df_eval.copy().reset_index(drop=True)

# ---------------------------------------------------------
# 2) Helper functions
# ---------------------------------------------------------

def _normalize_digits(text):
    text = str(text)
    arabic_digits = "٠١٢٣٤٥٦٧٨٩"
    persian_digits = "۰۱۲۳۴۵۶۷۸۹"
    english_digits = "0123456789"
    trans = {}

    for a, e in zip(arabic_digits, english_digits):
        trans[ord(a)] = e

    for p, e in zip(persian_digits, english_digits):
        trans[ord(p)] = e

    return text.translate(trans)


def _extract_article_numbers(value):
    if value is None:
        return []

    try:
        if pd.isna(value):
            return []
    except Exception:
        pass

    text = _normalize_digits(str(value)).strip()

    if text.lower() in ["", "nan", "none", "<na>", "n/a"]:
        return []

    # Convert values such as 54.0 into 54
    text = re.sub(r"(\d+)\s*[\.,]\s*0+\b", r"\1", text)

    nums = re.findall(r"\d+", text)

    out = []
    for n in nums:
        try:
            n_int = int(n)
            if 1 <= n_int <= 300:
                n_str = str(n_int)
                if n_str not in out:
                    out.append(n_str)
        except Exception:
            pass

    return out


def _article_numbers_to_text(value):
    """
    Keep article numbers comma-separated for compatibility.
    Example: 84,85
    """
    nums = _extract_article_numbers(value)
    return ",".join(nums) if nums else ""


def _first_nonempty_gold(row):
    candidate_cols = [
        "gold_article_numbers",
        "gold_article_number",
        "article_numbers",
        "article_number_int",
        "expected_article_number"
    ]

    for col in candidate_cols:
        if col in row.index:
            v = _article_numbers_to_text(row.get(col))
            if v:
                return v

    return ""


def short_text(value, max_len=150):
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass
    value = re.sub(r"\s+", " ", str(value)).strip()
    return value[:max_len] + ("..." if len(value) > max_len else "")


def hide_index_safe(styler):
    try:
        return styler.hide(axis="index")
    except Exception:
        try:
            return styler.hide_index()
        except Exception:
            return styler


def apply_style_compatible(styler, func, subset):
    if hasattr(styler, "map"):
        return styler.map(func, subset=subset)
    return styler.applymap(func, subset=subset)


def color_status(value):
    value = str(value)

    if value in ["PASS", "OK"]:
        return "background-color:#E2F0D9; color:#375623; font-weight:bold;"

    if value == "WARN":
        return "background-color:#FFF2CC; color:#7F6000; font-weight:bold;"

    if value == "FAIL":
        return "background-color:#FCE4D6; color:#9C0006; font-weight:bold;"

    return ""


def professional_table(df, caption="", width="100%", font_size="12px"):
    styler = (
        df.style
        .set_caption(caption)
        .set_table_styles([
            {
                "selector": "caption",
                "props": [
                    ("caption-side", "top"),
                    ("font-size", "17px"),
                    ("font-weight", "bold"),
                    ("color", "#1F4E78"),
                    ("text-align", "center"),
                    ("padding", "10px")
                ]
            },
            {
                "selector": "th",
                "props": [
                    ("background-color", "#1F4E78"),
                    ("color", "white"),
                    ("font-weight", "bold"),
                    ("text-align", "center"),
                    ("border", "1px solid #D9E2F3"),
                    ("padding", "8px"),
                    ("white-space", "normal")
                ]
            },
            {
                "selector": "td",
                "props": [
                    ("text-align", "center"),
                    ("border", "1px solid #D9E2F3"),
                    ("padding", "7px"),
                    ("vertical-align", "middle"),
                    ("white-space", "normal")
                ]
            },
            {
                "selector": "tbody tr:nth-child(even)",
                "props": [("background-color", "#F8FBFD")]
            },
            {
                "selector": "tbody tr:nth-child(odd)",
                "props": [("background-color", "#FFFFFF")]
            },
            {
                "selector": "table",
                "props": [
                    ("border-collapse", "collapse"),
                    ("width", width),
                    ("font-family", "Arial, Tahoma, sans-serif"),
                    ("font-size", font_size),
                    ("table-layout", "fixed"),
                    ("margin", "12px 0")
                ]
            }
        ])
    )

    return hide_index_safe(styler)

# ---------------------------------------------------------
# 3) Normalize gold article columns
# ---------------------------------------------------------

if "gold_article_numbers" not in df_eval_reviewed.columns:
    df_eval_reviewed["gold_article_numbers"] = ""

df_eval_reviewed["gold_article_numbers"] = df_eval_reviewed.apply(
    _first_nonempty_gold,
    axis=1
)

df_eval_reviewed["has_valid_gold_article"] = df_eval_reviewed["gold_article_numbers"].apply(
    lambda x: len(_extract_article_numbers(x)) > 0
)

if "eval_type_original" not in df_eval_reviewed.columns:
    df_eval_reviewed["eval_type_original"] = df_eval_reviewed["eval_type"]

# ---------------------------------------------------------
# 4) Set retrieval eligibility policy
# ---------------------------------------------------------

df_eval_reviewed["is_retrieval_eval"] = False
df_eval_reviewed["retrieval_eval_level"] = "not_applicable"
df_eval_reviewed["gold_validation_status"] = "not_required"

faq_mask = df_eval_reviewed["eval_type"].astype(str).eq("faq_retrieval")
legal_mask = df_eval_reviewed["eval_type"].astype(str).eq("legal_article_retrieval")
oos_mask = df_eval_reviewed["eval_type"].astype(str).eq("out_of_scope")

legal_valid_mask = legal_mask & df_eval_reviewed["has_valid_gold_article"]
legal_missing_mask = legal_mask & ~df_eval_reviewed["has_valid_gold_article"]

# FAQ evaluation
df_eval_reviewed.loc[faq_mask, "is_retrieval_eval"] = True
df_eval_reviewed.loc[faq_mask, "retrieval_eval_level"] = "faq_document_unit"
df_eval_reviewed.loc[faq_mask, "gold_validation_status"] = "faq_gold_document_unit"

# Legal evaluation with valid gold articles
df_eval_reviewed.loc[legal_valid_mask, "is_retrieval_eval"] = True
df_eval_reviewed.loc[legal_valid_mask, "retrieval_eval_level"] = "legal_article"
df_eval_reviewed.loc[legal_valid_mask, "gold_validation_status"] = "valid_gold_article"

# Legal questions missing gold articles
df_eval_reviewed.loc[legal_missing_mask, "is_retrieval_eval"] = False
df_eval_reviewed.loc[legal_missing_mask, "retrieval_eval_level"] = "legal_unlabeled_review"
df_eval_reviewed.loc[legal_missing_mask, "gold_validation_status"] = "missing_gold_article_not_used_for_retrieval_metrics"

# Out-of-scope questions
df_eval_reviewed.loc[oos_mask, "is_retrieval_eval"] = False
df_eval_reviewed.loc[oos_mask, "retrieval_eval_level"] = "router_oos"
df_eval_reviewed.loc[oos_mask, "gold_validation_status"] = "not_required_oos"

df_eval_reviewed["eval_type_for_retrieval"] = df_eval_reviewed["eval_type"]
df_eval_reviewed.loc[legal_missing_mask, "eval_type_for_retrieval"] = "legal_unlabeled_review"

# ---------------------------------------------------------
# 5) Build output subsets
# ---------------------------------------------------------

df_retrieval_eval_valid_only = (
    df_eval_reviewed[df_eval_reviewed["is_retrieval_eval"]]
    .copy()
    .reset_index(drop=True)
)

df_legal_missing_gold = (
    df_eval_reviewed[legal_missing_mask]
    .copy()
    .reset_index(drop=True)
)

df_legal_valid_gold = (
    df_eval_reviewed[legal_valid_mask]
    .copy()
    .reset_index(drop=True)
)

# ---------------------------------------------------------
# 6) Save reviewed outputs
# ---------------------------------------------------------

reviewed_eval_csv = FINAL_DIR / "rag_evaluation_dataset.csv"
reviewed_eval_xlsx = FINAL_DIR / "rag_evaluation_dataset.xlsx"

retrieval_valid_csv = FINAL_DIR / "rag_retrieval_evaluation_dataset_valid_only.csv"
retrieval_valid_xlsx = FINAL_DIR / "rag_retrieval_evaluation_dataset_valid_only.xlsx"

missing_gold_csv = FINAL_DIR / "legal_questions_missing_gold_articles.csv"
missing_gold_xlsx = FINAL_DIR / "legal_questions_missing_gold_articles.xlsx"

valid_gold_csv = FINAL_DIR / "legal_questions_with_valid_gold_articles.csv"
valid_gold_xlsx = FINAL_DIR / "legal_questions_with_valid_gold_articles.xlsx"

df_eval_reviewed.to_csv(reviewed_eval_csv, index=False, encoding="utf-8-sig")
df_eval_reviewed.to_excel(reviewed_eval_xlsx, index=False)

df_retrieval_eval_valid_only.to_csv(retrieval_valid_csv, index=False, encoding="utf-8-sig")
df_retrieval_eval_valid_only.to_excel(retrieval_valid_xlsx, index=False)

df_legal_missing_gold.to_csv(missing_gold_csv, index=False, encoding="utf-8-sig")
df_legal_missing_gold.to_excel(missing_gold_xlsx, index=False)

df_legal_valid_gold.to_csv(valid_gold_csv, index=False, encoding="utf-8-sig")
df_legal_valid_gold.to_excel(valid_gold_xlsx, index=False)

# ---------------------------------------------------------
# 7) Summary and assertions
# ---------------------------------------------------------

eval_audit_summary = pd.DataFrame([
    {
        "Metric": "All evaluation records",
        "Count": len(df_eval_reviewed),
        "Interpretation": "Complete evaluation dataset including FAQ, legal, and OOS."
    },
    {
        "Metric": "FAQ retrieval records",
        "Count": int(faq_mask.sum()),
        "Interpretation": "Valid for document-level FAQ retrieval evaluation."
    },
    {
        "Metric": "Legal records with valid gold articles",
        "Count": len(df_legal_valid_gold),
        "Interpretation": "Valid for article-level legal retrieval evaluation."
    },
    {
        "Metric": "Legal records missing gold articles",
        "Count": len(df_legal_missing_gold),
        "Interpretation": "Kept for qualitative review, excluded from article-level metrics."
    },
    {
        "Metric": "Out-of-scope records",
        "Count": int(oos_mask.sum()),
        "Interpretation": "Used for guardrail/router tests, not retrieval relevance metrics."
    },
    {
        "Metric": "Final retrieval-evaluable records",
        "Count": len(df_retrieval_eval_valid_only),
        "Interpretation": "Recommended subset for Stage 03 retrieval experiments."
    },
])

display(Markdown("""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:14px;
    margin:12px 0;
    border-radius:6px;
">
<h3 style="color:#1F4E78; margin-top:0;">
Stage 09.6 — Evaluation Dataset Gold Audit
</h3>
<p>
تمت إضافة أعلام تدقيق إلى داتا التقييم. أي سؤال قانوني بدون رقم مادة ذهبي لا يدخل في حساب 
Recall أو MRR أو nDCG، بل يبقى محفوظًا في ملف مراجعة منفصل.
</p>
</div>
"""))

display(
    professional_table(
        eval_audit_summary,
        caption="Table 9.12 — Evaluation Gold Audit Summary",
        width="90%",
        font_size="12px"
    )
)

# ---------------------------------------------------------
# 8) Safety checks table
# ---------------------------------------------------------

gold_audit_safety = pd.DataFrame([
    {
        "Safety Check": "Legal retrieval rows missing gold articles",
        "Value": len(df_legal_missing_gold),
        "Expected": 0,
        "Status": "PASS" if len(df_legal_missing_gold) == 0 else "WARN",
        "Interpretation": "Legal retrieval rows should have valid gold article numbers."
    },
    {
        "Safety Check": "Retrieval-evaluable OOS records",
        "Value": int(df_retrieval_eval_valid_only["eval_type"].astype(str).eq("out_of_scope").sum()),
        "Expected": 0,
        "Status": "PASS" if int(df_retrieval_eval_valid_only["eval_type"].astype(str).eq("out_of_scope").sum()) == 0 else "FAIL",
        "Interpretation": "Out-of-scope records should be excluded from retrieval metrics."
    },
    {
        "Safety Check": "Final retrieval-evaluable records",
        "Value": len(df_retrieval_eval_valid_only),
        "Expected": "> 0",
        "Status": "PASS" if len(df_retrieval_eval_valid_only) > 0 else "FAIL",
        "Interpretation": "This subset is used for retrieval metrics."
    }
])

safety_style = professional_table(
    gold_audit_safety,
    caption="Table 9.13 — Gold Audit Safety Checks",
    width="100%",
    font_size="12px"
)

safety_style = apply_style_compatible(
    safety_style,
    color_status,
    subset=["Status"]
)

display(safety_style)

# ---------------------------------------------------------
# 9) Preview valid legal gold questions
# ---------------------------------------------------------

legal_preview_cols = [
    "question",
    "gold_article_numbers",
    "gold_article_title",
    "gold_validation_status",
    "retrieval_eval_level"
]

legal_preview_cols = [c for c in legal_preview_cols if c in df_legal_valid_gold.columns]

legal_preview = df_legal_valid_gold[legal_preview_cols].head(12).copy()

if "question" in legal_preview.columns:
    legal_preview["question"] = legal_preview["question"].apply(lambda x: short_text(x, 130))

if "gold_article_title" in legal_preview.columns:
    legal_preview["gold_article_title"] = legal_preview["gold_article_title"].apply(lambda x: short_text(x, 130))

legal_style = professional_table(
    legal_preview,
    caption="Table 9.14 — Valid Manual Legal Questions for Retrieval Evaluation",
    width="100%",
    font_size="11.5px"
)

rtl_cols = [c for c in ["question", "gold_article_title"] if c in legal_preview.columns]

if rtl_cols:
    legal_style = legal_style.set_properties(
        subset=rtl_cols,
        **{
            "text-align": "right",
            "direction": "rtl",
            "line-height": "1.7"
        }
    )

display(legal_style)

# ---------------------------------------------------------
# 10) Assertions and update global df_eval
# ---------------------------------------------------------

assert (
    df_retrieval_eval_valid_only[
        df_retrieval_eval_valid_only["eval_type"].astype(str).eq("legal_article_retrieval")
    ]["has_valid_gold_article"].all()
), "Some legal retrieval-evaluation rows still lack gold article numbers."

assert int(df_retrieval_eval_valid_only["eval_type"].astype(str).eq("out_of_scope").sum()) == 0, (
    "Out-of-scope records should not be included in retrieval-evaluable subset."
)

df_eval = df_eval_reviewed.copy().reset_index(drop=True)

# ---------------------------------------------------------
# 11) Final message
# ---------------------------------------------------------

display(Markdown(f"""
<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#E2F0D9;
    border-right:5px solid #375623;
    padding:14px;
    margin:12px 0;
    border-radius:6px;
">

<b>Stage 09.6 completed successfully.</b><br>

إجمالي سجلات التقييم: <b>{len(df_eval_reviewed)}</b><br>
سجلات FAQ الصالحة لتقييم الاسترجاع: <b>{int(faq_mask.sum())}</b><br>
الأسئلة القانونية ذات المراجع الذهبية الصحيحة: <b>{len(df_legal_valid_gold)}</b><br>
الأسئلة القانونية بدون مرجع ذهبي: <b>{len(df_legal_missing_gold)}</b><br>
أسئلة خارج النطاق: <b>{int(oos_mask.sum())}</b><br>
السجلات النهائية الصالحة لحساب مقاييس الاسترجاع: <b>{len(df_retrieval_eval_valid_only)}</b><br>

</div>
"""))

print("Stage 09.6 completed successfully.")
print("All evaluation records:", len(df_eval_reviewed))
print("FAQ retrieval records:", int(faq_mask.sum()))
print("Legal records with valid gold:", len(df_legal_valid_gold))
print("Legal records missing gold:", len(df_legal_missing_gold))
print("Out-of-scope records:", int(oos_mask.sum()))
print("Final retrieval-evaluable records:", len(df_retrieval_eval_valid_only))
print("Saved reviewed evaluation dataset:", reviewed_eval_csv)
print("Saved valid retrieval subset:", retrieval_valid_csv)


<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#F8FBFD;
    border-right:5px solid #1F4E78;
    padding:14px;
    margin:12px 0;
    border-radius:6px;
">
<h3 style="color:#1F4E78; margin-top:0;">
Stage 09.6 — Evaluation Dataset Gold Audit
</h3>
<p>
تمت إضافة أعلام تدقيق إلى داتا التقييم. أي سؤال قانوني بدون رقم مادة ذهبي لا يدخل في حساب 
Recall أو MRR أو nDCG، بل يبقى محفوظًا في ملف مراجعة منفصل.
</p>
</div>


Metric,Count,Interpretation
All evaluation records,160,"Complete evaluation dataset including FAQ, legal, and OOS."
FAQ retrieval records,121,Valid for document-level FAQ retrieval evaluation.
Legal records with valid gold articles,24,Valid for article-level legal retrieval evaluation.
Legal records missing gold articles,0,"Kept for qualitative review, excluded from article-level metrics."
Out-of-scope records,15,"Used for guardrail/router tests, not retrieval relevance metrics."
Final retrieval-evaluable records,145,Recommended subset for Stage 03 retrieval experiments.


Safety Check,Value,Expected,Status,Interpretation
Legal retrieval rows missing gold articles,0,0,PASS,Legal retrieval rows should have valid gold article numbers.
Retrieval-evaluable OOS records,0,0,PASS,Out-of-scope records should be excluded from retrieval metrics.
Final retrieval-evaluable records,145,> 0,PASS,This subset is used for retrieval metrics.


question,gold_article_numbers,gold_article_title,gold_validation_status,retrieval_eval_level
كم مدة فترة التجربة في نظام العمل السعودي؟,53,المادة الثالثة والخمسون :,valid_gold_article,legal_article
هل يجوز وضع العامل تحت التجربة أكثر من مرة لدى نفس صاحب العمل؟,54,المادة الرابعة والخمسون:,valid_gold_article,legal_article
ما الحد الأقصى لساعات العمل اليومية في نظام العمل؟,98,المادة الثامنة والتسعون:,valid_gold_article,legal_article
ما الحد الأقصى لساعات العمل الأسبوعية؟,98,المادة الثامنة والتسعون:,valid_gold_article,legal_article
هل تختلف ساعات العمل في شهر رمضان؟,98,المادة الثامنة والتسعون:,valid_gold_article,legal_article
ما مدة الإجازة السنوية المستحقة للعامل؟,109,المادة التاسعة بعد المائة:,valid_gold_article,legal_article
متى تزيد الإجازة السنوية إلى ثلاثين يومًا؟,109,المادة التاسعة بعد المائة:,valid_gold_article,legal_article
هل يستحق العامل أجرًا عن أيام الإجازة غير المستخدمة عند انتهاء العمل؟,111,المادة الحادية عشرة بعد المائة:,valid_gold_article,legal_article
هل يستحق العامل إجازة عند الزواج أو وفاة أحد الأقارب؟,113,المادة الثالثة عشرة بعد المائة :,valid_gold_article,legal_article
هل للعامل الحق في إجازة مرضية؟,117,المادة السابعة عشرة بعد المائة:,valid_gold_article,legal_article



<div dir="rtl" style="
    text-align:right;
    line-height:2;
    font-size:15px;
    font-family:Arial, Tahoma, sans-serif;
    background-color:#E2F0D9;
    border-right:5px solid #375623;
    padding:14px;
    margin:12px 0;
    border-radius:6px;
">

<b>Stage 09.6 completed successfully.</b><br>

إجمالي سجلات التقييم: <b>160</b><br>
سجلات FAQ الصالحة لتقييم الاسترجاع: <b>121</b><br>
الأسئلة القانونية ذات المراجع الذهبية الصحيحة: <b>24</b><br>
الأسئلة القانونية بدون مرجع ذهبي: <b>0</b><br>
أسئلة خارج النطاق: <b>15</b><br>
السجلات النهائية الصالحة لحساب مقاييس الاسترجاع: <b>145</b><br>

</div>


Stage 09.6 completed successfully.
All evaluation records: 160
FAQ retrieval records: 121
Legal records with valid gold: 24
Legal records missing gold: 0
Out-of-scope records: 15
Final retrieval-evaluable records: 145
Saved reviewed evaluation dataset: saudi_labor_law_voice_agent_project\03_final\rag_evaluation_dataset.csv
Saved valid retrieval subset: saudi_labor_law_voice_agent_project\03_final\rag_retrieval_evaluation_dataset_valid_only.csv


In [26]:
# ---------------------------------------------------------
# 10) Academic minimum-size checks
# ---------------------------------------------------------

assert legal_count >= 20, "Legal evaluation questions should be at least 20 for stronger academic evaluation."
assert oos_count >= 10, "Out-of-scope questions should be at least 10 for reliable refusal testing."
assert faq_count > 0, "FAQ evaluation questions must exist."

if "normalize_eval_question_for_dedup" not in globals():
    def normalize_eval_question_for_dedup(text):
        text = "" if text is None else str(text)
        text = text.replace("\n", " ").replace("\r", " ")
        text = re.sub(r"\s+", " ", text).strip()
        return text

remaining_duplicate_questions = int(
    df_eval["question"]
    .fillna("")
    .astype(str)
    .apply(normalize_eval_question_for_dedup)
    .duplicated()
    .sum()
)

assert remaining_duplicate_questions == 0, "Duplicate evaluation questions found after deduplication."

allowed_eval_types = {
    "faq_retrieval",
    "legal_article_retrieval",
    "out_of_scope"
}

unknown_eval_types = set(df_eval["eval_type"].dropna().unique()) - allowed_eval_types
assert len(unknown_eval_types) == 0, f"Unknown eval types found: {unknown_eval_types}"

print("Academic enrichment checks passed successfully.")
print("Remaining duplicate questions:", remaining_duplicate_questions)


Academic enrichment checks passed successfully.
Remaining duplicate questions: 0


## Stage 10 — Structural Legal Chunking

This stage builds the **legal-structure-aware chunking strategy** used in the retrieval experiments.

The main methodological decision is that every active legal article is preserved as a coherent retrieval unit whenever possible. This is important because legal meaning is usually attached to complete article boundaries, not to arbitrary text windows. For FAQ or non-legal records, controlled word-based splitting is used only when a record is too long.

**Input requirements before running this stage**

- `df_kb_experiment`
- `df_faq_evaluation`
- `CHUNKS_DIR`
- `clean_basic_text`

**Main output**

- `04_chunks/rag_chunks_structural_legal_experiment.csv`

In [27]:

# =========================================================
# Stage 10 - Structural Legal Chunking
# تقسيم هيكلي يحافظ على حدود المواد القانونية
# =========================================================

from pathlib import Path
from IPython.display import display, Markdown
import pandas as pd
import numpy as np
import re

# ---------------------------------------------------------
# 1) Directory and dependency safety
# ---------------------------------------------------------

if "PROJECT_DIR" not in globals():
    PROJECT_DIR = Path("saudi_labor_law_voice_agent_project")
if "CHUNKS_DIR" not in globals():
    CHUNKS_DIR = PROJECT_DIR / "04_chunks"
if "REPORTS_DIR" not in globals():
    REPORTS_DIR = PROJECT_DIR / "05_reports"

CHUNKS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

if "clean_basic_text" not in globals():
    def clean_basic_text(value):
        """Safe whitespace cleaner used by chunking stages."""
        if value is None:
            return ""
        try:
            if pd.isna(value):
                return ""
        except Exception:
            pass
        value = str(value).replace("\r", " ").replace("\n", " ")
        value = re.sub(r"\s+", " ", value).strip()
        return value

required_stage10_objects = {
    "df_kb_experiment": pd.DataFrame,
    "df_faq_evaluation": pd.DataFrame,
}

missing_stage10_objects = [
    name for name, expected_type in required_stage10_objects.items()
    if name not in globals() or not isinstance(globals()[name], expected_type)
]

if missing_stage10_objects:
    raise NameError(
        "Missing required objects before Stage 10: "
        + ", ".join(missing_stage10_objects)
        + ". Run the previous data preparation stages first."
    )

assert len(df_kb_experiment) > 0, "df_kb_experiment is empty. Stage 10 cannot build chunks."
assert len(df_faq_evaluation) >= 0, "df_faq_evaluation must exist before Stage 10."

required_kb_columns = ["text_for_indexing", "source_type"]
missing_kb_columns = [c for c in required_kb_columns if c not in df_kb_experiment.columns]
if missing_kb_columns:
    raise KeyError("df_kb_experiment is missing required columns: " + ", ".join(missing_kb_columns))

# Ensure stable identifier columns exist.
df_kb_experiment = df_kb_experiment.copy().reset_index(drop=True)
if "document_unit_id" not in df_kb_experiment.columns:
    df_kb_experiment["document_unit_id"] = [f"DOC_{i+1:06d}" for i in range(len(df_kb_experiment))]
if "record_id" not in df_kb_experiment.columns:
    df_kb_experiment["record_id"] = df_kb_experiment["document_unit_id"].astype(str)

# ---------------------------------------------------------
# 2) Display helpers used in Stages 10-13
# ---------------------------------------------------------

def _safe_get(row, column, default=""):
    return row[column] if column in row.index and not pd.isna(row[column]) else default


def hide_index_safe(styler):
    try:
        return styler.hide(axis="index")
    except Exception:
        try:
            return styler.hide_index()
        except Exception:
            return styler


def _status_style(value):
    value = str(value)
    if value == "PASS":
        return "background-color:#E2F0D9; color:#375623; font-weight:bold;"
    if value == "WARN":
        return "background-color:#FFF2CC; color:#7F6000; font-weight:bold;"
    if value == "FAIL":
        return "background-color:#FCE4D6; color:#9C0006; font-weight:bold;"
    if value == "INFO":
        return "background-color:#DDEBF7; color:#1F4E78; font-weight:bold;"
    return ""


def professional_table(df, caption="", status_columns=("status",), width="100%", font_size="12px"):
    """Reusable professional table style compatible with multiple pandas versions."""
    styler = (
        df.style
        .set_caption(caption)
        .set_table_styles([
            {"selector": "caption", "props": [
                ("caption-side", "top"), ("font-size", "17px"), ("font-weight", "bold"),
                ("color", "#1F4E78"), ("text-align", "center"), ("padding", "10px")
            ]},
            {"selector": "th", "props": [
                ("background-color", "#1F4E78"), ("color", "white"), ("font-weight", "bold"),
                ("text-align", "center"), ("border", "1px solid #D9E2F3"),
                ("padding", "8px"), ("white-space", "normal")
            ]},
            {"selector": "td", "props": [
                ("text-align", "center"), ("border", "1px solid #D9E2F3"),
                ("padding", "7px"), ("vertical-align", "middle"), ("white-space", "normal")
            ]},
            {"selector": "tbody tr:nth-child(even)", "props": [("background-color", "#F8FBFD")]},
            {"selector": "tbody tr:nth-child(odd)", "props": [("background-color", "#FFFFFF")]},
            {"selector": "table", "props": [
                ("border-collapse", "collapse"), ("width", width),
                ("font-family", "Arial, Tahoma, sans-serif"), ("font-size", font_size),
                ("table-layout", "fixed"), ("margin", "12px 0")
            ]},
        ])
    )
    for col in status_columns:
        if col in df.columns:
            if hasattr(styler, "map"):
                styler = styler.map(_status_style, subset=[col])
            else:
                styler = styler.applymap(_status_style, subset=[col])
    return hide_index_safe(styler)


def _write_dataframe(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.suffix.lower() == ".csv":
        df.to_csv(path, index=False, encoding="utf-8-sig")
    elif path.suffix.lower() in [".xlsx", ".xls"]:
        df.to_excel(path, index=False)
    else:
        raise ValueError(f"Unsupported output format: {path}")
    return path

# ---------------------------------------------------------
# 3) Structural chunking helpers
# ---------------------------------------------------------

STRUCTURAL_MAX_WORDS = 350
STRUCTURAL_OVERLAP_WORDS = 50


def word_chunks(text, chunk_size=350, overlap=50):
    """Word-based fallback splitter with overlap."""
    text = clean_basic_text(text)
    words = text.split()

    if not words:
        return []

    if len(words) <= chunk_size:
        return [text]

    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size.")

    chunks = []
    start = 0

    while start < len(words):
        end = start + chunk_size
        chunks.append(" ".join(words[start:end]))
        if end >= len(words):
            break
        start = max(0, end - overlap)

    return chunks


def build_structural_context_header(row):
    """Build a compact legal context header: الباب، الفصل، المادة."""
    parts = []

    bab_label = clean_basic_text(_safe_get(row, "bab_label", ""))
    bab_title = clean_basic_text(_safe_get(row, "bab_title", ""))
    fasl = clean_basic_text(_safe_get(row, "fasl", ""))
    article_number_label = clean_basic_text(_safe_get(row, "article_number_label", ""))
    article_number = clean_basic_text(_safe_get(row, "article_number", ""))

    if bab_label or bab_title:
        parts.append(" - ".join([x for x in [bab_label, bab_title] if x]))
    if fasl:
        parts.append(fasl)
    if article_number_label:
        parts.append("المادة " + article_number_label)
    elif article_number:
        parts.append("المادة " + article_number)

    return " | ".join(parts)


def _is_repealed_status(value):
    value = clean_basic_text(value).lower()
    return value in {"repealed", "cancelled", "canceled", "ملغى", "ملغاة", "ملغي"}


def _build_chunk_record(row, chunk_id, strategy, chunk_text, chunk_order):
    chunk_text = clean_basic_text(chunk_text)
    return {
        "chunk_id": chunk_id,
        "chunking_strategy": strategy,
        "chunk_text": chunk_text,
        "chunk_word_count": len(chunk_text.split()),
        "chunk_char_count": len(chunk_text),
        "chunk_order": int(chunk_order),
        "parent_document_id": clean_basic_text(_safe_get(row, "document_unit_id", _safe_get(row, "record_id", ""))),
        "record_id": clean_basic_text(_safe_get(row, "record_id", _safe_get(row, "document_unit_id", ""))),
        "source_type": clean_basic_text(_safe_get(row, "source_type", "")),
        "source_name": clean_basic_text(_safe_get(row, "source_name", "")),
        "legal_category": clean_basic_text(_safe_get(row, "legal_category", "")),
        "rag_usage": clean_basic_text(_safe_get(row, "rag_usage", "")),
        "bab_label": clean_basic_text(_safe_get(row, "bab_label", "")),
        "bab_title": clean_basic_text(_safe_get(row, "bab_title", "")),
        "fasl": clean_basic_text(_safe_get(row, "fasl", "")),
        "article_title": clean_basic_text(_safe_get(row, "article_title", "")),
        "article_number": clean_basic_text(_safe_get(row, "article_number", "")),
        "article_number_label": clean_basic_text(_safe_get(row, "article_number_label", "")),
        "article_number_int": _safe_get(row, "article_number_int", ""),
        "article_status": clean_basic_text(_safe_get(row, "article_status", "")),
        "question": clean_basic_text(_safe_get(row, "question", "")),
        "answer": clean_basic_text(_safe_get(row, "answer", "")),
        "beneficiaries": clean_basic_text(_safe_get(row, "beneficiaries", "")),
        "sector": clean_basic_text(_safe_get(row, "sector", "")),
        "subsite": clean_basic_text(_safe_get(row, "subsite", "")),
        "source_url": clean_basic_text(_safe_get(row, "source_url", "")),
    }


def build_structural_chunks(df_kb, prefix="STRUCT"):
    """Build structural legal chunks from the experimental KB only."""
    rows = []
    df_kb = df_kb.copy().reset_index(drop=True)

    for _, row in df_kb.iterrows():
        source_type = clean_basic_text(_safe_get(row, "source_type", ""))
        is_legal = source_type == "legal_article"
        base_text = clean_basic_text(_safe_get(row, "text_for_indexing", ""))

        if not base_text:
            continue

        if is_legal:
            # Legal articles remain coherent units. They are not forced into the fixed word limit.
            header = build_structural_context_header(row)
            pieces = [(header + "\n" + base_text).strip() if header else base_text]
        else:
            pieces = word_chunks(
                base_text,
                chunk_size=STRUCTURAL_MAX_WORDS,
                overlap=STRUCTURAL_OVERLAP_WORDS,
            )

        for chunk_order, chunk_text in enumerate(pieces, start=1):
            rows.append(
                _build_chunk_record(
                    row=row,
                    chunk_id=f"{prefix}_{len(rows) + 1:06d}",
                    strategy="structural_legal_chunking",
                    chunk_text=chunk_text,
                    chunk_order=chunk_order,
                )
            )

    return pd.DataFrame(rows)

# ---------------------------------------------------------
# 4) Build and save structural chunks
# ---------------------------------------------------------

df_structural_chunks = build_structural_chunks(df_kb_experiment, prefix="STRUCT")

structural_output_csv = CHUNKS_DIR / "rag_chunks_structural_legal_experiment.csv"
structural_output_xlsx = CHUNKS_DIR / "rag_chunks_structural_legal_experiment.xlsx"
_write_dataframe(df_structural_chunks, structural_output_csv)
_write_dataframe(df_structural_chunks, structural_output_xlsx)

# ---------------------------------------------------------
# 5) Critical safety checks
# ---------------------------------------------------------

structural_empty_chunks = int((df_structural_chunks["chunk_text"].fillna("").astype(str).str.strip() == "").sum())
structural_missing_parent = int((df_structural_chunks["parent_document_id"].fillna("").astype(str).str.strip() == "").sum())
structural_duplicate_ids = int(df_structural_chunks["chunk_id"].duplicated().sum())
structural_repealed_chunks = int(
    df_structural_chunks.apply(
        lambda r: r.get("source_type", "") == "legal_article" and _is_repealed_status(r.get("article_status", "")),
        axis=1,
    ).sum()
)

faq_eval_ids = set(df_faq_evaluation.get("document_unit_id", pd.Series(dtype=str)).fillna("").astype(str))
structural_faq_chunk_ids = set(
    df_structural_chunks.loc[
        df_structural_chunks["source_type"].astype(str).eq("faq"),
        "parent_document_id"
    ].fillna("").astype(str)
)
structural_faq_eval_leakage = len(structural_faq_chunk_ids & faq_eval_ids)

structural_legal_duplicate_parent_chunks = int(
    df_structural_chunks.loc[df_structural_chunks["source_type"].astype(str).eq("legal_article"), "parent_document_id"]
    .duplicated()
    .sum()
)

assert structural_empty_chunks == 0, "Stage 10 failed: empty structural chunks were created."
assert structural_missing_parent == 0, "Stage 10 failed: some chunks are missing parent_document_id."
assert structural_duplicate_ids == 0, "Stage 10 failed: duplicated structural chunk_id values were found."
assert structural_repealed_chunks == 0, "Stage 10 failed: repealed legal articles found inside structural chunks."
assert structural_faq_eval_leakage == 0, "Stage 10 failed: FAQ evaluation records leaked into structural chunks."
assert structural_legal_duplicate_parent_chunks == 0, "Stage 10 failed: structural legal articles should remain one chunk per legal parent document."

# ---------------------------------------------------------
# 6) Professional summary table
# ---------------------------------------------------------

structural_summary = pd.DataFrame([
    ["Total structural chunks", len(df_structural_chunks), "PASS" if len(df_structural_chunks) > 0 else "FAIL"],
    ["Legal chunks", int((df_structural_chunks["source_type"] == "legal_article").sum()), "INFO"],
    ["FAQ chunks", int((df_structural_chunks["source_type"] == "faq").sum()), "INFO"],
    ["Average words per chunk", round(float(df_structural_chunks["chunk_word_count"].mean()), 2), "INFO"],
    ["Maximum words per chunk", int(df_structural_chunks["chunk_word_count"].max()), "INFO"],
    ["Empty chunks", structural_empty_chunks, "PASS"],
    ["Repealed legal chunks", structural_repealed_chunks, "PASS"],
    ["FAQ evaluation leakage", structural_faq_eval_leakage, "PASS"],
    ["Output CSV", str(structural_output_csv), "PASS" if structural_output_csv.exists() else "FAIL"],
], columns=["check", "value", "status"])

display(professional_table(structural_summary, "Stage 10 — Structural Chunking Summary"))

display(Markdown(f"""
<div dir="rtl" style="text-align:right; line-height:2; font-family:Arial, Tahoma, sans-serif; background-color:#F8FBFD; border-right:5px solid #1F4E78; padding:16px; border-radius:6px;">
<b>Stage 10 Result:</b><br>
تم إنشاء ملف التقسيم الهيكلي بنجاح. يحافظ هذا التقسيم على المادة القانونية كوحدة كاملة، ويمنع دخول المواد الملغاة أو سجلات FAQ المحجوزة للتقييم داخل Chunks الخاصة بالتجارب.
</div>
"""))


check,value,status
Total structural chunks,490,PASS
Legal chunks,211,INFO
FAQ chunks,279,INFO
Average words per chunk,92.580000,INFO
Maximum words per chunk,656,INFO
Empty chunks,0,PASS
Repealed legal chunks,0,PASS
FAQ evaluation leakage,0,PASS
Output CSV,saudi_labor_law_voice_agent_project\04_chunks\rag_chunks_structural_legal_experiment.csv,PASS



<div dir="rtl" style="text-align:right; line-height:2; font-family:Arial, Tahoma, sans-serif; background-color:#F8FBFD; border-right:5px solid #1F4E78; padding:16px; border-radius:6px;">
<b>Stage 10 Result:</b><br>
تم إنشاء ملف التقسيم الهيكلي بنجاح. يحافظ هذا التقسيم على المادة القانونية كوحدة كاملة، ويمنع دخول المواد الملغاة أو سجلات FAQ المحجوزة للتقييم داخل Chunks الخاصة بالتجارب.
</div>


## Stage 11 — Fixed-size Chunking Baseline

This stage builds the **baseline chunking strategy** used for comparison with the structural legal chunking strategy.

The baseline uses a fixed word window with overlap. It is useful as a neutral comparison point because it does not use the legal structure of the document. The retrieval notebook can later compare both strategies using Recall@k, MRR, nDCG@5, and latency.

**Configuration**

- Chunk size: `500` words
- Overlap: `80` words

**Main output**

- `04_chunks/rag_chunks_fixed_size_experiment.csv`

In [28]:

# =========================================================
# Stage 11 - Fixed-size Chunking Baseline
# خط أساس للمقارنة مع التقسيم الهيكلي
# =========================================================

from pathlib import Path
from IPython.display import display, Markdown
import pandas as pd
import re

# ---------------------------------------------------------
# 1) Preconditions
# ---------------------------------------------------------

if "PROJECT_DIR" not in globals():
    PROJECT_DIR = Path("saudi_labor_law_voice_agent_project")
if "CHUNKS_DIR" not in globals():
    CHUNKS_DIR = PROJECT_DIR / "04_chunks"
if "REPORTS_DIR" not in globals():
    REPORTS_DIR = PROJECT_DIR / "05_reports"

CHUNKS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

required_stage11_objects = {
    "df_kb_experiment": pd.DataFrame,
    "df_faq_evaluation": pd.DataFrame,
}

missing_stage11_objects = [
    name for name, expected_type in required_stage11_objects.items()
    if name not in globals() or not isinstance(globals()[name], expected_type)
]

if missing_stage11_objects:
    raise NameError(
        "Missing required objects before Stage 11: "
        + ", ".join(missing_stage11_objects)
        + ". Run Stages 01-10 first."
    )

if "clean_basic_text" not in globals():
    def clean_basic_text(value):
        if value is None:
            return ""
        try:
            if pd.isna(value):
                return ""
        except Exception:
            pass
        return re.sub(r"\s+", " ", str(value).replace("\n", " ").replace("\r", " ")).strip()

assert "text_for_indexing" in df_kb_experiment.columns, "df_kb_experiment must contain text_for_indexing."
assert "source_type" in df_kb_experiment.columns, "df_kb_experiment must contain source_type."

# ---------------------------------------------------------
# 2) Fixed-size splitter
# ---------------------------------------------------------

CHUNK_SIZE_WORDS = 500
CHUNK_OVERLAP_WORDS = 80


def fixed_word_chunks(text, chunk_size=500, overlap=80):
    text = clean_basic_text(text)
    words = text.split()

    if not words:
        return []

    if len(words) <= chunk_size:
        return [text]

    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size.")

    chunks = []
    start = 0

    while start < len(words):
        end = start + chunk_size
        chunks.append(" ".join(words[start:end]))
        if end >= len(words):
            break
        start = max(0, end - overlap)

    return chunks


def build_fixed_chunks(df_kb, chunk_size=500, overlap=80, prefix="FIXED"):
    rows = []
    df_kb = df_kb.copy().reset_index(drop=True)

    if "document_unit_id" not in df_kb.columns:
        df_kb["document_unit_id"] = [f"DOC_{i+1:06d}" for i in range(len(df_kb))]
    if "record_id" not in df_kb.columns:
        df_kb["record_id"] = df_kb["document_unit_id"].astype(str)

    for _, row in df_kb.iterrows():
        pieces = fixed_word_chunks(
            _safe_get(row, "text_for_indexing", ""),
            chunk_size=chunk_size,
            overlap=overlap,
        )

        for chunk_order, chunk_text in enumerate(pieces, start=1):
            rows.append(
                _build_chunk_record(
                    row=row,
                    chunk_id=f"{prefix}_{len(rows) + 1:06d}",
                    strategy="fixed_size_chunking_baseline",
                    chunk_text=chunk_text,
                    chunk_order=chunk_order,
                )
            )

    return pd.DataFrame(rows)

# ---------------------------------------------------------
# 3) Build and save fixed-size chunks
# ---------------------------------------------------------

df_fixed_chunks = build_fixed_chunks(
    df_kb_experiment,
    chunk_size=CHUNK_SIZE_WORDS,
    overlap=CHUNK_OVERLAP_WORDS,
    prefix="FIXED",
)

fixed_output_csv = CHUNKS_DIR / "rag_chunks_fixed_size_experiment.csv"
fixed_output_xlsx = CHUNKS_DIR / "rag_chunks_fixed_size_experiment.xlsx"
_write_dataframe(df_fixed_chunks, fixed_output_csv)
_write_dataframe(df_fixed_chunks, fixed_output_xlsx)

# ---------------------------------------------------------
# 4) Critical safety checks
# ---------------------------------------------------------

fixed_empty_chunks = int((df_fixed_chunks["chunk_text"].fillna("").astype(str).str.strip() == "").sum())
fixed_missing_parent = int((df_fixed_chunks["parent_document_id"].fillna("").astype(str).str.strip() == "").sum())
fixed_duplicate_ids = int(df_fixed_chunks["chunk_id"].duplicated().sum())
fixed_repealed_chunks = int(
    df_fixed_chunks.apply(
        lambda r: r.get("source_type", "") == "legal_article" and _is_repealed_status(r.get("article_status", "")),
        axis=1,
    ).sum()
)

faq_eval_ids = set(df_faq_evaluation.get("document_unit_id", pd.Series(dtype=str)).fillna("").astype(str))
fixed_faq_chunk_ids = set(
    df_fixed_chunks.loc[
        df_fixed_chunks["source_type"].astype(str).eq("faq"),
        "parent_document_id"
    ].fillna("").astype(str)
)
fixed_faq_eval_leakage = len(fixed_faq_chunk_ids & faq_eval_ids)
fixed_max_words = int(df_fixed_chunks["chunk_word_count"].max()) if len(df_fixed_chunks) else 0

assert fixed_empty_chunks == 0, "Stage 11 failed: empty fixed-size chunks were created."
assert fixed_missing_parent == 0, "Stage 11 failed: some chunks are missing parent_document_id."
assert fixed_duplicate_ids == 0, "Stage 11 failed: duplicated fixed-size chunk_id values were found."
assert fixed_repealed_chunks == 0, "Stage 11 failed: repealed legal articles found inside fixed-size chunks."
assert fixed_faq_eval_leakage == 0, "Stage 11 failed: FAQ evaluation records leaked into fixed-size chunks."
assert fixed_max_words <= CHUNK_SIZE_WORDS, "Stage 11 failed: fixed-size chunks exceeded the configured chunk size."

# ---------------------------------------------------------
# 5) Professional summary table
# ---------------------------------------------------------

fixed_summary = pd.DataFrame([
    ["Total fixed-size chunks", len(df_fixed_chunks), "PASS" if len(df_fixed_chunks) > 0 else "FAIL"],
    ["Legal chunks", int((df_fixed_chunks["source_type"] == "legal_article").sum()), "INFO"],
    ["FAQ chunks", int((df_fixed_chunks["source_type"] == "faq").sum()), "INFO"],
    ["Average words per chunk", round(float(df_fixed_chunks["chunk_word_count"].mean()), 2), "INFO"],
    ["Maximum words per chunk", fixed_max_words, "PASS" if fixed_max_words <= CHUNK_SIZE_WORDS else "FAIL"],
    ["Empty chunks", fixed_empty_chunks, "PASS"],
    ["Repealed legal chunks", fixed_repealed_chunks, "PASS"],
    ["FAQ evaluation leakage", fixed_faq_eval_leakage, "PASS"],
    ["Output CSV", str(fixed_output_csv), "PASS" if fixed_output_csv.exists() else "FAIL"],
], columns=["check", "value", "status"])

display(professional_table(fixed_summary, "Stage 11 — Fixed-size Chunking Baseline Summary"))

display(Markdown(f"""
<div dir="rtl" style="text-align:right; line-height:2; font-family:Arial, Tahoma, sans-serif; background-color:#F8FBFD; border-right:5px solid #1F4E78; padding:16px; border-radius:6px;">
<b>Stage 11 Result:</b><br>
تم إنشاء خط الأساس Fixed-size Chunking بنجاح باستخدام حجم {CHUNK_SIZE_WORDS} كلمة وتداخل {CHUNK_OVERLAP_WORDS} كلمة. هذا الملف جاهز للمقارنة مع التقسيم الهيكلي في Notebook 03.
</div>
"""))


check,value,status
Total fixed-size chunks,490,PASS
Legal chunks,212,INFO
FAQ chunks,278,INFO
Average words per chunk,87.200000,INFO
Maximum words per chunk,500,PASS
Empty chunks,0,PASS
Repealed legal chunks,0,PASS
FAQ evaluation leakage,0,PASS
Output CSV,saudi_labor_law_voice_agent_project\04_chunks\rag_chunks_fixed_size_experiment.csv,PASS



<div dir="rtl" style="text-align:right; line-height:2; font-family:Arial, Tahoma, sans-serif; background-color:#F8FBFD; border-right:5px solid #1F4E78; padding:16px; border-radius:6px;">
<b>Stage 11 Result:</b><br>
تم إنشاء خط الأساس Fixed-size Chunking بنجاح باستخدام حجم 500 كلمة وتداخل 80 كلمة. هذا الملف جاهز للمقارنة مع التقسيم الهيكلي في Notebook 03.
</div>


## Stage 12 — Chunk Quality Report

This stage compares the two chunking strategies before moving to the retrieval notebook.

The report validates the following points:

1. Number of chunks generated by each strategy.
2. Mean, median, and maximum chunk length.
3. No empty chunks.
4. No duplicated `chunk_id` values.
5. No missing `parent_document_id` values.
6. No repealed legal articles inside chunk files.
7. No FAQ evaluation leakage inside chunk files.
8. Structural legal chunking preserves one chunk per legal parent document.

**Main outputs**

- `05_reports/chunk_quality_report.csv`
- `05_reports/chunk_quality_report.xlsx`
- `05_reports/chunk_length_summary.csv`
- `05_reports/chunk_length_summary.xlsx`

In [29]:

# =========================================================
# Stage 12 - Chunk Quality Report
# تقرير جودة التقسيم ومقارنة Structural vs Fixed-size
# =========================================================

from pathlib import Path
from IPython.display import display, Markdown
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1) Load chunk data if the variables are not already in memory
# ---------------------------------------------------------

structural_chunk_path = CHUNKS_DIR / "rag_chunks_structural_legal_experiment.csv"
fixed_chunk_path = CHUNKS_DIR / "rag_chunks_fixed_size_experiment.csv"

if "df_structural_chunks" not in globals():
    assert structural_chunk_path.exists(), "Structural chunk file is missing. Run Stage 10 first."
    df_structural_chunks = pd.read_csv(structural_chunk_path)

if "df_fixed_chunks" not in globals():
    assert fixed_chunk_path.exists(), "Fixed-size chunk file is missing. Run Stage 11 first."
    df_fixed_chunks = pd.read_csv(fixed_chunk_path)

assert len(df_structural_chunks) > 0, "Structural chunks are empty."
assert len(df_fixed_chunks) > 0, "Fixed-size chunks are empty."

# ---------------------------------------------------------
# 2) Normalize length columns
# ---------------------------------------------------------

def add_chunk_length_columns(df):
    df = df.copy()
    df["chunk_text"] = df["chunk_text"].fillna("").astype(str).apply(clean_basic_text)
    df["chunk_word_len"] = df["chunk_text"].str.split().apply(len)
    df["chunk_char_len"] = df["chunk_text"].str.len()
    return df


df_structural_chunks = add_chunk_length_columns(df_structural_chunks)
df_fixed_chunks = add_chunk_length_columns(df_fixed_chunks)

# ---------------------------------------------------------
# 3) Quality check helpers
# ---------------------------------------------------------

quality_rows = []
length_summary_rows = []


def _pass_fail(condition):
    return "PASS" if bool(condition) else "FAIL"


def add_quality_row(strategy, check, value, status, interpretation):
    quality_rows.append({
        "strategy": strategy,
        "check": check,
        "value": value,
        "status": status,
        "interpretation": interpretation,
    })


def evaluate_chunks(strategy, df_chunks, max_allowed_words=None, structural_legal=False):
    empty_chunks = int((df_chunks["chunk_text"].fillna("").astype(str).str.strip() == "").sum())
    missing_parent = int((df_chunks["parent_document_id"].fillna("").astype(str).str.strip() == "").sum())
    duplicated_chunk_ids = int(df_chunks["chunk_id"].duplicated().sum())
    repealed_chunks = int(
        df_chunks.apply(
            lambda r: r.get("source_type", "") == "legal_article" and _is_repealed_status(r.get("article_status", "")),
            axis=1,
        ).sum()
    )

    faq_eval_ids = set(df_faq_evaluation.get("document_unit_id", pd.Series(dtype=str)).fillna("").astype(str))
    faq_chunk_parent_ids = set(
        df_chunks.loc[
            df_chunks["source_type"].astype(str).eq("faq"),
            "parent_document_id"
        ].fillna("").astype(str)
    )
    faq_eval_leakage = len(faq_chunk_parent_ids & faq_eval_ids)

    total_chunks = int(len(df_chunks))
    mean_words = round(float(df_chunks["chunk_word_len"].mean()), 2) if total_chunks else 0
    median_words = round(float(df_chunks["chunk_word_len"].median()), 2) if total_chunks else 0
    max_words = int(df_chunks["chunk_word_len"].max()) if total_chunks else 0
    min_words = int(df_chunks["chunk_word_len"].min()) if total_chunks else 0

    length_summary_rows.append({
        "strategy": strategy,
        "total_chunks": total_chunks,
        "min_words": min_words,
        "mean_words": mean_words,
        "median_words": median_words,
        "max_words": max_words,
        "total_characters": int(df_chunks["chunk_char_len"].sum()),
    })

    add_quality_row(strategy, "Chunk count", total_chunks, _pass_fail(total_chunks > 0), "The strategy must produce at least one chunk.")
    add_quality_row(strategy, "Empty chunks", empty_chunks, _pass_fail(empty_chunks == 0), "Empty retrieval chunks are not valid for embedding.")
    add_quality_row(strategy, "Missing parent_document_id", missing_parent, _pass_fail(missing_parent == 0), "Every chunk must be traceable to the source document.")
    add_quality_row(strategy, "Duplicated chunk_id", duplicated_chunk_ids, _pass_fail(duplicated_chunk_ids == 0), "Chunk IDs must be unique before vector indexing.")
    add_quality_row(strategy, "Repealed legal chunks", repealed_chunks, _pass_fail(repealed_chunks == 0), "Repealed legal articles must remain archived and not indexed.")
    add_quality_row(strategy, "FAQ evaluation leakage", faq_eval_leakage, _pass_fail(faq_eval_leakage == 0), "FAQ held-out evaluation records must not appear in chunk files.")
    add_quality_row(strategy, "Mean words per chunk", mean_words, "INFO", "Average retrieval granularity.")
    add_quality_row(strategy, "Median words per chunk", median_words, "INFO", "Median retrieval granularity.")
    add_quality_row(strategy, "Maximum words per chunk", max_words, "INFO", "Longest chunk in this strategy.")

    if max_allowed_words is not None:
        add_quality_row(
            strategy,
            "Configured maximum word limit",
            f"{max_words} / {max_allowed_words}",
            _pass_fail(max_words <= max_allowed_words),
            "Fixed-size baseline must respect the configured maximum word size."
        )

    if structural_legal:
        legal_parent_duplicates = int(
            df_chunks.loc[df_chunks["source_type"].astype(str).eq("legal_article"), "parent_document_id"]
            .duplicated()
            .sum()
        )
        add_quality_row(
            strategy,
            "Structural legal article boundary preservation",
            legal_parent_duplicates,
            _pass_fail(legal_parent_duplicates == 0),
            "Structural legal chunking should preserve one chunk per legal parent document."
        )


# ---------------------------------------------------------
# 4) Run checks for both strategies
# ---------------------------------------------------------

evaluate_chunks(
    strategy="Structural Chunking",
    df_chunks=df_structural_chunks,
    max_allowed_words=None,
    structural_legal=True,
)

evaluate_chunks(
    strategy="Fixed-size Chunking Baseline",
    df_chunks=df_fixed_chunks,
    max_allowed_words=CHUNK_SIZE_WORDS,
    structural_legal=False,
)

chunk_quality_report = pd.DataFrame(quality_rows)
chunk_length_summary = pd.DataFrame(length_summary_rows)

# ---------------------------------------------------------
# 5) Save reports
# ---------------------------------------------------------

_write_dataframe(chunk_quality_report, REPORTS_DIR / "chunk_quality_report.csv")
_write_dataframe(chunk_quality_report, REPORTS_DIR / "chunk_quality_report.xlsx")
_write_dataframe(chunk_length_summary, REPORTS_DIR / "chunk_length_summary.csv")
_write_dataframe(chunk_length_summary, REPORTS_DIR / "chunk_length_summary.xlsx")

# ---------------------------------------------------------
# 6) Display professional tables
# ---------------------------------------------------------

display(professional_table(chunk_length_summary, "Stage 12 — Chunk Length Summary", status_columns=()))
display(professional_table(chunk_quality_report, "Stage 12 — Chunk Quality Report"))

critical_failures = chunk_quality_report[chunk_quality_report["status"].eq("FAIL")]
assert critical_failures.empty, "Stage 12 failed: one or more chunk quality checks failed. Review chunk_quality_report."

display(Markdown("""
<div dir="rtl" style="text-align:right; line-height:2; font-family:Arial, Tahoma, sans-serif; background-color:#E2F0D9; border-right:5px solid #375623; padding:16px; border-radius:6px;">
<b>Stage 12 Result:</b><br>
كل فحوصات جودة التقسيم الحرجة نجحت. لا توجد مقاطع فارغة، ولا مواد ملغاة داخل ملفات التقسيم، ولا يوجد تسريب من FAQ evaluation إلى ملفات Chunks.
</div>
"""))


strategy,total_chunks,min_words,mean_words,median_words,max_words,total_characters
Structural Chunking,490,32,92.580000,79.000000,656,259893
Fixed-size Chunking Baseline,490,27,87.200000,73.000000,500,245742


strategy,check,value,status,interpretation
Structural Chunking,Chunk count,490,PASS,The strategy must produce at least one chunk.
Structural Chunking,Empty chunks,0,PASS,Empty retrieval chunks are not valid for embedding.
Structural Chunking,Missing parent_document_id,0,PASS,Every chunk must be traceable to the source document.
Structural Chunking,Duplicated chunk_id,0,PASS,Chunk IDs must be unique before vector indexing.
Structural Chunking,Repealed legal chunks,0,PASS,Repealed legal articles must remain archived and not indexed.
Structural Chunking,FAQ evaluation leakage,0,PASS,FAQ held-out evaluation records must not appear in chunk files.
Structural Chunking,Mean words per chunk,92.580000,INFO,Average retrieval granularity.
Structural Chunking,Median words per chunk,79.000000,INFO,Median retrieval granularity.
Structural Chunking,Maximum words per chunk,656,INFO,Longest chunk in this strategy.
Structural Chunking,Structural legal article boundary preservation,0,PASS,Structural legal chunking should preserve one chunk per legal parent document.



<div dir="rtl" style="text-align:right; line-height:2; font-family:Arial, Tahoma, sans-serif; background-color:#E2F0D9; border-right:5px solid #375623; padding:16px; border-radius:6px;">
<b>Stage 12 Result:</b><br>
كل فحوصات جودة التقسيم الحرجة نجحت. لا توجد مقاطع فارغة، ولا مواد ملغاة داخل ملفات التقسيم، ولا يوجد تسريب من FAQ evaluation إلى ملفات Chunks.
</div>


## Stage 13 — Final Output Manifest and Notebook 03 Readiness

This final stage confirms that all files required by the retrieval notebook are available and valid.

After this stage passes, move to:

`03_RAG_Retrieval_Evaluation_ChromaDB_READY.ipynb`

Notebook 03 should use the following files exactly:

- `03_final/rag_knowledge_base_dataset_experiment_no_leakage_active_law_only.csv`
- `03_final/rag_retrieval_evaluation_dataset_valid_only.csv`
- `04_chunks/rag_chunks_structural_legal_experiment.csv`
- `04_chunks/rag_chunks_fixed_size_experiment.csv`

This stage also saves a final manifest and a compact summary table for academic reporting.

In [30]:

# =========================================================
# Stage 13 - Final Output Manifest / Final Summary
# فحص نهائي لمخرجات Notebook 02 قبل الانتقال إلى Notebook 03
# =========================================================

from pathlib import Path
from IPython.display import display, Markdown
import pandas as pd
import json

# ---------------------------------------------------------
# 1) Directory safety
# ---------------------------------------------------------

if "PROJECT_DIR" not in globals():
    PROJECT_DIR = Path("saudi_labor_law_voice_agent_project")
if "FINAL_DIR" not in globals():
    FINAL_DIR = PROJECT_DIR / "03_final"
if "CHUNKS_DIR" not in globals():
    CHUNKS_DIR = PROJECT_DIR / "04_chunks"
if "REPORTS_DIR" not in globals():
    REPORTS_DIR = PROJECT_DIR / "05_reports"

FINAL_DIR.mkdir(parents=True, exist_ok=True)
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# 2) Ensure required final files are written from in-memory objects when available
# ---------------------------------------------------------

kb_active_path = FINAL_DIR / "rag_knowledge_base_dataset_experiment_no_leakage_active_law_only.csv"
eval_full_path = FINAL_DIR / "rag_evaluation_dataset.csv"
eval_valid_path = FINAL_DIR / "rag_retrieval_evaluation_dataset_valid_only.csv"
structural_path = CHUNKS_DIR / "rag_chunks_structural_legal_experiment.csv"
fixed_path = CHUNKS_DIR / "rag_chunks_fixed_size_experiment.csv"
repealed_archive_path = FINAL_DIR / "legal_articles_repealed_archive_not_indexed.csv"

# Experimental KB
if "df_kb_experiment" in globals() and isinstance(df_kb_experiment, pd.DataFrame):
    _write_dataframe(df_kb_experiment, kb_active_path)
    # Legacy compatibility copy for older retrieval notebooks.
    _write_dataframe(df_kb_experiment, FINAL_DIR / "rag_knowledge_base_dataset_experiment_no_leakage.csv")

# Full evaluation dataset
if "df_eval_reviewed" in globals() and isinstance(df_eval_reviewed, pd.DataFrame):
    df_eval_for_output = df_eval_reviewed.copy()
elif "df_eval" in globals() and isinstance(df_eval, pd.DataFrame):
    df_eval_for_output = df_eval.copy()
else:
    df_eval_for_output = None

if df_eval_for_output is not None:
    _write_dataframe(df_eval_for_output, eval_full_path)

    if "is_retrieval_eval" in df_eval_for_output.columns:
        df_eval_valid_only = df_eval_for_output[df_eval_for_output["is_retrieval_eval"].astype(bool)].copy()
    else:
        # Conservative fallback: exclude out-of-scope questions from retrieval metrics if the eligibility flag is missing.
        df_eval_valid_only = df_eval_for_output[
            ~df_eval_for_output.get("eval_type", pd.Series(dtype=str)).astype(str).eq("out_of_scope")
        ].copy()

    _write_dataframe(df_eval_valid_only, eval_valid_path)

# Repealed legal archive
if "df_articles_repealed" in globals() and isinstance(df_articles_repealed, pd.DataFrame):
    _write_dataframe(df_articles_repealed, repealed_archive_path)

# Chunks are already written in Stage 10 and Stage 11, but save again if variables are in memory.
if "df_structural_chunks" in globals() and isinstance(df_structural_chunks, pd.DataFrame):
    _write_dataframe(df_structural_chunks, structural_path)
if "df_fixed_chunks" in globals() and isinstance(df_fixed_chunks, pd.DataFrame):
    _write_dataframe(df_fixed_chunks, fixed_path)

# ---------------------------------------------------------
# 3) Build final manifest
# ---------------------------------------------------------

required_files = [
    {
        "file_group": "Notebook 03 required input",
        "file_name": "Experimental KB - active law only",
        "file_path": kb_active_path,
        "notebook03_usage": "Knowledge base for fair retrieval evaluation",
    },
    {
        "file_group": "Notebook 03 required input",
        "file_name": "Retrieval evaluation dataset - valid only",
        "file_path": eval_valid_path,
        "notebook03_usage": "Queries with valid gold labels for retrieval metrics",
    },
    {
        "file_group": "Notebook 03 required input",
        "file_name": "Structural chunks",
        "file_path": structural_path,
        "notebook03_usage": "Legal-structure-aware chunking experiment",
    },
    {
        "file_group": "Notebook 03 required input",
        "file_name": "Fixed-size chunks",
        "file_path": fixed_path,
        "notebook03_usage": "Baseline fixed-size chunking experiment",
    },
    {
        "file_group": "Audit output",
        "file_name": "Full evaluation dataset",
        "file_path": eval_full_path,
        "notebook03_usage": "Full evaluation and audit copy",
    },
    {
        "file_group": "Audit output",
        "file_name": "Repealed legal archive",
        "file_path": repealed_archive_path,
        "notebook03_usage": "Archived only; must not be indexed",
    },
]


def _csv_row_count(path):
    path = Path(path)
    if not path.exists() or path.suffix.lower() != ".csv":
        return None
    try:
        return int(len(pd.read_csv(path)))
    except Exception:
        return None

manifest_rows = []
for item in required_files:
    path = Path(item["file_path"])
    exists = path.exists()
    size_kb = round(path.stat().st_size / 1024, 2) if exists else 0
    manifest_rows.append({
        "file_group": item["file_group"],
        "file_name": item["file_name"],
        "file_path": str(path),
        "exists": exists,
        "row_count": _csv_row_count(path),
        "size_kb": size_kb,
        "notebook03_usage": item["notebook03_usage"],
        "status": "PASS" if exists and size_kb > 0 else "FAIL",
    })

final_output_manifest = pd.DataFrame(manifest_rows)

# ---------------------------------------------------------
# 4) Final safety summary
# ---------------------------------------------------------

def _count_rows(path):
    count = _csv_row_count(path)
    return int(count) if count is not None else 0

summary_rows = [
    {
        "category": "Knowledge Base",
        "metric": "Experimental KB records",
        "value": _count_rows(kb_active_path),
        "status": "PASS" if kb_active_path.exists() and _count_rows(kb_active_path) > 0 else "FAIL",
        "interpretation": "Main active-law-only knowledge base for Notebook 03.",
    },
    {
        "category": "Evaluation",
        "metric": "Full evaluation questions",
        "value": _count_rows(eval_full_path),
        "status": "PASS" if eval_full_path.exists() and _count_rows(eval_full_path) > 0 else "FAIL",
        "interpretation": "Full evaluation dataset retained for audit and qualitative analysis.",
    },
    {
        "category": "Evaluation",
        "metric": "Retrieval-valid evaluation questions",
        "value": _count_rows(eval_valid_path),
        "status": "PASS" if eval_valid_path.exists() and _count_rows(eval_valid_path) > 0 else "FAIL",
        "interpretation": "Only records valid for objective retrieval metrics.",
    },
    {
        "category": "Chunks",
        "metric": "Structural chunks",
        "value": _count_rows(structural_path),
        "status": "PASS" if structural_path.exists() and _count_rows(structural_path) > 0 else "FAIL",
        "interpretation": "Structural legal chunking output.",
    },
    {
        "category": "Chunks",
        "metric": "Fixed-size chunks",
        "value": _count_rows(fixed_path),
        "status": "PASS" if fixed_path.exists() and _count_rows(fixed_path) > 0 else "FAIL",
        "interpretation": "Baseline chunking output.",
    },
    {
        "category": "Safety",
        "metric": "Chunk quality critical failures",
        "value": int(chunk_quality_report[chunk_quality_report["status"].eq("FAIL")].shape[0]) if "chunk_quality_report" in globals() else 0,
        "status": "PASS" if "chunk_quality_report" in globals() and chunk_quality_report[chunk_quality_report["status"].eq("FAIL")].empty else "FAIL",
        "interpretation": "Must be zero before moving to retrieval experiments.",
    },
]

stage13_final_summary_table = pd.DataFrame(summary_rows)

# ---------------------------------------------------------
# 5) Save final manifest and summary
# ---------------------------------------------------------

_write_dataframe(final_output_manifest, REPORTS_DIR / "stage13_final_output_manifest.csv")
_write_dataframe(final_output_manifest, REPORTS_DIR / "stage13_final_output_manifest.xlsx")
_write_dataframe(stage13_final_summary_table, REPORTS_DIR / "stage13_final_summary_table.csv")
_write_dataframe(stage13_final_summary_table, REPORTS_DIR / "stage13_final_summary_table.xlsx")

with open(REPORTS_DIR / "stage13_final_output_manifest.json", "w", encoding="utf-8") as f:
    json.dump(final_output_manifest.to_dict(orient="records"), f, ensure_ascii=False, indent=2)

next_notebook_files = final_output_manifest[final_output_manifest["file_group"].eq("Notebook 03 required input")].copy()
_write_dataframe(next_notebook_files, REPORTS_DIR / "notebook03_required_files.csv")
_write_dataframe(next_notebook_files, REPORTS_DIR / "notebook03_required_files.xlsx")

# ---------------------------------------------------------
# 6) Display professional outputs
# ---------------------------------------------------------

display(professional_table(stage13_final_summary_table, "Stage 13 — Final Summary Table"))
display(professional_table(final_output_manifest, "Stage 13 — Final Output Manifest"))
display(professional_table(next_notebook_files[["file_name", "file_path", "row_count", "status", "notebook03_usage"]], "Files to Use in Notebook 03"))

# ---------------------------------------------------------
# 7) Final assertions
# ---------------------------------------------------------

missing_or_empty = final_output_manifest[final_output_manifest["status"].eq("FAIL")]
summary_failures = stage13_final_summary_table[stage13_final_summary_table["status"].eq("FAIL")]

assert missing_or_empty.empty, "Stage 13 failed: one or more required files are missing or empty."
assert summary_failures.empty, "Stage 13 failed: one or more final summary checks failed."

# ---------------------------------------------------------
# 8) Final academic note
# ---------------------------------------------------------

display(Markdown(f"""
<div dir="rtl" style="text-align:right; line-height:2; font-family:Arial, Tahoma, sans-serif; background-color:#E2F0D9; border-right:5px solid #375623; padding:16px; border-radius:6px;">
<h3 style="margin-top:0; color:#375623;">Final Status — PASS</h3>
تم توليد كل ملفات Notebook 03 المطلوبة بنجاح، وجميع فحوصات السلامة الحرجة اجتازت الاختبار.
<br><br>
<b>الخطوة التالية:</b><br>
افتح النوتبوك التالي: <code>03_RAG_Retrieval_Evaluation_ChromaDB_READY.ipynb</code><br>
واستخدم الملفات الموضحة في جدول <b>Files to Use in Notebook 03</b> أعلاه.
</div>
"""))


category,metric,value,status,interpretation
Knowledge Base,Experimental KB records,489,PASS,Main active-law-only knowledge base for Notebook 03.
Evaluation,Full evaluation questions,160,PASS,Full evaluation dataset retained for audit and qualitative analysis.
Evaluation,Retrieval-valid evaluation questions,145,PASS,Only records valid for objective retrieval metrics.
Chunks,Structural chunks,490,PASS,Structural legal chunking output.
Chunks,Fixed-size chunks,490,PASS,Baseline chunking output.
Safety,Chunk quality critical failures,0,PASS,Must be zero before moving to retrieval experiments.


file_group,file_name,file_path,exists,row_count,size_kb,notebook03_usage,status
Notebook 03 required input,Experimental KB - active law only,saudi_labor_law_voice_agent_project\03_final\rag_knowledge_base_dataset_experiment_no_leakage_active_law_only.csv,True,489,1525.580000,Knowledge base for fair retrieval evaluation,PASS
Notebook 03 required input,Retrieval evaluation dataset - valid only,saudi_labor_law_voice_agent_project\03_final\rag_retrieval_evaluation_dataset_valid_only.csv,True,145,183.450000,Queries with valid gold labels for retrieval metrics,PASS
Notebook 03 required input,Structural chunks,saudi_labor_law_voice_agent_project\04_chunks\rag_chunks_structural_legal_experiment.csv,True,490,889.610000,Legal-structure-aware chunking experiment,PASS
Notebook 03 required input,Fixed-size chunks,saudi_labor_law_voice_agent_project\04_chunks\rag_chunks_fixed_size_experiment.csv,True,490,862.880000,Baseline fixed-size chunking experiment,PASS
Audit output,Full evaluation dataset,saudi_labor_law_voice_agent_project\03_final\rag_evaluation_dataset.csv,True,160,187.520000,Full evaluation and audit copy,PASS
Audit output,Repealed legal archive,saudi_labor_law_voice_agent_project\03_final\legal_articles_repealed_archive_not_indexed.csv,True,38,41.620000,Archived only; must not be indexed,PASS


file_name,file_path,row_count,status,notebook03_usage
Experimental KB - active law only,saudi_labor_law_voice_agent_project\03_final\rag_knowledge_base_dataset_experiment_no_leakage_active_law_only.csv,489,PASS,Knowledge base for fair retrieval evaluation
Retrieval evaluation dataset - valid only,saudi_labor_law_voice_agent_project\03_final\rag_retrieval_evaluation_dataset_valid_only.csv,145,PASS,Queries with valid gold labels for retrieval metrics
Structural chunks,saudi_labor_law_voice_agent_project\04_chunks\rag_chunks_structural_legal_experiment.csv,490,PASS,Legal-structure-aware chunking experiment
Fixed-size chunks,saudi_labor_law_voice_agent_project\04_chunks\rag_chunks_fixed_size_experiment.csv,490,PASS,Baseline fixed-size chunking experiment



<div dir="rtl" style="text-align:right; line-height:2; font-family:Arial, Tahoma, sans-serif; background-color:#E2F0D9; border-right:5px solid #375623; padding:16px; border-radius:6px;">
<h3 style="margin-top:0; color:#375623;">Final Status — PASS</h3>
تم توليد كل ملفات Notebook 03 المطلوبة بنجاح، وجميع فحوصات السلامة الحرجة اجتازت الاختبار.
<br><br>
<b>الخطوة التالية:</b><br>
افتح النوتبوك التالي: <code>03_RAG_Retrieval_Evaluation_ChromaDB_READY.ipynb</code><br>
واستخدم الملفات الموضحة في جدول <b>Files to Use in Notebook 03</b> أعلاه.
</div>
